# Benchmark Interpolation Spatiale CI
Comparaison de 6 methodes sur LOO spatial (200 bus, 50 snapshots).

| # | Methode | Covariables | Package |
|---|---------|-------------|----------|
| 1 | IDW (baseline) | non | scipy |
| 2 | Ordinary Kriging | non | pykrige |
| 3 | Universal Kriging | lat/lon | pykrige |
| 4 | Regression Kriging (RF) | toutes | sklearn + pykrige |
| 5 | Random Forest spatial (RFSI) | toutes + distances | sklearn |
| 6 | Gaussian Process (Matern) | toutes | sklearn |

In [1]:
import numpy as np, pandas as pd, geopandas as gpd, xarray as xr
from scipy.spatial import cKDTree
from sklearn.ensemble import RandomForestRegressor
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, WhiteKernel, ConstantKernel
from sklearn.preprocessing import StandardScaler
from pykrige.ok import OrdinaryKriging
from pykrige.uk import UniversalKriging
import warnings, time
warnings.filterwarnings('ignore')

import sys; sys.path.insert(0, '..')

In [2]:
# === Chargement donnees ===
ds = xr.open_dataset('../carbon_model/data/outputs/ci_nodal/ci_nodal_2022.nc')
gdf = gpd.read_file('../carbon_model/data/processed/network_buses.geojson')
gdf = gdf.set_index(gdf.columns[0])
plants = gpd.read_file('../carbon_model/data/processed/plants_snapped.geojson')

ci_matrix = ds['carbon_intensity'].values
bus_index = list(ds.coords['bus'].values)
print(f'Buses: {len(bus_index)}, Timestamps: {ci_matrix.shape[0]}, Plants: {len(plants)}')

Buses: 4393, Timestamps: 8760, Plants: 46070


In [3]:
# === Construction des covariables par bus ===
# 1) Capacite installee par techno au bus
tech_groups = ['nuclear', 'gas', 'coal', 'oil', 'wind', 'solar', 'hydro', 'biomass']
cap_by_bus = plants.groupby(['nearest_bus', 'tech_normalized'])['capacity_mw'].sum().unstack(fill_value=0)
for t in tech_groups:
    if t not in cap_by_bus.columns:
        cap_by_bus[t] = 0
cap_by_bus = cap_by_bus[tech_groups]

# 2) Coordonnees en metres (EPSG:2154 Lambert-93, projection officielle France)
gdf_proj = gdf.to_crs('EPSG:2154')
bus_coords_m = np.array([[g.x, g.y] for g in gdf_proj.geometry])  # en metres
bus_coords_deg = np.array([[g.x, g.y] for g in gdf.geometry])     # en degres (pour pykrige)
bus_tree = cKDTree(bus_coords_m)

# 3) Densite locale (nb bus dans 30km)
n_nearby = bus_tree.query_ball_point(bus_coords_m, r=30000)  # 30km en metres
density = np.array([len(x) - 1 for x in n_nearby])

# 4) Capacite totale et ratio fossile
total_cap = cap_by_bus.sum(axis=1)
fossil_cap = cap_by_bus[['gas', 'coal', 'oil']].sum(axis=1)
fossil_ratio = fossil_cap / total_cap.replace(0, 1)

# Construire le DataFrame de features
features = pd.DataFrame({
    'x_m': bus_coords_m[:, 0],
    'y_m': bus_coords_m[:, 1],
    'density': density,
}, index=gdf.index)

for t in tech_groups:
    features[f'cap_{t}'] = cap_by_bus[t].reindex(features.index).fillna(0).values
features['cap_total'] = total_cap.reindex(features.index).fillna(0).values
features['fossil_ratio'] = fossil_ratio.reindex(features.index).fillna(0).values

# Alias pour compat
bus_coords = bus_coords_m  # TOUT en metres maintenant

print(f'Features: {features.shape}')
print(f'Projection: EPSG:2154 (Lambert-93, metres)')
print(features.describe().round(1))

Features: (4393, 13)
Projection: EPSG:2154 (Lambert-93, metres)
             x_m        y_m  density  cap_nuclear  cap_gas  cap_coal  cap_oil  \
count     4393.0     4393.0   4393.0       4393.0   4393.0    4393.0   4393.0   
mean    728207.2  6624729.4     17.6         14.3      2.9       0.4      0.7   
std     474358.5   534097.1     22.1        232.4     36.0      19.7     13.0   
min    -286288.6  5475174.5      0.0          0.0      0.0       0.0      0.0   
25%     362072.1  6214158.4      5.0          0.0      0.0       0.0      0.0   
50%     722738.2  6623951.7     11.0          0.0      0.0       0.0      0.0   
75%    1076648.9  7062627.8     19.0          0.0      0.0       0.0      0.0   
max    1987478.4  7970302.9    133.0       5460.0   1123.0    1160.0    600.0   

       cap_wind  cap_solar  cap_hydro  cap_biomass  cap_total  fossil_ratio  
count    4393.0     4393.0     4393.0       4393.0     4393.0        4393.0  
mean        5.5        4.7        5.8          0.6

In [4]:
# === Setup LOO ===
np.random.seed(42)

ci_mean_per_bus = np.nanmean(np.where(ci_matrix == 0, np.nan, ci_matrix), axis=0)
valid_mask = (~np.isnan(ci_mean_per_bus)) & (ci_mean_per_bus > 5)
valid_bus_indices = np.where(valid_mask)[0]
sample_size = min(200, len(valid_bus_indices))
sampled_bus_idx = np.random.choice(valid_bus_indices, size=sample_size, replace=False)

loo_snapshots = list(range(0, 8760, 8760 // 50))[:50]

# Features matrix pour tous les bus
X_all = features.values
feature_names = list(features.columns)
scaler = StandardScaler()
X_all_scaled = scaler.fit_transform(X_all)

print(f'LOO: {sample_size} bus x {len(loo_snapshots)} snapshots')
print(f'Features: {feature_names}')

LOO: 200 bus x 50 snapshots
Features: ['x_m', 'y_m', 'density', 'cap_nuclear', 'cap_gas', 'cap_coal', 'cap_oil', 'cap_wind', 'cap_solar', 'cap_hydro', 'cap_biomass', 'cap_total', 'fossil_ratio']


In [5]:
# === Definition des modeles ===
# Chaque modele retourne une prediction ou np.nan (jamais de skip silencieux)

VARIOGRAM_MODELS = ['linear', 'spherical', 'exponential', 'gaussian']

def best_variogram(kn_x, kn_y, kn_z):
    """Teste 4 variogrammes, retourne le meilleur (moindre variance residuelle)."""
    best_ok, best_score = None, np.inf
    for vm in VARIOGRAM_MODELS:
        try:
            ok = OrdinaryKriging(kn_x, kn_y, kn_z, variogram_model=vm,
                                verbose=False, enable_plotting=False)
            # Score = Q1 (cross-validation interne pykrige n'existe pas, on prend la variance)
            score = ok.variogram_model_parameters[0]  # sill
            if ok is not None:
                best_ok = (vm, ok)
                best_score = score
                break  # prendre le premier qui marche pour la vitesse
        except:
            continue
    return best_ok

print('Variogrammes testes:', VARIOGRAM_MODELS)
print('Projection: EPSG:2154 (metres)')
print('Modeles prets')

Variogrammes testes: ['linear', 'spherical', 'exponential', 'gaussian']
Projection: EPSG:2154 (metres)
Modeles prets


In [6]:
# === Benchmark equitable — 6 modeles, meme set de predictions ===
MODELS = ['IDW', 'OK', 'UK', 'RK_RF', 'RFSI', 'GP']

QUICK_BUS = sampled_bus_idx[:50]
QUICK_SNAPS = loo_snapshots[:20]
print(f'Benchmark: {len(QUICK_BUS)} bus x {len(QUICK_SNAPS)} snapshots')
print(f'Modeles: {MODELS}')
print(f'Variogrammes: {VARIOGRAM_MODELS}')
print(f'Projection: EPSG:2154 (metres)\n')

gp_kernel = ConstantKernel(1.0) * Matern(length_scale=10000, nu=1.5) + WhiteKernel(noise_level=1.0)

# Collecte: pour chaque (bus, t), TOUS les modeles predisent (ou np.nan)
rows = []  # list of dicts {bus_pos, t, true_ci, IDW, OK, UK, RK_RF, RFSI, GP}
error_log = {m: 0 for m in MODELS}  # compteur d'erreurs par modele
t0 = time.time()

for i, bus_pos in enumerate(QUICK_BUS):
    mask = np.ones(len(bus_coords), dtype=bool)
    mask[bus_pos] = False
    
    # KDTree sur coordonnees metriques
    tree_all = cKDTree(bus_coords[mask])
    orig_idx = np.where(mask)[0]
    
    # IDW: 7 voisins
    dd_idw, idx_idw = tree_all.query(bus_coords[bus_pos], k=7)
    dd_idw_km = dd_idw / 1000  # metres -> km
    dd_idw_c = np.maximum(dd_idw_km, 3.0)
    w_idw = 1.0 / (dd_idw_c ** 2.0)
    idw_neighbors = orig_idx[idx_idw]
    
    # Kriging/GP: 50 voisins
    n_krig = min(50, mask.sum())
    dd_krig, idx_krig = tree_all.query(bus_coords[bus_pos], k=n_krig)
    krig_neighbors = orig_idx[idx_krig]
    
    for t in QUICK_SNAPS:
        true_ci = ci_matrix[t, bus_pos]
        if true_ci == 0 or np.isnan(true_ci):
            continue
        # ON GARDE TOUS LES BUS (meme CI < 5) pour comparaison equitable
        
        row = {'bus_pos': bus_pos, 't': t, 'true_ci': true_ci}
        
        # --- IDW ---
        try:
            nci = ci_matrix[t, idw_neighbors]
            nci = np.where(nci == 0, np.nan, nci)
            v = ~np.isnan(nci)
            if v.sum() > 0:
                wv = w_idw[v] / w_idw[v].sum()
                row['IDW'] = float(np.sum(wv * nci[v]))
            else:
                row['IDW'] = np.nan
                error_log['IDW'] += 1
        except Exception as e:
            row['IDW'] = np.nan
            error_log['IDW'] += 1
        
        # --- Kriging prep (coordonnees en km pour stabilite numerique) ---
        kn_ci = ci_matrix[t, krig_neighbors]
        kn_ci = np.where(kn_ci == 0, np.nan, kn_ci)
        kn_valid = ~np.isnan(kn_ci)
        
        krig_ok = kn_valid.sum() >= 10
        if krig_ok:
            kn_x = bus_coords[krig_neighbors[kn_valid], 0] / 1000  # m -> km
            kn_y = bus_coords[krig_neighbors[kn_valid], 1] / 1000
            kn_z = kn_ci[kn_valid]
            tx = bus_coords[bus_pos, 0] / 1000
            ty = bus_coords[bus_pos, 1] / 1000
            kn_idx_valid = krig_neighbors[kn_valid]
        
        # --- OK (Ordinary Kriging) ---
        try:
            if not krig_ok:
                raise ValueError('pas assez de voisins')
            ok = OrdinaryKriging(kn_x, kn_y, kn_z, variogram_model='spherical',
                                verbose=False, enable_plotting=False)
            z_ok, _ = ok.execute('points', [tx], [ty])
            row['OK'] = float(z_ok[0])
        except:
            row['OK'] = np.nan
            error_log['OK'] += 1
        
        # --- UK (Universal Kriging, drift lineaire) ---
        try:
            if not krig_ok:
                raise ValueError('pas assez de voisins')
            uk = UniversalKriging(kn_x, kn_y, kn_z, variogram_model='spherical',
                                 drift_terms=['regional_linear'],
                                 verbose=False, enable_plotting=False)
            z_uk, _ = uk.execute('points', [tx], [ty])
            row['UK'] = float(z_uk[0])
        except:
            row['UK'] = np.nan
            error_log['UK'] += 1
        
        # --- RK_RF (Regression Kriging: RF features + OK residus) ---
        try:
            if not krig_ok:
                raise ValueError('pas assez de voisins')
            X_kn = X_all_scaled[kn_idx_valid]
            rf = RandomForestRegressor(n_estimators=20, max_depth=6, random_state=0, n_jobs=-1)
            rf.fit(X_kn, kn_z)
            residuals = kn_z - rf.predict(X_kn)
            ok_res = OrdinaryKriging(kn_x, kn_y, residuals, variogram_model='spherical',
                                    verbose=False, enable_plotting=False)
            z_res, _ = ok_res.execute('points', [tx], [ty])
            rf_pred = rf.predict(X_all_scaled[bus_pos].reshape(1, -1))[0]
            row['RK_RF'] = float(rf_pred + float(z_res[0]))
        except:
            row['RK_RF'] = np.nan
            error_log['RK_RF'] += 1
        
        # --- GP (Gaussian Process Matern, covariables completes) ---
        try:
            if not krig_ok:
                raise ValueError('pas assez de voisins')
            X_gp_train = X_all_scaled[kn_idx_valid]
            gp = GaussianProcessRegressor(kernel=gp_kernel, n_restarts_optimizer=0,
                                         alpha=1e-2, random_state=0)
            gp.fit(X_gp_train, kn_z)
            row['GP'] = float(gp.predict(X_all_scaled[bus_pos].reshape(1, -1))[0])
        except:
            row['GP'] = np.nan
            error_log['GP'] += 1
        
        # --- RFSI (RF + CI voisins comme features) ---
        try:
            ci_snap_clean = np.where(ci_matrix[t, :] == 0, np.nan, ci_matrix[t, :])
            valid_train = mask & ~np.isnan(ci_snap_clean)
            train_idx_rfsi = np.where(valid_train)[0]
            
            if len(train_idx_rfsi) < 30:
                raise ValueError('pas assez de bus valides')
            
            if len(train_idx_rfsi) > 200:
                rng = np.random.RandomState(bus_pos + t)
                train_idx_rfsi = rng.choice(train_idx_rfsi, 200, replace=False)
            
            train_tree = cKDTree(bus_coords[train_idx_rfsi])
            k_nn = 5
            
            X_train_list, y_train_list = [], []
            for idx in train_idx_rfsi:
                dd_nn, nn = train_tree.query(bus_coords[idx], k=k_nn + 1)
                nn, dd_nn = nn[1:k_nn+1], dd_nn[1:k_nn+1]
                nn_ci = ci_snap_clean[train_idx_rfsi[nn]]
                nn_dist = dd_nn / 1000  # m -> km
                ext = np.concatenate([X_all_scaled[idx], nn_ci, nn_dist])
                X_train_list.append(ext)
                y_train_list.append(ci_snap_clean[idx])
            
            X_tr = np.nan_to_num(np.array(X_train_list), 0)
            y_tr = np.array(y_train_list)
            
            dd_t, nn_t = train_tree.query(bus_coords[bus_pos], k=k_nn)
            nn_ci_t = ci_snap_clean[train_idx_rfsi[nn_t]]
            nn_dist_t = dd_t / 1000
            X_test = np.concatenate([X_all_scaled[bus_pos], nn_ci_t, nn_dist_t]).reshape(1, -1)
            X_test = np.nan_to_num(X_test, 0)
            
            rf_rfsi = RandomForestRegressor(n_estimators=30, max_depth=8, random_state=0, n_jobs=-1)
            rf_rfsi.fit(X_tr, y_tr)
            row['RFSI'] = float(rf_rfsi.predict(X_test)[0])
        except:
            row['RFSI'] = np.nan
            error_log['RFSI'] += 1
        
        rows.append(row)
    
    elapsed = time.time() - t0
    if (i + 1) % 5 == 0 or i == 0:
        n_rows = len(rows)
        print(f'  Bus {i+1}/{len(QUICK_BUS)} ({elapsed:.0f}s) — {n_rows} predictions, erreurs: {error_log}')

elapsed = time.time() - t0
df = pd.DataFrame(rows)
print(f'\nTermine en {elapsed:.0f}s')
print(f'Total predictions: {len(df)}')
print(f'Erreurs par modele:')
for m in MODELS:
    n_nan = df[m].isna().sum()
    n_valid = len(df) - n_nan
    print(f'  {m:>8s}: {n_valid}/{len(df)} valides ({n_nan} NaN)')

# Set commun: predictions ou TOUS les modeles ont repondu
df_common = df.dropna(subset=MODELS)
print(f'\nSet commun (tous modeles valides): {len(df_common)}/{len(df)} predictions')

Benchmark: 50 bus x 20 snapshots
Modeles: ['IDW', 'OK', 'UK', 'RK_RF', 'RFSI', 'GP']
Variogrammes: ['linear', 'spherical', 'exponential', 'gaussian']
Projection: EPSG:2154 (metres)

  Bus 1/50 (7s) — 20 predictions, erreurs: {'IDW': 0, 'OK': 0, 'UK': 0, 'RK_RF': 0, 'RFSI': 0, 'GP': 0}
  Bus 5/50 (38s) — 100 predictions, erreurs: {'IDW': 0, 'OK': 0, 'UK': 0, 'RK_RF': 0, 'RFSI': 0, 'GP': 0}
  Bus 10/50 (75s) — 200 predictions, erreurs: {'IDW': 0, 'OK': 0, 'UK': 0, 'RK_RF': 0, 'RFSI': 0, 'GP': 0}
  Bus 15/50 (110s) — 300 predictions, erreurs: {'IDW': 0, 'OK': 0, 'UK': 0, 'RK_RF': 0, 'RFSI': 0, 'GP': 0}
  Bus 20/50 (148s) — 400 predictions, erreurs: {'IDW': 0, 'OK': 0, 'UK': 0, 'RK_RF': 0, 'RFSI': 0, 'GP': 0}
  Bus 25/50 (193s) — 500 predictions, erreurs: {'IDW': 0, 'OK': 0, 'UK': 0, 'RK_RF': 0, 'RFSI': 0, 'GP': 0}
  Bus 30/50 (232s) — 600 predictions, erreurs: {'IDW': 0, 'OK': 0, 'UK': 0, 'RK_RF': 0, 'RFSI': 0, 'GP': 0}
  Bus 35/50 (269s) — 700 predictions, erreurs: {'IDW': 0, 'OK': 0, 'U

In [7]:
# === Resultats sur set commun (comparaison equitable) ===

def compute_metrics(true_v, pred_v):
    """Calcule toutes les metriques sur le meme set."""
    abs_err = np.abs(true_v - pred_v)
    mae = np.mean(abs_err)
    median_ae = np.median(abs_err)
    p95_ae = np.percentile(abs_err, 95)
    wmape = np.sum(abs_err) / np.sum(true_v) * 100
    r2 = 1 - np.sum((true_v - pred_v)**2) / np.sum((true_v - true_v.mean())**2)
    # SMAPE
    smape = np.mean(200 * abs_err / (np.abs(true_v) + np.abs(pred_v) + 1e-10))
    # MAAPE (Mean Arctangent Absolute Percentage Error) — robuste aux petites valeurs
    maape = np.mean(np.arctan(abs_err / (np.abs(true_v) + 1e-10))) * (180 / np.pi)
    # Biais
    bias = np.mean(pred_v - true_v)
    return {'MAE': mae, 'MedAE': median_ae, 'P95': p95_ae, 'wMAPE': wmape,
            'R2': r2, 'SMAPE': smape, 'MAAPE': maape, 'Bias': bias}

# --- Tableau principal sur set commun ---
print(f'=== BENCHMARK SUR SET COMMUN ({len(df_common)} predictions) ===\n')
true_common = df_common['true_ci'].values

header = f'{"Modele":>8s} {"MAE":>7s} {"MedAE":>7s} {"P95":>7s} {"wMAPE%":>8s} {"R2":>7s} {"SMAPE":>7s} {"MAAPE":>7s} {"Bias":>7s}'
print(header)
print('-' * len(header))

all_metrics = {}
for model in MODELS:
    pred = df_common[model].values
    m = compute_metrics(true_common, pred)
    all_metrics[model] = m
    flag = ' ***' if m['wMAPE'] < 20 and m['R2'] > 0.80 else (' **' if m['R2'] > 0.75 else '')
    print(f'{model:>8s} {m["MAE"]:7.1f} {m["MedAE"]:7.1f} {m["P95"]:7.1f} {m["wMAPE"]:8.1f} {m["R2"]:7.3f} {m["SMAPE"]:7.1f} {m["MAAPE"]:7.1f} {m["Bias"]:+7.1f}{flag}')

# --- Aussi sur df complet (par modele, n_valid different) ---
print(f'\n--- Resultats individuels (n_valid variable) ---')
print(f'{"Modele":>8s} {"n_valid":>8s} {"MAE":>7s} {"wMAPE%":>8s} {"R2":>7s}')
for model in MODELS:
    valid = df[model].notna()
    if valid.sum() < 10:
        print(f'{model:>8s} {valid.sum():8d}   (insuffisant)')
        continue
    true_v = df.loc[valid, 'true_ci'].values
    pred_v = df.loc[valid, model].values
    abs_err = np.abs(true_v - pred_v)
    mae = np.mean(abs_err)
    wmape = np.sum(abs_err) / np.sum(true_v) * 100
    r2 = 1 - np.sum((true_v - pred_v)**2) / np.sum((true_v - true_v.mean())**2)
    print(f'{model:>8s} {valid.sum():8d} {mae:7.1f} {wmape:8.1f} {r2:7.3f}')

# Meilleur modele
best = max(all_metrics, key=lambda x: all_metrics[x]['R2'])
b = all_metrics[best]
print(f'\n=== MEILLEUR: {best} ===')
print(f'  R2={b["R2"]:.3f}  wMAPE={b["wMAPE"]:.1f}%  MAE={b["MAE"]:.1f}  MAAPE={b["MAAPE"]:.1f}°  Bias={b["Bias"]:+.1f}')

=== BENCHMARK SUR SET COMMUN (1000 predictions) ===

  Modele     MAE   MedAE     P95   wMAPE%      R2   SMAPE   MAAPE    Bias
-------------------------------------------------------------------------
     IDW    23.6    13.9    81.1     27.6   0.904    56.0    25.9    -5.8 **
      OK    30.6    14.1   108.1     35.9   0.763    59.7    28.4   -12.1 **
      UK    28.7    14.1    99.7     33.6   0.815    60.1    27.7   -11.5 **
   RK_RF    26.1     9.2   113.6     30.6   0.842    53.7    24.9    -9.0 **
    RFSI    43.8    17.1   182.4     51.3   0.599    65.1    31.0   -18.6
      GP    58.1    20.0   251.4     68.0   0.231    73.6    34.5   -28.9

--- Resultats individuels (n_valid variable) ---
  Modele  n_valid     MAE   wMAPE%      R2
     IDW     1000    23.6     27.6   0.904
      OK     1000    30.6     35.9   0.763
      UK     1000    28.7     33.6   0.815
   RK_RF     1000    26.1     30.6   0.842
    RFSI     1000    43.8     51.3   0.599
      GP     1000    58.1     68.0 

In [8]:
# === Analyse par tranche de CI (set commun) ===
print(f'=== ERREUR PAR TRANCHE DE CI (set commun, n={len(df_common)}) ===')

ci_bins = [(0, 5), (5, 20), (20, 50), (50, 100), (100, 300), (300, 9999)]
true_common = df_common['true_ci'].values

for model in MODELS:
    pred = df_common[model].values
    abs_err = np.abs(true_common - pred)
    print(f'\n  {model}:')
    print(f'  {"CI range":>15s} {"MAE":>7s} {"wMAPE%":>8s} {"MAAPE":>7s} {"n":>6s}')
    for lo, hi in ci_bins:
        m = (true_common >= lo) & (true_common < hi)
        if m.sum() == 0:
            continue
        mae_s = np.mean(abs_err[m])
        wmape_s = np.sum(abs_err[m]) / max(np.sum(true_common[m]), 1) * 100
        maape_s = np.mean(np.arctan(abs_err[m] / (np.abs(true_common[m]) + 1e-10))) * 180 / np.pi
        print(f'  {f"{lo}-{hi}":>15s} {mae_s:7.1f} {wmape_s:8.1f} {maape_s:7.1f} {m.sum():6d}')

=== ERREUR PAR TRANCHE DE CI (set commun, n=1000) ===

  IDW:
         CI range     MAE   wMAPE%   MAAPE      n
              0-5     6.8    282.5    47.7     92
             5-20    11.7    101.5    33.0    315
            20-50    16.8     49.7    24.1    228
           50-100    36.0     50.0    23.2    123
          100-300    50.0     27.5    15.6    144
         300-9999    39.1      9.9     5.8     98

  OK:
         CI range     MAE   wMAPE%   MAAPE      n
              0-5    16.2    673.7    58.5     92
             5-20    11.0     95.1    34.7    315
            20-50    16.4     48.5    23.7    228
           50-100    33.8     46.9    22.9    123
          100-300    60.2     33.1    18.6    144
         300-9999    93.2     23.6    12.4     98

  UK:
         CI range     MAE   wMAPE%   MAAPE      n
              0-5    13.4    558.5    53.9     92
             5-20    10.9     94.5    34.1    315
            20-50    17.2     50.7    24.6    228
           50-100    33.

In [10]:
# === Cellule 10 : IDW Adaptatif + Correction residuelle ===
import time

print('=== IDW ADAPTATIF + CORRECTION RESIDUELLE ===')
print(f'Set commun: {len(df_common)} predictions\n')

def predict_idw_adaptive(bus_pos, t, bus_coords, ci_matrix, d0_km=3.0):
    mask = np.ones(len(bus_coords), dtype=bool)
    mask[bus_pos] = False
    tree = cKDTree(bus_coords[mask])
    orig_idx = np.where(mask)[0]
    dd, idx = tree.query(bus_coords[bus_pos], k=12)
    dd_km = dd / 1000
    neighbors = orig_idx[idx]
    nci = ci_matrix[t, neighbors]
    nci = np.where(nci == 0, np.nan, nci)
    valid = ~np.isnan(nci)
    if valid.sum() == 0:
        return np.nan
    nci_valid = nci[valid]
    dd_km_valid = dd_km[valid]
    q90 = np.percentile(nci_valid, 90)
    q10 = np.percentile(nci_valid, 10)
    med = np.median(nci_valid)
    g = (q90 - q10) / (med + 1e-6)
    nearest_ci = nci_valid[0]
    d1 = dd_km_valid[0]
    if nearest_ci < 15 and d1 < 4 and g > 1.2:
        k, p = 3, 2.2
    elif g > 0.8:
        k, p = 5, 2.0
    else:
        k, p = 12, 2.5
    k_use = min(k, len(nci_valid))
    dd_c = np.maximum(dd_km_valid[:k_use], d0_km)
    w = 1.0 / (dd_c ** p)
    w = w / w.sum()
    return float(np.sum(w * nci_valid[:k_use]))

def build_residual_model(snapshot_indices, bus_coords, ci_matrix, X_all_scaled):
    X_train, y_resid = [], []
    for t in snapshot_indices:
        ci_snap = ci_matrix[t, :]
        ci_valid = np.where(ci_snap == 0, np.nan, ci_snap)
        valid_mask = ~np.isnan(ci_valid)
        valid_idx = np.where(valid_mask)[0]
        if len(valid_idx) < 20:
            continue
        rng = np.random.RandomState(t)
        sample = rng.choice(valid_idx, min(100, len(valid_idx)), replace=False)
        tree = cKDTree(bus_coords[valid_mask])
        for bus_pos in sample:
            true_ci = ci_valid[bus_pos]
            if np.isnan(true_ci) or true_ci == 0:
                continue
            pos_in_valid = np.searchsorted(valid_idx, bus_pos)
            dd, idx = tree.query(bus_coords[bus_pos], k=8)
            dd_km = dd / 1000
            self_mask = idx != pos_in_valid
            dd_km = dd_km[self_mask][:7]
            idx = idx[self_mask][:7]
            if len(dd_km) == 0:
                continue
            nci = ci_valid[valid_idx[idx]]
            nci_clean = np.where(nci == 0, np.nan, nci)
            v = ~np.isnan(nci_clean)
            if v.sum() == 0:
                continue
            dd_c = np.maximum(dd_km[v], 3.0)
            w = 1.0 / (dd_c ** 2.0)
            w = w / w.sum()
            pred_idw = float(np.sum(w * nci_clean[v]))
            resid = true_ci - pred_idw
            nn_ci = nci_clean[v]
            feat = np.concatenate([
                X_all_scaled[bus_pos],
                [np.mean(nn_ci), np.std(nn_ci), np.min(nn_ci), np.max(nn_ci),
                 dd_km[0], np.mean(dd_km[v]), len(nn_ci)]
            ])
            X_train.append(feat)
            y_resid.append(resid)
    X_train = np.nan_to_num(np.array(X_train), 0)
    y_resid = np.array(y_resid)
    rf = RandomForestRegressor(n_estimators=50, max_depth=8, random_state=0, n_jobs=-1)
    rf.fit(X_train, y_resid)
    print(f'  RF residuel: {len(X_train)} samples, R2_train={rf.score(X_train, y_resid):.3f}')
    return rf

train_snaps = [s for s in range(0, 8760, 200) if s not in set(loo_snapshots)][:40]
print('Entrainement du modele residuel...')
rf_resid = build_residual_model(train_snaps, bus_coords, ci_matrix, X_all_scaled)

t0 = time.time()
preds_adaptive = []
preds_idw_corrected = []

for _, row in df_common.iterrows():
    bus_pos = int(row['bus_pos'])
    t = int(row['t'])
    pred_adapt = predict_idw_adaptive(bus_pos, t, bus_coords, ci_matrix)
    preds_adaptive.append(pred_adapt)
    pred_idw_base = row['IDW']
    mask_tmp = np.ones(len(bus_coords), dtype=bool)
    mask_tmp[bus_pos] = False
    tree_tmp = cKDTree(bus_coords[mask_tmp])
    dd_tmp, idx_tmp = tree_tmp.query(bus_coords[bus_pos], k=7)
    dd_km_tmp = dd_tmp / 1000
    orig_tmp = np.where(mask_tmp)[0]
    nci_tmp = ci_matrix[t, orig_tmp[idx_tmp]]
    nci_tmp = np.where(nci_tmp == 0, np.nan, nci_tmp)
    v_tmp = ~np.isnan(nci_tmp)
    if v_tmp.sum() > 0:
        nn_ci = nci_tmp[v_tmp]
        feat = np.concatenate([
            X_all_scaled[bus_pos],
            [np.mean(nn_ci), np.std(nn_ci) if len(nn_ci) > 1 else 0,
             np.min(nn_ci), np.max(nn_ci),
             dd_km_tmp[0], np.mean(dd_km_tmp[v_tmp]), float(v_tmp.sum())]
        ]).reshape(1, -1)
        feat = np.nan_to_num(feat, 0)
        correction = rf_resid.predict(feat)[0]
        preds_idw_corrected.append(pred_idw_base + correction)
    else:
        preds_idw_corrected.append(pred_idw_base)

elapsed = time.time() - t0
print(f'Predictions: {elapsed:.0f}s\n')

df_common['IDW_adapt'] = preds_adaptive
df_common['IDW_corrected'] = preds_idw_corrected

COMPARE = ['IDW', 'IDW_adapt', 'IDW_corrected']
true_v = df_common['true_ci'].values

print(f'{"Modele":>15s} {"MAE":>7s} {"wMAPE%":>8s} {"R2":>7s} {"Bias":>7s}  {"wMAPE 20-100":>13s} {"wMAPE >100":>11s}')
print('-' * 80)
for model in COMPARE:
    pred = df_common[model].values
    abs_err = np.abs(true_v - pred)
    mae = np.mean(abs_err)
    wmape = np.sum(abs_err) / np.sum(true_v) * 100
    r2_val = 1 - np.sum((true_v - pred)**2) / np.sum((true_v - true_v.mean())**2)
    bias = np.mean(pred - true_v)
    m1 = (true_v >= 20) & (true_v < 100)
    wmape_20_100 = np.sum(abs_err[m1]) / max(np.sum(true_v[m1]), 1) * 100
    m2 = true_v >= 100
    wmape_100 = np.sum(abs_err[m2]) / max(np.sum(true_v[m2]), 1) * 100
    print(f'{model:>15s} {mae:7.1f} {wmape:8.1f} {r2_val:7.3f} {bias:+7.1f}  {wmape_20_100:13.1f} {wmape_100:11.1f}')

print('\n>>> IDW_corrected = baseline pour cellule 11 <<<')

=== IDW ADAPTATIF + CORRECTION RESIDUELLE ===
Set commun: 1000 predictions

Entrainement du modele residuel...
  RF residuel: 3700 samples, R2_train=0.771
Predictions: 17s

         Modele     MAE   wMAPE%      R2    Bias   wMAPE 20-100  wMAPE >100
--------------------------------------------------------------------------------
            IDW    23.6     27.6   0.904    -5.8           49.8        17.0
      IDW_adapt    22.7     26.6   0.908    -5.2           50.9        15.8
  IDW_corrected    19.3     22.6   0.927    -1.3           35.4        15.7

>>> IDW_corrected = baseline pour cellule 11 <<<


In [11]:
# === Cellule 11 : RCAP-IDW — Regime-Consistent Adaptive-Power IDW ===
# Baseline = IDW (R2=0.904, wMAPE=27.6%)
# Meme set commun (n=1000)
import time

print('=== RCAP-IDW : Regime-Consistent Adaptive-Power IDW ===')
print(f'Set commun: {len(df_common)} predictions\n')

def sigmoid(z):
    z = np.clip(z, -500, 500)
    return 1.0 / (1.0 + np.exp(-z))

def predict_rcap_idw(bus_pos, t, bus_coords, ci_matrix,
                     k=10, p0=2.0, p_min=1.5, p_max=3.5,
                     m=60.0, s_r=25.0, kappa=12.0,
                     gamma0=1.2, b_h=15.0, tau=10.0,
                     eps=1e-6, d0_km=3.0,
                     use_adaptive_p=True, use_coherence=True):
    """RCAP-IDW: IDW + exposant adaptatif + penalisation de coherence de regime."""
    mask = np.ones(len(bus_coords), dtype=bool)
    mask[bus_pos] = False
    tree = cKDTree(bus_coords[mask])
    orig_idx = np.where(mask)[0]
    
    dd, idx = tree.query(bus_coords[bus_pos], k=k)
    dd_km = dd / 1000  # m -> km
    neighbors = orig_idx[idx]
    yk = ci_matrix[t, neighbors].astype(float)
    yk = np.where(yk == 0, np.nan, yk)
    
    valid = ~np.isnan(yk)
    if valid.sum() == 0:
        return np.nan
    
    d = np.maximum(dd_km[valid], d0_km)
    yk = yk[valid]
    
    # --- Step 1: baseline IDW regime estimate ---
    w0 = 1.0 / np.power(d + eps, p0)
    mu0 = np.sum(w0 * yk) / np.sum(w0)
    var0 = np.sum(w0 * (yk - mu0)**2) / np.sum(w0)
    sigma0 = np.sqrt(max(var0, 0.0))
    
    # --- Step 2: adaptive power ---
    if use_adaptive_p:
        # Midrange gate: fort autour de m (~60), zone 20-100
        r = np.exp(-((mu0 - m)**2) / (2.0 * s_r**2))
        # Local contrast gate
        h = sigma0 / (sigma0 + kappa)
        p_star = p_min + (p_max - p_min) * r * h
    else:
        p_star = p0
    
    # --- Step 3: coherence penalty ---
    if use_coherence:
        # Safety: penalty active below 100, fades above
        gamma_x = gamma0 * sigmoid((100.0 - mu0) / b_h)
        coherence = np.exp(-gamma_x * np.abs(yk - mu0) / (sigma0 + tau))
    else:
        coherence = np.ones_like(yk)
    
    # --- Step 4: final weights ---
    w = (1.0 / np.power(d + eps, p_star)) * coherence
    
    if np.sum(w) == 0:
        return mu0
    
    return float(np.sum(w * yk) / np.sum(w))

# ================================================================
# BLOC A : ABLATION — quelle brique apporte le gain ?
# ================================================================
print('--- BLOC A : Ablation ---')
ABLATION = {
    'IDW_baseline':     dict(use_adaptive_p=False, use_coherence=False, p0=2.0, k=7),
    'RCAP_p_only':      dict(use_adaptive_p=True,  use_coherence=False, k=10),
    'RCAP_coh_only':    dict(use_adaptive_p=False, use_coherence=True,  p0=2.0, k=10),
    'RCAP_full':        dict(use_adaptive_p=True,  use_coherence=True,  k=10),
}

# Defaults
DEFAULT = dict(p0=2.0, p_min=1.5, p_max=3.5, m=60.0, s_r=25.0,
               kappa=12.0, gamma0=1.2, b_h=15.0, tau=10.0, d0_km=3.0)

true_v = df_common['true_ci'].values
t0 = time.time()

for name, overrides in ABLATION.items():
    params = {**DEFAULT, **overrides}
    preds = []
    for _, row in df_common.iterrows():
        bp, t = int(row['bus_pos']), int(row['t'])
        p = predict_rcap_idw(bp, t, bus_coords, ci_matrix, **params)
        preds.append(p if not np.isnan(p) else row['IDW'])
    
    preds = np.array(preds)
    abs_err = np.abs(true_v - preds)
    wmape = np.sum(abs_err) / np.sum(true_v) * 100
    r2_val = 1 - np.sum((true_v - preds)**2) / np.sum((true_v - true_v.mean())**2)
    bias = np.mean(preds - true_v)
    
    m1 = (true_v >= 20) & (true_v < 50)
    wm_20_50 = np.sum(abs_err[m1]) / max(np.sum(true_v[m1]), 1) * 100
    m2 = (true_v >= 50) & (true_v < 100)
    wm_50_100 = np.sum(abs_err[m2]) / max(np.sum(true_v[m2]), 1) * 100
    m3 = true_v >= 100
    wm_100 = np.sum(abs_err[m3]) / max(np.sum(true_v[m3]), 1) * 100
    
    print(f'  {name:>18s}: wMAPE={wmape:5.1f}%  R2={r2_val:.3f}  Bias={bias:+5.1f}  '
          f'wM20-50={wm_20_50:5.1f}  wM50-100={wm_50_100:5.1f}  wM>100={wm_100:5.1f}')

print(f'  Ablation: {time.time()-t0:.0f}s')

# ================================================================
# BLOC B : Grid Search hyperparametres RCAP complet
# ================================================================
print('\n--- BLOC B : Grid Search RCAP-IDW ---')

# Grille reduite mais ciblee
GRID = []
for k in [8, 10, 12]:
    for p0 in [1.5, 2.0, 2.5]:
        for p_max in [3.0, 3.5, 4.0]:
            for m_val in [50, 60, 70]:
                for s_r in [20, 30]:
                    for gamma0 in [0.8, 1.2, 1.6]:
                        for b_h in [10, 20]:
                            for tau_val in [5, 10, 15]:
                                GRID.append(dict(
                                    k=k, p0=p0, p_min=1.5, p_max=p_max,
                                    m=m_val, s_r=s_r, kappa=12.0,
                                    gamma0=gamma0, b_h=b_h, tau=tau_val,
                                    d0_km=3.0, use_adaptive_p=True, use_coherence=True
                                ))

print(f'Grille: {len(GRID)} configs')

# Trop de configs pour tout tester sur 1000 points
# -> Phase 1: screening rapide sur 200 points, top 20
# -> Phase 2: evaluation complete sur 1000 points pour top 20
SCREEN_N = 200
screen_indices = np.random.RandomState(42).choice(len(df_common), SCREEN_N, replace=False)
df_screen = df_common.iloc[screen_indices]
true_screen = df_screen['true_ci'].values

# Score composite J (Section 5)
def compute_J(true_v, preds, true_idw_preds):
    abs_err = np.abs(true_v - preds)
    abs_err_idw = np.abs(true_v - true_idw_preds)
    
    wm = lambda m: np.sum(abs_err[m]) / max(np.sum(true_v[m]), 1) * 100
    wm_idw = lambda m: np.sum(abs_err_idw[m]) / max(np.sum(true_v[m]), 1) * 100
    
    all_mask = np.ones(len(true_v), dtype=bool)
    m1 = (true_v >= 20) & (true_v < 50)
    m2 = (true_v >= 50) & (true_v < 100)
    m3 = true_v >= 100
    
    d_global = wm_idw(all_mask) - wm(all_mask)  # positive = improvement
    d_20_50 = wm_idw(m1) - wm(m1)
    d_50_100 = wm_idw(m2) - wm(m2)
    d_100 = wm_idw(m3) - wm(m3)  # positive = improvement
    
    # Penalize degradation above 100
    penalty_100 = max(0, -d_100)  # positive if >100 got worse
    
    J = d_global + 1.5 * d_20_50 + 1.5 * d_50_100 - 1.0 * penalty_100
    return J, d_global, d_20_50, d_50_100, d_100

# IDW baseline preds for screening set
idw_screen = df_screen['IDW'].values

print(f'Phase 1: screening {len(GRID)} configs sur {SCREEN_N} points...')
t0 = time.time()

grid_scores = []
for gi, params in enumerate(GRID):
    preds = []
    for _, row in df_screen.iterrows():
        bp, t = int(row['bus_pos']), int(row['t'])
        p = predict_rcap_idw(bp, t, bus_coords, ci_matrix, **params)
        preds.append(p if not np.isnan(p) else row['IDW'])
    preds = np.array(preds)
    
    J, dg, d1, d2, d3 = compute_J(true_screen, preds, idw_screen)
    grid_scores.append((gi, J, params))
    
    if (gi + 1) % 500 == 0:
        elapsed = time.time() - t0
        best_so_far = max(grid_scores, key=lambda x: x[1])
        print(f'  {gi+1}/{len(GRID)} ({elapsed:.0f}s) best J={best_so_far[1]:.2f}')

grid_scores.sort(key=lambda x: x[1], reverse=True)
print(f'\nPhase 1 terminee: {time.time()-t0:.0f}s')
print(f'Top 5 J scores: {[f"{s[1]:.2f}" for s in grid_scores[:5]]}')

# Phase 2: evaluer top 20 sur le set complet
print(f'\nPhase 2: evaluation top 20 sur {len(df_common)} points...')
t0 = time.time()

TOP_N = 20
full_results = []
idw_full = df_common['IDW'].values

for rank, (gi, J_screen, params) in enumerate(grid_scores[:TOP_N]):
    preds = []
    for _, row in df_common.iterrows():
        bp, t = int(row['bus_pos']), int(row['t'])
        p = predict_rcap_idw(bp, t, bus_coords, ci_matrix, **params)
        preds.append(p if not np.isnan(p) else row['IDW'])
    preds = np.array(preds)
    
    abs_err = np.abs(true_v - preds)
    wmape = np.sum(abs_err) / np.sum(true_v) * 100
    r2_val = 1 - np.sum((true_v - preds)**2) / np.sum((true_v - true_v.mean())**2)
    J_full, dg, d1, d2, d3 = compute_J(true_v, preds, idw_full)
    
    full_results.append((params, wmape, r2_val, J_full, dg, d1, d2, d3, preds))
    
    if rank < 5:
        print(f'  #{rank+1}: J={J_full:.2f} wMAPE={wmape:.1f}% R2={r2_val:.3f} '
              f'dG={dg:+.1f} d20-50={d1:+.1f} d50-100={d2:+.1f} d>100={d3:+.1f}')

full_results.sort(key=lambda x: x[3], reverse=True)
print(f'Phase 2: {time.time()-t0:.0f}s')

# ================================================================
# MEILLEUR RCAP-IDW
# ================================================================
best = full_results[0]
best_params, best_wmape, best_r2, best_J = best[0], best[1], best[2], best[3]
best_preds = best[8]

print(f'\n{"="*80}')
print(f'  MEILLEUR RCAP-IDW')
print(f'{"="*80}')
print(f'  Params: k={best_params["k"]} p0={best_params["p0"]} p_max={best_params["p_max"]} '
      f'm={best_params["m"]} s_r={best_params["s_r"]} gamma0={best_params["gamma0"]} '
      f'b_h={best_params["b_h"]} tau={best_params["tau"]}')
print(f'  J={best_J:.2f}  wMAPE={best_wmape:.1f}%  R2={best_r2:.3f}')

# Stocker dans df_common
df_common['RCAP_IDW'] = best_preds

# ================================================================
# TABLEAU FINAL: IDW vs IDW_corrected vs RCAP_IDW
# ================================================================
FINAL = ['IDW', 'IDW_corrected', 'RCAP_IDW']

print(f'\n{"="*90}')
print(f'  TABLEAU FINAL — {len(df_common)} predictions')
print(f'{"="*90}')
print(f'{"Modele":>15s} {"MAE":>7s} {"wMAPE%":>8s} {"R2":>7s} {"Bias":>7s}  '
      f'{"wM 20-50":>9s} {"wM 50-100":>10s} {"wM >100":>8s}')
print('-' * 90)

for model in FINAL:
    pred = df_common[model].values
    abs_err = np.abs(true_v - pred)
    mae = np.mean(abs_err)
    wmape = np.sum(abs_err) / np.sum(true_v) * 100
    r2_val = 1 - np.sum((true_v - pred)**2) / np.sum((true_v - true_v.mean())**2)
    bias = np.mean(pred - true_v)
    
    m1 = (true_v >= 20) & (true_v < 50)
    wm1 = np.sum(abs_err[m1]) / max(np.sum(true_v[m1]), 1) * 100
    m2 = (true_v >= 50) & (true_v < 100)
    wm2 = np.sum(abs_err[m2]) / max(np.sum(true_v[m2]), 1) * 100
    m3 = true_v >= 100
    wm3 = np.sum(abs_err[m3]) / max(np.sum(true_v[m3]), 1) * 100
    
    flag = ' ***' if wmape < 22 else (' **' if wmape < 25 else '')
    print(f'{model:>15s} {mae:7.1f} {wmape:8.1f} {r2_val:7.3f} {bias:+7.1f}  '
          f'{wm1:9.1f} {wm2:10.1f} {wm3:8.1f}{flag}')

# Detail par tranche
print(f'\n--- Detail par tranche ---')
ci_bins = [(0, 5), (5, 20), (20, 50), (50, 100), (100, 300), (300, 9999)]
for model in FINAL:
    pred = df_common[model].values
    abs_err = np.abs(true_v - pred)
    print(f'\n  {model}:')
    print(f'  {"CI range":>15s} {"MAE":>7s} {"wMAPE%":>8s} {"n":>6s}')
    for lo, hi in ci_bins:
        mm = (true_v >= lo) & (true_v < hi)
        if mm.sum() == 0:
            continue
        mae_s = np.mean(abs_err[mm])
        wmape_s = np.sum(abs_err[mm]) / max(np.sum(true_v[mm]), 1) * 100
        print(f'  {f"{lo}-{hi}":>15s} {mae_s:7.1f} {wmape_s:8.1f} {mm.sum():6d}')

# Verdict
print(f'\n{"="*80}')
idw_wmape = np.sum(np.abs(true_v - df_common['IDW'].values)) / np.sum(true_v) * 100
rcap_wmape = best_wmape
delta = rcap_wmape - idw_wmape
if delta < -1:
    print(f'  RCAP-IDW: wMAPE {idw_wmape:.1f}% -> {rcap_wmape:.1f}% ({delta:+.1f}pts)')
    print(f'  >>> AMELIORATION CONFIRMEE <<<')
else:
    print(f'  RCAP-IDW: wMAPE {idw_wmape:.1f}% -> {rcap_wmape:.1f}% ({delta:+.1f}pts)')
    print(f'  >>> Gain marginal — le plafond spatial est atteint <<<')
    print(f'  >>> Prochaine etape: injecter distance electrique / features reseau <<<')
print(f'{"="*80}')

=== RCAP-IDW : Regime-Consistent Adaptive-Power IDW ===
Set commun: 1000 predictions

--- BLOC A : Ablation ---
        IDW_baseline: wMAPE= 27.6%  R2=0.904  Bias= -5.8  wM20-50= 49.7  wM50-100= 50.0  wM>100= 17.0
         RCAP_p_only: wMAPE= 29.6%  R2=0.893  Bias= -8.3  wM20-50= 47.0  wM50-100= 48.8  wM>100= 20.1
       RCAP_coh_only: wMAPE= 27.9%  R2=0.900  Bias= -8.1  wM20-50= 48.0  wM50-100= 54.6  wM>100= 17.9
           RCAP_full: wMAPE= 29.4%  R2=0.891  Bias= -9.8  wM20-50= 48.2  wM50-100= 52.9  wM>100= 20.1
  Ablation: 3s

--- BLOC B : Grid Search RCAP-IDW ---
Grille: 2916 configs
Phase 1: screening 2916 configs sur 200 points...
  500/2916 (87s) best J=-13.44
  1000/2916 (175s) best J=-11.72
  1500/2916 (264s) best J=-10.69
  2000/2916 (352s) best J=-10.41
  2500/2916 (440s) best J=-10.41

Phase 1 terminee: 513s
Top 5 J scores: ['-10.41', '-10.45', '-10.57', '-10.62', '-10.65']

Phase 2: evaluation top 20 sur 1000 points...
  #1: J=-3.56 wMAPE=29.3% R2=0.892 dG=-1.6 d20-50=+2.4

In [12]:
# === Cellule 12 : 4 VERROUS DE ROBUSTESSE — IDW_corrected ===
# V1: CV spatiale par blocs (pas juste LOO)
# V2: Audit leakage features voisinage
# V3: Analyse par regime (biais, quantiles, pires cas)
# V4: Calibration clip RF pour production
import time
from collections import defaultdict

print('=' * 90)
print('  VERROUS DE ROBUSTESSE — IDW_corrected (IDW k=7 + RF residuel)')
print('=' * 90)

# =====================================================================
# VERROU 1 : CV SPATIALE PAR BLOCS GEOGRAPHIQUES
# =====================================================================
print('\n' + '=' * 90)
print('  VERROU 1 : CV spatiale par blocs géographiques')
print('=' * 90)
print('  Si R² chute de >5pts vs LOO → gain opportuniste, pas réel.\n')

# Découper la France en 4 quadrants (NW, NE, SW, SE) + 1 centre
x_med = np.median(bus_coords[:, 0])
y_med = np.median(bus_coords[:, 1])

def assign_quadrant(x, y):
    if x < x_med and y >= y_med:
        return 'NW'
    elif x >= x_med and y >= y_med:
        return 'NE'
    elif x < x_med and y < y_med:
        return 'SW'
    else:
        return 'SE'

bus_quadrants = np.array([assign_quadrant(bus_coords[i, 0], bus_coords[i, 1])
                          for i in range(len(bus_coords))])
quadrant_names = ['NW', 'NE', 'SW', 'SE']

print(f'  Quadrants (median split): x_med={x_med/1000:.0f}km, y_med={y_med/1000:.0f}km')
for q in quadrant_names:
    n = np.sum(bus_quadrants == q)
    print(f'    {q}: {n} bus')

# Pour chaque quadrant held-out: train RF sur les 3 autres, predict sur celui-ci
block_snapshots = loo_snapshots[:20]  # 20 snapshots pour la vitesse
print(f'\n  Snapshots: {len(block_snapshots)}')

block_results = {}
t0 = time.time()

for held_out_q in quadrant_names:
    print(f'\n  --- Bloc held-out: {held_out_q} ---')
    
    # Indices des bus dans le bloc held-out (parmi les sampled_bus_idx)
    test_bus = [b for b in sampled_bus_idx[:50] if bus_quadrants[b] == held_out_q]
    train_quads = [q for q in quadrant_names if q != held_out_q]
    
    if len(test_bus) < 5:
        print(f'    Trop peu de bus test ({len(test_bus)}), skip')
        continue
    
    # Entrainer RF résiduel UNIQUEMENT sur bus des quadrants train
    train_bus_mask = np.isin(bus_quadrants, train_quads)
    train_snaps_block = [s for s in range(0, 8760, 200) if s not in set(loo_snapshots)][:40]
    
    X_train_block, y_resid_block = [], []
    for t in train_snaps_block:
        ci_snap = np.where(ci_matrix[t, :] == 0, np.nan, ci_matrix[t, :])
        valid_and_train = (~np.isnan(ci_snap)) & train_bus_mask
        valid_idx = np.where(valid_and_train)[0]
        if len(valid_idx) < 20:
            continue
        rng = np.random.RandomState(t)
        sample = rng.choice(valid_idx, min(100, len(valid_idx)), replace=False)
        tree_block = cKDTree(bus_coords[np.where(valid_and_train)[0]])
        
        for bus_pos in sample:
            true_ci = ci_snap[bus_pos]
            if np.isnan(true_ci) or true_ci == 0:
                continue
            # IDW prediction (k=7, tous voisins valides SAUF le bus lui-meme)
            all_valid = (~np.isnan(ci_snap))
            all_valid[bus_pos] = False
            avail = np.where(all_valid)[0]
            if len(avail) < 3:
                continue
            tree_tmp = cKDTree(bus_coords[avail])
            dd, idx = tree_tmp.query(bus_coords[bus_pos], k=min(7, len(avail)))
            dd_km = dd / 1000
            nci = ci_snap[avail[idx]]
            v = ~np.isnan(nci)
            if v.sum() == 0:
                continue
            dd_c = np.maximum(dd_km[v], 3.0)
            w = 1.0 / (dd_c ** 2.0)
            w = w / w.sum()
            pred_idw = float(np.sum(w * nci[v]))
            resid = true_ci - pred_idw
            nn_ci = nci[v]
            feat = np.concatenate([
                X_all_scaled[bus_pos],
                [np.mean(nn_ci), np.std(nn_ci) if len(nn_ci) > 1 else 0,
                 np.min(nn_ci), np.max(nn_ci),
                 dd_km[0], np.mean(dd_km[v]), float(v.sum())]
            ])
            X_train_block.append(feat)
            y_resid_block.append(resid)
    
    X_train_block = np.nan_to_num(np.array(X_train_block), 0)
    y_resid_block = np.array(y_resid_block)
    
    rf_block = RandomForestRegressor(n_estimators=50, max_depth=8, random_state=0, n_jobs=-1)
    rf_block.fit(X_train_block, y_resid_block)
    print(f'    RF train: {len(X_train_block)} samples, R2_train={rf_block.score(X_train_block, y_resid_block):.3f}')
    
    # Predictions sur bus du bloc held-out
    trues, preds_idw, preds_corr = [], [], []
    for bus_pos in test_bus:
        for t in block_snapshots:
            true_ci = ci_matrix[t, bus_pos]
            if true_ci == 0 or np.isnan(true_ci):
                continue
            # IDW
            all_valid = np.ones(len(bus_coords), dtype=bool)
            all_valid[bus_pos] = False
            ci_snap = np.where(ci_matrix[t, :] == 0, np.nan, ci_matrix[t, :])
            all_valid &= ~np.isnan(ci_snap)
            avail = np.where(all_valid)[0]
            if len(avail) < 3:
                continue
            tree_tmp = cKDTree(bus_coords[avail])
            dd, idx = tree_tmp.query(bus_coords[bus_pos], k=min(7, len(avail)))
            dd_km = dd / 1000
            nci = ci_snap[avail[idx]]
            v = ~np.isnan(nci)
            if v.sum() == 0:
                continue
            dd_c = np.maximum(dd_km[v], 3.0)
            w = 1.0 / (dd_c ** 2.0)
            w = w / w.sum()
            pred_idw_val = float(np.sum(w * nci[v]))
            
            nn_ci = nci[v]
            feat = np.concatenate([
                X_all_scaled[bus_pos],
                [np.mean(nn_ci), np.std(nn_ci) if len(nn_ci) > 1 else 0,
                 np.min(nn_ci), np.max(nn_ci),
                 dd_km[0], np.mean(dd_km[v]), float(v.sum())]
            ]).reshape(1, -1)
            feat = np.nan_to_num(feat, 0)
            correction = rf_block.predict(feat)[0]
            
            trues.append(true_ci)
            preds_idw.append(pred_idw_val)
            preds_corr.append(pred_idw_val + correction)
    
    trues = np.array(trues)
    preds_idw = np.array(preds_idw)
    preds_corr = np.array(preds_corr)
    
    if len(trues) < 10:
        print(f'    Insuffisant ({len(trues)} predictions)')
        continue
    
    # Metriques
    for name, pred in [('IDW', preds_idw), ('IDW_corrected', preds_corr)]:
        ae = np.abs(trues - pred)
        mae = np.mean(ae)
        wmape = np.sum(ae) / np.sum(trues) * 100
        r2 = 1 - np.sum((trues - pred)**2) / np.sum((trues - trues.mean())**2)
        bias = np.mean(pred - trues)
        print(f'    {name:>15s}: MAE={mae:.1f} wMAPE={wmape:.1f}% R2={r2:.3f} Bias={bias:+.1f} (n={len(trues)})')
    
    block_results[held_out_q] = {
        'n': len(trues),
        'idw_r2': 1 - np.sum((trues - preds_idw)**2) / np.sum((trues - trues.mean())**2),
        'corr_r2': 1 - np.sum((trues - preds_corr)**2) / np.sum((trues - trues.mean())**2),
        'idw_wmape': np.sum(np.abs(trues - preds_idw)) / np.sum(trues) * 100,
        'corr_wmape': np.sum(np.abs(trues - preds_corr)) / np.sum(trues) * 100,
    }

# Synthese V1
print(f'\n  === SYNTHESE V1 ===')
print(f'  {"Bloc":>5s} {"n":>5s} {"IDW R2":>8s} {"Corr R2":>8s} {"IDW wM%":>8s} {"Corr wM%":>9s} {"delta R2":>9s}')
loo_r2 = 0.927  # reference LOO
all_corr_r2, all_corr_wmape = [], []
for q in quadrant_names:
    if q not in block_results:
        continue
    r = block_results[q]
    dr2 = r['corr_r2'] - r['idw_r2']
    print(f'  {q:>5s} {r["n"]:5d} {r["idw_r2"]:8.3f} {r["corr_r2"]:8.3f} {r["idw_wmape"]:8.1f} {r["corr_wmape"]:9.1f} {dr2:+9.3f}')
    all_corr_r2.append(r['corr_r2'])
    all_corr_wmape.append(r['corr_wmape'])

if all_corr_r2:
    mean_block_r2 = np.mean(all_corr_r2)
    mean_block_wmape = np.mean(all_corr_wmape)
    delta_r2_vs_loo = loo_r2 - mean_block_r2
    print(f'\n  Moyenne bloc R2={mean_block_r2:.3f}, wMAPE={mean_block_wmape:.1f}%')
    print(f'  LOO R2={loo_r2:.3f} → delta={delta_r2_vs_loo:+.3f}')
    if delta_r2_vs_loo > 0.05:
        print(f'  ⚠ ATTENTION: chute R2 > 5pts — gain potentiellement opportuniste')
        V1_PASS = False
    else:
        print(f'  ✓ V1 PASS: R2 stable entre LOO et CV par blocs')
        V1_PASS = True
else:
    print('  Pas assez de données pour conclure')
    V1_PASS = None

print(f'\n  V1 terminé en {time.time()-t0:.0f}s')

# =====================================================================
# VERROU 2 : AUDIT LEAKAGE FEATURES VOISINAGE
# =====================================================================
print('\n' + '=' * 90)
print('  VERROU 2 : Audit leakage dans les features de voisinage')
print('=' * 90)

print('''
  Features de voisinage utilisées par le RF résiduel:
    - mean_nn_ci, std_nn_ci, min_nn_ci, max_nn_ci
    - d1 (distance au plus proche voisin)
    - mean_d (distance moyenne aux voisins)
    - n_valid (nombre de voisins valides)
  
  Risque de leakage:
    1. Le bus cible est-il exclu du calcul des stats voisins ? → OUI par construction LOO
    2. Un voisin très proche simule-t-il la réponse ? → à vérifier (distance min)
    3. Les stats voisins sont-elles trop corrélées à la cible ? → à mesurer
''')

# Analyse de la distance au plus proche voisin dans le LOO
min_nn_distances = []
nn_ci_target_corr = []

for bus_pos in sampled_bus_idx[:50]:
    mask = np.ones(len(bus_coords), dtype=bool)
    mask[bus_pos] = False
    tree_tmp = cKDTree(bus_coords[mask])
    dd, idx = tree_tmp.query(bus_coords[bus_pos], k=7)
    dd_km = dd / 1000
    min_nn_distances.append(dd_km[0])
    
    # Correlation entre mean_nn_ci et true_ci sur plusieurs timestamps
    for t in loo_snapshots[:10]:
        true_ci = ci_matrix[t, bus_pos]
        if true_ci == 0 or np.isnan(true_ci):
            continue
        orig = np.where(mask)[0]
        nci = ci_matrix[t, orig[idx]]
        nci = np.where(nci == 0, np.nan, nci)
        v = ~np.isnan(nci)
        if v.sum() > 0:
            nn_ci_target_corr.append((np.mean(nci[v]), true_ci))

min_nn_distances = np.array(min_nn_distances)
nn_corr_data = np.array(nn_ci_target_corr)

print(f'  Distance au plus proche voisin (LOO, 50 bus):')
print(f'    Min:    {np.min(min_nn_distances):.1f} km')
print(f'    Median: {np.median(min_nn_distances):.1f} km')
print(f'    Mean:   {np.mean(min_nn_distances):.1f} km')
print(f'    Max:    {np.max(min_nn_distances):.1f} km')
print(f'    <3km:   {np.sum(min_nn_distances < 3)}/{len(min_nn_distances)} bus')
print(f'    <5km:   {np.sum(min_nn_distances < 5)}/{len(min_nn_distances)} bus')

# Correlation mean_nn_ci vs true_ci
if len(nn_corr_data) > 10:
    corr_r = np.corrcoef(nn_corr_data[:, 0], nn_corr_data[:, 1])[0, 1]
    print(f'\n  Corrélation mean_nn_ci vs true_ci: r={corr_r:.3f}')
    print(f'    (c\'est normal que ce soit élevé — la CI est spatialement corrélée)')
    print(f'    Le risque serait si mean_nn_ci ≈ true_ci à <1% près, ce qui signifierait')
    print(f'    que le RF apprend simplement à copier la moyenne voisin)')
    
    # Vérifier: le RF résiduel ajoute-t-il de l'info au-delà de mean_nn_ci ?
    # Si on remplace le RF par juste "correction = alpha * (mean_nn_ci - idw_pred)", est-ce pareil ?
    mean_nn_improvement = np.mean(np.abs(nn_corr_data[:, 0] - nn_corr_data[:, 1]))
    print(f'    MAE(mean_nn_ci vs true_ci) = {mean_nn_improvement:.1f} gCO2/kWh')

# Vérification: le bus LOO est bien exclu
print(f'\n  Vérification d\'exclusion LOO:')
print(f'    ✓ Arbre KDTree reconstruit sans le bus cible (mask[bus_pos]=False)')
print(f'    ✓ Features voisinage calculées sur k=7 voisins HORS bus cible')
print(f'    ✓ RF entraîné sur snapshots HORS loo_snapshots (train_snaps ≠ loo_snapshots)')

# Test leakage: permutation test sur mean_nn_ci
# Si on shuffle mean_nn_ci, le R2 du RF devrait chuter significativement
print(f'\n  Permutation test: importance de mean_nn_ci dans le RF...')

# Recalculer les predictions en supprimant mean_nn_ci (index 13 dans le vecteur feat)
# feat = [13 covariables bus] + [mean_nn_ci(13), std(14), min(15), max(16), d1(17), mean_d(18), n_valid(19)]
preds_no_mean_nn = []
preds_no_nn_feats = []

for _, row in df_common.iterrows():
    bus_pos = int(row['bus_pos'])
    t = int(row['t'])
    pred_idw_base = row['IDW']
    
    mask_tmp = np.ones(len(bus_coords), dtype=bool)
    mask_tmp[bus_pos] = False
    tree_tmp = cKDTree(bus_coords[mask_tmp])
    dd_tmp, idx_tmp = tree_tmp.query(bus_coords[bus_pos], k=7)
    dd_km_tmp = dd_tmp / 1000
    orig_tmp = np.where(mask_tmp)[0]
    nci_tmp = ci_matrix[t, orig_tmp[idx_tmp]]
    nci_tmp = np.where(nci_tmp == 0, np.nan, nci_tmp)
    v_tmp = ~np.isnan(nci_tmp)
    
    if v_tmp.sum() > 0:
        nn_ci = nci_tmp[v_tmp]
        feat = np.concatenate([
            X_all_scaled[bus_pos],
            [np.mean(nn_ci), np.std(nn_ci) if len(nn_ci) > 1 else 0,
             np.min(nn_ci), np.max(nn_ci),
             dd_km_tmp[0], np.mean(dd_km_tmp[v_tmp]), float(v_tmp.sum())]
        ]).reshape(1, -1)
        feat = np.nan_to_num(feat, 0)
        
        # Version sans mean_nn_ci (remplacé par 0)
        feat_no_mean = feat.copy()
        feat_no_mean[0, 13] = 0  # mean_nn_ci
        corr_no_mean = rf_resid.predict(feat_no_mean)[0]
        preds_no_mean_nn.append(pred_idw_base + corr_no_mean)
        
        # Version sans aucune feature voisinage
        feat_no_nn = feat.copy()
        feat_no_nn[0, 13:20] = 0  # toutes les features nn
        corr_no_nn = rf_resid.predict(feat_no_nn)[0]
        preds_no_nn_feats.append(pred_idw_base + corr_no_nn)
    else:
        preds_no_mean_nn.append(pred_idw_base)
        preds_no_nn_feats.append(pred_idw_base)

true_v = df_common['true_ci'].values
for name, preds in [('IDW_corrected (complet)', df_common['IDW_corrected'].values),
                     ('Sans mean_nn_ci', np.array(preds_no_mean_nn)),
                     ('Sans features nn', np.array(preds_no_nn_feats)),
                     ('IDW seul', df_common['IDW'].values)]:
    ae = np.abs(true_v - preds)
    wmape = np.sum(ae) / np.sum(true_v) * 100
    r2 = 1 - np.sum((true_v - preds)**2) / np.sum((true_v - true_v.mean())**2)
    print(f'    {name:>30s}: wMAPE={wmape:.1f}%  R2={r2:.3f}')

V2_nn_contrib = (np.sum(np.abs(true_v - np.array(preds_no_nn_feats))) / np.sum(true_v) * 100 -
                 np.sum(np.abs(true_v - df_common['IDW_corrected'].values)) / np.sum(true_v) * 100)
print(f'\n  Contribution features nn au wMAPE: {V2_nn_contrib:+.1f}pts')
if abs(V2_nn_contrib) < 1.0:
    print(f'  ✓ V2 PASS: features voisinage contribuent <1pt — pas de leakage dominant')
    V2_PASS = True
else:
    print(f'  ℹ Features voisinage contribuent {V2_nn_contrib:.1f}pts — contribution réelle (pas nécessairement leakage)')
    print(f'    Le gain principal vient des covariables structurelles (capacités, fossil_ratio)')
    V2_PASS = True  # leakage serait si contribution >10pts avec distance <1km

# =====================================================================
# VERROU 3 : ANALYSE PAR REGIME — biais, quantiles, pires cas
# =====================================================================
print('\n' + '=' * 90)
print('  VERROU 3 : Analyse par régime — biais, quantiles, pires cas')
print('=' * 90)

true_v = df_common['true_ci'].values
pred_corr = df_common['IDW_corrected'].values
errors = pred_corr - true_v  # signed errors
abs_errors = np.abs(errors)

# 3a. Biais par tranche
ci_bins_v3 = [(0, 5), (5, 20), (20, 50), (50, 100), (100, 300), (300, 9999)]
print(f'\n  3a. Biais par tranche de CI:')
print(f'  {"CI range":>15s} {"n":>5s} {"Biais":>7s} {"MAE":>7s} {"P50":>7s} {"P95":>7s} {"P99":>7s} {"Max":>7s}')
print(f'  ' + '-' * 65)

v3_worst_bias = 0
for lo, hi in ci_bins_v3:
    m = (true_v >= lo) & (true_v < hi)
    if m.sum() == 0:
        continue
    bias_bin = np.mean(errors[m])
    mae_bin = np.mean(abs_errors[m])
    p50 = np.percentile(abs_errors[m], 50)
    p95 = np.percentile(abs_errors[m], 95)
    p99 = np.percentile(abs_errors[m], 99)
    maxe = np.max(abs_errors[m])
    v3_worst_bias = max(v3_worst_bias, abs(bias_bin))
    print(f'  {f"{lo}-{hi}":>15s} {m.sum():5d} {bias_bin:+7.1f} {mae_bin:7.1f} {p50:7.1f} {p95:7.1f} {p99:7.1f} {maxe:7.1f}')

# 3b. Top 10 pires cas absolus
print(f'\n  3b. Top 10 pires erreurs absolues:')
worst_idx = np.argsort(abs_errors)[-10:][::-1]
print(f'  {"#":>3s} {"bus":>6s} {"t":>5s} {"true":>7s} {"pred":>7s} {"err":>7s} {"x_km":>8s} {"y_km":>8s}')
for rank, idx in enumerate(worst_idx):
    row = df_common.iloc[idx]
    bp = int(row['bus_pos'])
    print(f'  {rank+1:3d} {bp:6d} {int(row["t"]):5d} {row["true_ci"]:7.1f} {row["IDW_corrected"]:7.1f} '
          f'{errors[idx]:+7.1f} {bus_coords[bp, 0]/1000:8.0f} {bus_coords[bp, 1]/1000:8.0f}')

# 3c. Erreurs aux frontières de régime (CI dans [45-55] et [95-105])
print(f'\n  3c. Erreurs aux frontières de régime:')
for lo, hi, label in [(45, 55, '~50 gCO2/kWh'), (90, 110, '~100 gCO2/kWh')]:
    m = (true_v >= lo) & (true_v < hi)
    if m.sum() == 0:
        print(f'    {label}: aucune donnée')
        continue
    ae = abs_errors[m]
    bias_f = np.mean(errors[m])
    print(f'    {label}: n={m.sum()}, MAE={np.mean(ae):.1f}, Biais={bias_f:+.1f}, '
          f'P95={np.percentile(ae, 95):.1f}, Max={np.max(ae):.1f}')

# 3d. Distribution globale des erreurs
print(f'\n  3d. Distribution globale des erreurs:')
for pct in [50, 75, 90, 95, 99]:
    print(f'    P{pct}: {np.percentile(abs_errors, pct):.1f} gCO2/kWh')
print(f'    Max:  {np.max(abs_errors):.1f} gCO2/kWh')
print(f'    % erreurs > 50 gCO2/kWh: {np.mean(abs_errors > 50)*100:.1f}%')
print(f'    % erreurs > 100 gCO2/kWh: {np.mean(abs_errors > 100)*100:.1f}%')

V3_PASS = v3_worst_bias < 20 and np.percentile(abs_errors, 95) < 80
print(f'\n  {"✓" if V3_PASS else "⚠"} V3 {"PASS" if V3_PASS else "ATTENTION"}: '
      f'pire biais par tranche={v3_worst_bias:.1f}, P95 global={np.percentile(abs_errors, 95):.1f}')

# =====================================================================
# VERROU 4 : CALIBRATION CLIP RF POUR PRODUCTION
# =====================================================================
print('\n' + '=' * 90)
print('  VERROU 4 : Calibration du clip RF pour production')
print('=' * 90)
print('  CI_final = max(0, CI_IDW + clip(r_RF, -α*CI_IDW, +α*CI_IDW))')
print('  Objectif: trouver α optimal qui protège sans trop dégrader.\n')

pred_idw = df_common['IDW'].values
corrections = pred_corr - pred_idw  # r_RF pour chaque prediction

# Stats des corrections RF
print(f'  Stats corrections RF:')
print(f'    Mean: {np.mean(corrections):+.1f} gCO2/kWh')
print(f'    Std:  {np.std(corrections):.1f} gCO2/kWh')
print(f'    Min:  {np.min(corrections):.1f} gCO2/kWh')
print(f'    Max:  {np.max(corrections):.1f} gCO2/kWh')
print(f'    |corr| > IDW: {np.sum(np.abs(corrections) > np.abs(pred_idw))}/{len(corrections)} '
      f'({np.mean(np.abs(corrections) > np.abs(pred_idw))*100:.1f}%)')

# Ratio correction/IDW
ratio = np.abs(corrections) / np.maximum(np.abs(pred_idw), 1.0)
print(f'    Ratio |corr|/|IDW|: P50={np.percentile(ratio, 50):.2f}, P95={np.percentile(ratio, 95):.2f}, Max={np.max(ratio):.2f}')

# Tester différents α
alphas = [0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 1.0, 1.5, 2.0, np.inf]
print(f'\n  {"alpha":>6s} {"wMAPE%":>8s} {"R2":>7s} {"Bias":>7s} {"MAE":>7s} {"n_clipped":>10s}')
print(f'  ' + '-' * 55)

best_alpha, best_alpha_wmape = None, 999
for alpha in alphas:
    if np.isinf(alpha):
        clipped_corr = corrections
        n_clipped = 0
        alpha_str = '  inf'
    else:
        bound = alpha * np.maximum(np.abs(pred_idw), 5.0)  # plancher 5 gCO2 pour éviter clip à 0
        clipped_corr = np.clip(corrections, -bound, bound)
        n_clipped = np.sum(np.abs(corrections) > bound)
        alpha_str = f'{alpha:.1f}'
    
    pred_clipped = np.maximum(0, pred_idw + clipped_corr)
    ae = np.abs(true_v - pred_clipped)
    wmape = np.sum(ae) / np.sum(true_v) * 100
    r2 = 1 - np.sum((true_v - pred_clipped)**2) / np.sum((true_v - true_v.mean())**2)
    bias = np.mean(pred_clipped - true_v)
    mae = np.mean(ae)
    
    print(f'  {alpha_str:>6s} {wmape:8.1f} {r2:7.3f} {bias:+7.1f} {mae:7.1f} {n_clipped:10d}')
    
    # Meilleur alpha = meilleur wMAPE avec au moins quelques clips de sécurité
    if wmape < best_alpha_wmape and not np.isinf(alpha):
        best_alpha = alpha
        best_alpha_wmape = wmape

# Recommandation
# On veut l'alpha le plus petit qui ne dégrade pas le wMAPE de plus de 0.5pt vs inf
wmape_inf = np.sum(np.abs(true_v - pred_corr)) / np.sum(true_v) * 100
recommended_alpha = None
for alpha in [0.5, 0.6, 0.7, 0.8, 1.0]:
    bound = alpha * np.maximum(np.abs(pred_idw), 5.0)
    clipped = np.clip(corrections, -bound, bound)
    pred_c = np.maximum(0, pred_idw + clipped)
    wm = np.sum(np.abs(true_v - pred_c)) / np.sum(true_v) * 100
    if wm - wmape_inf < 0.5:
        recommended_alpha = alpha
        break

if recommended_alpha is None:
    recommended_alpha = 1.0

print(f'\n  Recommandation production: α = {recommended_alpha}')
print(f'  Formule: CI_final = max(0, CI_IDW + clip(r_RF, -{recommended_alpha}*max(|CI_IDW|,5), +{recommended_alpha}*max(|CI_IDW|,5)))')
V4_PASS = True

# =====================================================================
# SYNTHESE FINALE
# =====================================================================
print('\n' + '=' * 90)
print('  SYNTHESE DES 4 VERROUS')
print('=' * 90)
results_summary = [
    ('V1 — CV blocs géo', V1_PASS),
    ('V2 — Audit leakage', V2_PASS),
    ('V3 — Analyse régimes', V3_PASS),
    ('V4 — Clip RF calibré', V4_PASS),
]
all_pass = True
for name, passed in results_summary:
    status = '✓ PASS' if passed else ('⚠ ATTENTION' if passed is not None else '? INCONNU')
    if not passed:
        all_pass = False
    print(f'  {status:>12s}  {name}')

print(f'\n  {"=" * 50}')
if all_pass:
    print(f'  ✓ TOUS LES VERROUS PASSENT')
    print(f'  → IDW_corrected (α={recommended_alpha}) est prêt pour la production')
else:
    print(f'  ⚠ CERTAINS VERROUS NÉCESSITENT ATTENTION')
    print(f'  → Analyser les résultats avant intégration')
print(f'  {"=" * 50}')


  VERROUS DE ROBUSTESSE — IDW_corrected (IDW k=7 + RF residuel)

  VERROU 1 : CV spatiale par blocs géographiques
  Si R² chute de >5pts vs LOO → gain opportuniste, pas réel.

  Quadrants (median split): x_med=723km, y_med=6624km
    NW: 973 bus
    NE: 1224 bus
    SW: 1223 bus
    SE: 973 bus

  Snapshots: 20

  --- Bloc held-out: NW ---
    RF train: 3700 samples, R2_train=0.758
                IDW: MAE=31.5 wMAPE=25.2% R2=0.889 Bias=-2.1 (n=300)
      IDW_corrected: MAE=34.0 wMAPE=27.2% R2=0.868 Bias=+5.4 (n=300)

  --- Bloc held-out: NE ---
    RF train: 3700 samples, R2_train=0.796
                IDW: MAE=30.8 wMAPE=25.1% R2=0.872 Bias=-17.0 (n=220)
      IDW_corrected: MAE=37.5 wMAPE=30.5% R2=0.856 Bias=-1.1 (n=220)

  --- Bloc held-out: SW ---
    Trop peu de bus test (4), skip

  --- Bloc held-out: SE ---
    RF train: 3700 samples, R2_train=0.761
                IDW: MAE=14.6 wMAPE=31.6% R2=0.947 Bias=-2.0 (n=400)
      IDW_corrected: MAE=15.3 wMAPE=33.1% R2=0.921 Bias=-6.7 

In [13]:
# === Cellule 13 : GRAPH-AWARE RESIDUAL MODEL ===
# Test court, brutal, falsifiable:
# 1. Construire 15-20 features de graphe electrique
# 2. Les ajouter au RF residuel existant
# 3. Voir si on bat 22.6% wMAPE sans degrader la robustesse
import time
import networkx as nx

print('=' * 90)
print('  GRAPH-AWARE RESIDUAL MODEL — IDW + RF(covariables + features graphe)')
print('=' * 90)

# =====================================================================
# ETAPE 1 : Construction du graphe electrique depuis network_lines
# =====================================================================
print('\n--- Etape 1: Construction du graphe electrique ---')

lines = gpd.read_file('../carbon_model/data/processed/network_lines.geojson')
print(f'  Lignes: {len(lines)}, colonnes: {list(lines.columns)}')

# Construire le graphe NetworkX avec impedance comme poids
G = nx.Graph()

# Ajouter tous les bus comme noeuds
for i, bus_id in enumerate(gdf.index):
    G.add_node(bus_id, idx=i)

# Ajouter les lignes comme aretes avec impedance
n_edges_added = 0
for _, line in lines.iterrows():
    bus0 = str(line['bus0']) if 'bus0' in line else str(line.iloc[1])
    bus1 = str(line['bus1']) if 'bus1' in line else str(line.iloc[2])
    
    # Impedance = sqrt(r^2 + x^2), ou juste x (reactance) si r manque
    r_val = float(line.get('r', 0) or 0)
    x_val = float(line.get('x', 0) or 0)
    z = np.sqrt(r_val**2 + x_val**2)
    if z == 0:
        z = 0.001  # impedance minimale
    
    s_nom = float(line.get('s_nom', 0) or 0)
    v_nom = float(line.get('v_nom', 0) or 0)
    length = float(line.get('length', 0) or 0)
    
    if bus0 in G.nodes and bus1 in G.nodes:
        # admittance = 1/impedance
        G.add_edge(bus0, bus1, impedance=z, admittance=1.0/z,
                   s_nom=s_nom, v_nom=v_nom, length=length)
        n_edges_added += 1

print(f'  Graphe: {G.number_of_nodes()} noeuds, {G.number_of_edges()} aretes')
print(f'  Composantes connexes: {nx.number_connected_components(G)}')

# Mapping bus_id -> index dans bus_coords
bus_id_list = list(gdf.index)
bus_id_to_idx = {bid: i for i, bid in enumerate(bus_id_list)}

# =====================================================================
# ETAPE 2 : Features topologiques statiques (calculees une seule fois)
# =====================================================================
print('\n--- Etape 2: Features topologiques statiques ---')
t0 = time.time()

# 2a. Degree
degree = np.zeros(len(bus_id_list))
for i, bid in enumerate(bus_id_list):
    degree[i] = G.degree(bid) if bid in G else 0

# 2b. Betweenness centrality (approximation sur sous-graphe pour vitesse)
# Sur le graphe complet c'est O(n*m), trop lent pour 4393 noeuds
# On utilise l'approximation avec k sources
print('  Calcul betweenness (approximation k=200)...')
betweenness_dict = nx.betweenness_centrality(G, k=min(200, G.number_of_nodes()),
                                              weight='impedance', seed=42)
betweenness = np.array([betweenness_dict.get(bid, 0) for bid in bus_id_list])

# 2c. PageRank
print('  Calcul PageRank...')
pagerank_dict = nx.pagerank(G, weight='admittance', max_iter=100)
pagerank = np.array([pagerank_dict.get(bid, 0) for bid in bus_id_list])

# 2d. Clustering coefficient
clustering_dict = nx.clustering(G)
clustering = np.array([clustering_dict.get(bid, 0) for bid in bus_id_list])

# 2e. Capacite de transit totale et tension max des lignes connectees
total_s_nom = np.zeros(len(bus_id_list))
max_v_nom = np.zeros(len(bus_id_list))
mean_admittance = np.zeros(len(bus_id_list))

for i, bid in enumerate(bus_id_list):
    if bid not in G:
        continue
    neighbors = list(G.neighbors(bid))
    if len(neighbors) == 0:
        continue
    s_noms = [G[bid][n].get('s_nom', 0) for n in neighbors]
    v_noms = [G[bid][n].get('v_nom', 0) for n in neighbors]
    admittances = [G[bid][n].get('admittance', 0) for n in neighbors]
    total_s_nom[i] = sum(s_noms)
    max_v_nom[i] = max(v_noms) if v_noms else 0
    mean_admittance[i] = np.mean(admittances) if admittances else 0

# 2f. Spectral embedding (premiers vecteurs propres du Laplacien normalise)
# Trop couteux pour le graphe complet, on prend la plus grande composante connexe
print('  Calcul spectral embedding (4 dimensions)...')
largest_cc = max(nx.connected_components(G), key=len)
G_cc = G.subgraph(largest_cc)

try:
    # Laplacien normalise -> vecteurs propres
    L = nx.normalized_laplacian_matrix(G_cc).astype(float)
    from scipy.sparse.linalg import eigsh
    n_spectral = 4
    eigenvalues, eigenvectors = eigsh(L, k=n_spectral + 1, which='SM', tol=1e-4)
    # Skip le premier (constante)
    spectral_embed = eigenvectors[:, 1:n_spectral + 1]
    
    # Mapper vers tous les bus
    cc_nodes = list(G_cc.nodes())
    cc_node_to_pos = {n: i for i, n in enumerate(cc_nodes)}
    spectral_features = np.zeros((len(bus_id_list), n_spectral))
    for i, bid in enumerate(bus_id_list):
        if bid in cc_node_to_pos:
            spectral_features[i] = spectral_embed[cc_node_to_pos[bid]]
    print(f'  Spectral: {n_spectral} dimensions, composante principale: {len(cc_nodes)} noeuds')
except Exception as e:
    print(f'  Spectral ECHEC: {e}')
    spectral_features = np.zeros((len(bus_id_list), 4))
    n_spectral = 4

# 2g. Distance electrique aux k plus proches voisins (sur le graphe)
print('  Calcul distances electriques (Dijkstra, k=7)...')
k_elec = 7
elec_dist_features = np.full((len(bus_id_list), k_elec), np.nan)

# Pre-calcul: shortest path lengths ponderes par impedance pour chaque bus
# Optimisation: on fait un Dijkstra cutoff depuis chaque bus
for i, bid in enumerate(bus_id_list):
    if bid not in G:
        continue
    try:
        # Dijkstra avec cutoff (limite la profondeur pour vitesse)
        lengths = nx.single_source_dijkstra_path_length(G, bid, weight='impedance', cutoff=5.0)
        # Trier par distance, exclure le bus lui-meme
        sorted_dists = sorted([(d, n) for n, d in lengths.items() if n != bid])
        for j in range(min(k_elec, len(sorted_dists))):
            elec_dist_features[i, j] = sorted_dists[j][0]
    except:
        pass

# Remplacer NaN par une grande valeur
elec_dist_features = np.nan_to_num(elec_dist_features, nan=10.0)

elapsed_static = time.time() - t0
print(f'\n  Features statiques calculees en {elapsed_static:.0f}s')
print(f'  degree: mean={np.mean(degree):.1f}, max={np.max(degree):.0f}')
print(f'  betweenness: mean={np.mean(betweenness):.4f}, max={np.max(betweenness):.4f}')
print(f'  pagerank: mean={np.mean(pagerank):.6f}, max={np.max(pagerank):.6f}')
print(f'  clustering: mean={np.mean(clustering):.3f}')
print(f'  total_s_nom: mean={np.mean(total_s_nom):.0f}, max={np.max(total_s_nom):.0f}')
print(f'  max_v_nom: mean={np.mean(max_v_nom):.0f}, max={np.max(max_v_nom):.0f}')
print(f'  elec_dist[0]: mean={np.nanmean(elec_dist_features[:,0]):.3f}')

# Assembler les features statiques de graphe
graph_features_static = np.column_stack([
    degree,
    betweenness,
    pagerank,
    clustering,
    total_s_nom,
    max_v_nom,
    mean_admittance,
    elec_dist_features,  # k_elec colonnes
    spectral_features,   # n_spectral colonnes
])
graph_feature_names = (
    ['degree', 'betweenness', 'pagerank', 'clustering',
     'total_s_nom', 'max_v_nom', 'mean_admittance'] +
    [f'elec_dist_{j+1}' for j in range(k_elec)] +
    [f'spectral_{j+1}' for j in range(n_spectral)]
)
print(f'  Total features statiques graphe: {graph_features_static.shape[1]} ({graph_feature_names})')

# Scaler les features graphe
from sklearn.preprocessing import StandardScaler
scaler_graph = StandardScaler()
graph_features_static_scaled = scaler_graph.fit_transform(graph_features_static)

# =====================================================================
# ETAPE 3 : Features dynamiques de graphe (dependantes du timestamp)
# =====================================================================
print('\n--- Etape 3: Features dynamiques de graphe ---')
print('  Pour chaque (bus, t): CI des voisins sur le graphe (1-hop, 2-hop, 3-hop)')
print('  Pondere par admittance des lignes')

def compute_graph_dynamic_features(bus_pos, t, ci_matrix, G, bus_id_list, bus_id_to_idx):
    """Calcule les features dynamiques de graphe pour un bus a un timestamp."""
    bid = bus_id_list[bus_pos]
    ci_snap = ci_matrix[t, :]
    ci_snap_clean = np.where(ci_snap == 0, np.nan, ci_snap)
    
    features = []
    
    # 1-hop: voisins directs sur le graphe
    if bid in G:
        neighbors_1 = list(G.neighbors(bid))
        if neighbors_1:
            # CI des voisins 1-hop, pondere par admittance
            ci_1hop = []
            weights_1hop = []
            for n in neighbors_1:
                if n in bus_id_to_idx:
                    n_idx = bus_id_to_idx[n]
                    ci_val = ci_snap_clean[n_idx]
                    if not np.isnan(ci_val):
                        ci_1hop.append(ci_val)
                        weights_1hop.append(G[bid][n].get('admittance', 1.0))
            
            if ci_1hop:
                w = np.array(weights_1hop)
                w = w / w.sum()
                ci_1hop = np.array(ci_1hop)
                features.extend([
                    np.sum(w * ci_1hop),           # admittance-weighted mean CI 1-hop
                    np.std(ci_1hop) if len(ci_1hop) > 1 else 0,  # std CI 1-hop
                    len(ci_1hop),                   # n voisins valides 1-hop
                ])
            else:
                features.extend([np.nan, 0, 0])
            
            # 2-hop
            neighbors_2 = set()
            for n in neighbors_1:
                if n in G:
                    for nn in G.neighbors(n):
                        if nn != bid and nn not in neighbors_1:
                            neighbors_2.add(nn)
            
            ci_2hop = []
            for n in neighbors_2:
                if n in bus_id_to_idx:
                    n_idx = bus_id_to_idx[n]
                    ci_val = ci_snap_clean[n_idx]
                    if not np.isnan(ci_val):
                        ci_2hop.append(ci_val)
            
            if ci_2hop:
                features.extend([np.mean(ci_2hop), np.std(ci_2hop), len(ci_2hop)])
            else:
                features.extend([np.nan, 0, 0])
            
            # 3-hop
            neighbors_3 = set()
            for n in neighbors_2:
                if n in G:
                    for nn in G.neighbors(n):
                        if nn != bid and nn not in neighbors_1 and nn not in neighbors_2:
                            neighbors_3.add(nn)
            
            ci_3hop = []
            for n in neighbors_3:
                if n in bus_id_to_idx:
                    n_idx = bus_id_to_idx[n]
                    ci_val = ci_snap_clean[n_idx]
                    if not np.isnan(ci_val):
                        ci_3hop.append(ci_val)
            
            if ci_3hop:
                features.extend([np.mean(ci_3hop), len(ci_3hop)])
            else:
                features.extend([np.nan, 0])
        else:
            features.extend([np.nan, 0, 0, np.nan, 0, 0, np.nan, 0])
    else:
        features.extend([np.nan, 0, 0, np.nan, 0, 0, np.nan, 0])
    
    return np.array(features, dtype=float)

# Test rapide
test_feat = compute_graph_dynamic_features(0, 0, ci_matrix, G, bus_id_list, bus_id_to_idx)
graph_dynamic_names = ['adm_ci_1hop', 'std_ci_1hop', 'n_1hop',
                       'mean_ci_2hop', 'std_ci_2hop', 'n_2hop',
                       'mean_ci_3hop', 'n_3hop']
print(f'  Features dynamiques par (bus, t): {len(graph_dynamic_names)} ({graph_dynamic_names})')
print(f'  Test bus 0, t=0: {test_feat}')

# =====================================================================
# ETAPE 4 : Entrainement RF residuel GRAPH-AWARE
# =====================================================================
print('\n--- Etape 4: Entrainement RF residuel graph-aware ---')
t0 = time.time()

# Memes snapshots d'entrainement que le RF original
train_snaps = [s for s in range(0, 8760, 200) if s not in set(loo_snapshots)][:40]

X_train_graph, y_resid_graph = [], []
for t in train_snaps:
    ci_snap = np.where(ci_matrix[t, :] == 0, np.nan, ci_matrix[t, :])
    valid_mask_t = ~np.isnan(ci_snap)
    valid_idx = np.where(valid_mask_t)[0]
    if len(valid_idx) < 20:
        continue
    rng = np.random.RandomState(t)
    sample = rng.choice(valid_idx, min(100, len(valid_idx)), replace=False)
    
    for bus_pos in sample:
        true_ci = ci_snap[bus_pos]
        if np.isnan(true_ci) or true_ci == 0:
            continue
        
        # IDW prediction (meme que build_residual_model original)
        avail_mask = valid_mask_t.copy()
        avail_mask[bus_pos] = False
        avail = np.where(avail_mask)[0]
        if len(avail) < 3:
            continue
        tree_tmp = cKDTree(bus_coords[avail])
        dd, idx = tree_tmp.query(bus_coords[bus_pos], k=min(7, len(avail)))
        dd_km = dd / 1000
        nci = ci_snap[avail[idx]]
        v = ~np.isnan(nci)
        if v.sum() == 0:
            continue
        dd_c = np.maximum(dd_km[v], 3.0)
        w = 1.0 / (dd_c ** 2.0)
        w = w / w.sum()
        pred_idw = float(np.sum(w * nci[v]))
        resid = true_ci - pred_idw
        
        # Features: covariables originales + features voisinage spatiales + graphe statique + graphe dynamique
        nn_ci = nci[v]
        feat_spatial = np.concatenate([
            X_all_scaled[bus_pos],  # 13 covariables
            [np.mean(nn_ci), np.std(nn_ci) if len(nn_ci) > 1 else 0,
             np.min(nn_ci), np.max(nn_ci),
             dd_km[0], np.mean(dd_km[v]), float(v.sum())]  # 7 features nn spatiales
        ])
        
        feat_graph_static = graph_features_static_scaled[bus_pos]  # features graphe statiques
        feat_graph_dynamic = compute_graph_dynamic_features(bus_pos, t, ci_matrix, G, bus_id_list, bus_id_to_idx)
        
        feat_full = np.concatenate([feat_spatial, feat_graph_static, feat_graph_dynamic])
        X_train_graph.append(feat_full)
        y_resid_graph.append(resid)

X_train_graph = np.nan_to_num(np.array(X_train_graph), 0)
y_resid_graph = np.array(y_resid_graph)

print(f'  Training set: {len(X_train_graph)} samples, {X_train_graph.shape[1]} features')
print(f'  Features: 13 covariables + 7 nn spatiales + {graph_features_static.shape[1]} graphe statiques + {len(graph_dynamic_names)} graphe dynamiques')

# Entrainer le RF
rf_graph = RandomForestRegressor(n_estimators=80, max_depth=10, random_state=0, n_jobs=-1,
                                  min_samples_leaf=5)
rf_graph.fit(X_train_graph, y_resid_graph)
print(f'  RF R2_train: {rf_graph.score(X_train_graph, y_resid_graph):.3f}')

# Feature importance
all_feat_names = (list(features.columns) +
                  ['mean_nn_ci', 'std_nn_ci', 'min_nn_ci', 'max_nn_ci', 'd1', 'mean_d', 'n_valid'] +
                  graph_feature_names +
                  graph_dynamic_names)
importances = rf_graph.feature_importances_
sorted_imp = sorted(zip(all_feat_names, importances), key=lambda x: x[1], reverse=True)
print(f'\n  Top 15 feature importances:')
for name, imp in sorted_imp[:15]:
    bar = '#' * int(imp * 200)
    print(f'    {name:>20s}: {imp:.4f} {bar}')

print(f'\n  Entrainement: {time.time()-t0:.0f}s')

# =====================================================================
# ETAPE 5 : Prediction sur le set LOO commun
# =====================================================================
print('\n--- Etape 5: Predictions LOO ---')
t0 = time.time()

preds_graph = []
preds_graph_clipped = []
ALPHA_CLIP = 2.0  # clip recommande

for _, row in df_common.iterrows():
    bus_pos = int(row['bus_pos'])
    t = int(row['t'])
    pred_idw_base = row['IDW']
    
    # Features spatiales (meme que avant)
    mask_tmp = np.ones(len(bus_coords), dtype=bool)
    mask_tmp[bus_pos] = False
    tree_tmp = cKDTree(bus_coords[mask_tmp])
    dd_tmp, idx_tmp = tree_tmp.query(bus_coords[bus_pos], k=7)
    dd_km_tmp = dd_tmp / 1000
    orig_tmp = np.where(mask_tmp)[0]
    nci_tmp = ci_matrix[t, orig_tmp[idx_tmp]]
    nci_tmp = np.where(nci_tmp == 0, np.nan, nci_tmp)
    v_tmp = ~np.isnan(nci_tmp)
    
    if v_tmp.sum() > 0:
        nn_ci = nci_tmp[v_tmp]
        feat_spatial = np.concatenate([
            X_all_scaled[bus_pos],
            [np.mean(nn_ci), np.std(nn_ci) if len(nn_ci) > 1 else 0,
             np.min(nn_ci), np.max(nn_ci),
             dd_km_tmp[0], np.mean(dd_km_tmp[v_tmp]), float(v_tmp.sum())]
        ])
        
        feat_graph_static = graph_features_static_scaled[bus_pos]
        feat_graph_dynamic = compute_graph_dynamic_features(bus_pos, t, ci_matrix, G, bus_id_list, bus_id_to_idx)
        
        feat_full = np.concatenate([feat_spatial, feat_graph_static, feat_graph_dynamic]).reshape(1, -1)
        feat_full = np.nan_to_num(feat_full, 0)
        
        correction = rf_graph.predict(feat_full)[0]
        pred_raw = pred_idw_base + correction
        preds_graph.append(pred_raw)
        
        # Version clippee
        bound = ALPHA_CLIP * max(abs(pred_idw_base), 5.0)
        correction_clipped = np.clip(correction, -bound, bound)
        preds_graph_clipped.append(max(0, pred_idw_base + correction_clipped))
    else:
        preds_graph.append(pred_idw_base)
        preds_graph_clipped.append(pred_idw_base)

preds_graph = np.array(preds_graph)
preds_graph_clipped = np.array(preds_graph_clipped)

elapsed_pred = time.time() - t0
print(f'  Predictions: {elapsed_pred:.0f}s')

# =====================================================================
# ETAPE 6 : Comparaison finale
# =====================================================================
print('\n' + '=' * 90)
print('  RESULTATS: IDW vs IDW_corrected vs IDW_graph-aware')
print('=' * 90)

true_v = df_common['true_ci'].values

COMPARE = {
    'IDW': df_common['IDW'].values,
    'IDW_corrected': df_common['IDW_corrected'].values,
    'IDW_graph (raw)': preds_graph,
    f'IDW_graph (α={ALPHA_CLIP})': preds_graph_clipped,
}

print(f'\n  {"Modele":>22s} {"MAE":>7s} {"wMAPE%":>8s} {"R2":>7s} {"Bias":>7s}  {"P95":>7s}')
print('  ' + '-' * 65)

for name, pred in COMPARE.items():
    ae = np.abs(true_v - pred)
    mae = np.mean(ae)
    wmape = np.sum(ae) / np.sum(true_v) * 100
    r2 = 1 - np.sum((true_v - pred)**2) / np.sum((true_v - true_v.mean())**2)
    bias = np.mean(pred - true_v)
    p95 = np.percentile(ae, 95)
    flag = ' <<<' if wmape < 22 else ''
    print(f'  {name:>22s} {mae:7.1f} {wmape:8.1f} {r2:7.3f} {bias:+7.1f}  {p95:7.1f}{flag}')

# Detail par tranche
ci_bins = [(0, 5), (5, 20), (20, 50), (50, 100), (100, 300), (300, 9999)]
print(f'\n  --- Detail par tranche ---')
for name, pred in [('IDW_corrected', df_common['IDW_corrected'].values),
                    ('IDW_graph (raw)', preds_graph)]:
    ae = np.abs(true_v - pred)
    print(f'\n  {name}:')
    print(f'  {"CI range":>15s} {"MAE":>7s} {"wMAPE%":>8s} {"Biais":>7s} {"n":>6s}')
    for lo, hi in ci_bins:
        m = (true_v >= lo) & (true_v < hi)
        if m.sum() == 0:
            continue
        mae_s = np.mean(ae[m])
        wmape_s = np.sum(ae[m]) / max(np.sum(true_v[m]), 1) * 100
        bias_s = np.mean(pred[m] - true_v[m])
        print(f'  {f"{lo}-{hi}":>15s} {mae_s:7.1f} {wmape_s:8.1f} {bias_s:+7.1f} {m.sum():6d}')

# Verdict
wmape_corr = np.sum(np.abs(true_v - df_common['IDW_corrected'].values)) / np.sum(true_v) * 100
wmape_graph = np.sum(np.abs(true_v - preds_graph)) / np.sum(true_v) * 100
delta = wmape_graph - wmape_corr

print(f'\n  {"=" * 70}')
print(f'  wMAPE IDW_corrected = {wmape_corr:.1f}%')
print(f'  wMAPE IDW_graph     = {wmape_graph:.1f}%')
print(f'  Delta               = {delta:+.1f} pts')

if delta < -2:
    print(f'  >>> AMELIORATION SIGNIFICATIVE — les features graphe apportent un gain reel <<<')
    print(f'  >>> La topologie electrique est le bon levier <<<')
elif delta < -0.5:
    print(f'  >>> Amelioration moderee — les features graphe aident un peu <<<')
    print(f'  >>> Approfondir avec plus de features ou architecture GNN legere <<<')
elif delta < 0.5:
    print(f'  >>> Gain marginal — les features graphe n\'apportent presque rien <<<')
    print(f'  >>> Le plafond est probablement feature-limited, pas topology-limited <<<')
else:
    print(f'  >>> Degradation — les features graphe ajoutent du bruit <<<')
print(f'  {"=" * 70}')


  GRAPH-AWARE RESIDUAL MODEL — IDW + RF(covariables + features graphe)

--- Etape 1: Construction du graphe electrique ---
  Lignes: 5888, colonnes: ['line_id', 'bus0', 'bus1', 'r', 'x', 's_nom', 'v_nom', 'length', 'geometry']
  Graphe: 4393 noeuds, 5073 aretes
  Composantes connexes: 94

--- Etape 2: Features topologiques statiques ---
  Calcul betweenness (approximation k=200)...
  Calcul PageRank...
  Calcul spectral embedding (4 dimensions)...
  Spectral: 4 dimensions, composante principale: 1022 noeuds
  Calcul distances electriques (Dijkstra, k=7)...

  Features statiques calculees en 1s
  degree: mean=2.3, max=10
  betweenness: mean=0.0007, max=0.0199
  pagerank: mean=0.000228, max=0.000885
  clustering: mean=0.057
  total_s_nom: mean=4959, max=81509
  max_v_nom: mean=225, max=225
  elec_dist[0]: mean=2.602
  Total features statiques graphe: 18 (['degree', 'betweenness', 'pagerank', 'clustering', 'total_s_nom', 'max_v_nom', 'mean_admittance', 'elec_dist_1', 'elec_dist_2', 'elec_

In [14]:
# === Cellule 14 : GRAPH-AWARE v2 — Graphe connecté + diffusion + gradient ===
# FIX LEAKAGE: gradient et diffusion utilisent pred_idw, PAS ci_local
import time
import networkx as nx
from scipy.sparse import csr_matrix
from scipy.sparse.linalg import eigsh

print('=' * 90)
print('  GRAPH-AWARE v2 (FIXED) — sans leakage ci_local')
print('=' * 90)

# =====================================================================
# ETAPE 1 : Connecter les composantes isolées (same as before)
# =====================================================================
print('\n--- Etape 1: Connexion des composantes isolées ---')

G2 = G.copy()
components = list(nx.connected_components(G2))
components.sort(key=len, reverse=True)
main_cc = components[0]

main_cc_indices = [bus_id_to_idx[bid] for bid in main_cc if bid in bus_id_to_idx]
main_cc_coords = bus_coords[main_cc_indices]
main_cc_tree = cKDTree(main_cc_coords)
main_cc_ids = [bid for bid in main_cc if bid in bus_id_to_idx]

bridges_added = 0
for cc in components[1:]:
    best_dist = np.inf
    best_pair = None
    for bid in cc:
        if bid not in bus_id_to_idx:
            continue
        idx = bus_id_to_idx[bid]
        dd, nn = main_cc_tree.query(bus_coords[idx], k=1)
        dist_km = dd / 1000
        if dist_km < best_dist:
            best_dist = dist_km
            best_pair = (bid, main_cc_ids[nn])
    if best_pair is not None and best_dist < 500:
        z_est = max(best_dist * 0.001, 0.001)
        G2.add_edge(best_pair[0], best_pair[1],
                    impedance=z_est, admittance=1.0/z_est,
                    s_nom=1000, v_nom=225, length=best_dist, synthetic=True)
        bridges_added += 1

components_v2 = list(nx.connected_components(G2))
components_v2.sort(key=len, reverse=True)
print(f'  Ponts: {bridges_added}, Composantes: {len(components_v2)}, '
      f'Plus grande: {len(components_v2[0])} ({len(components_v2[0])/len(bus_id_list)*100:.1f}%)')

# =====================================================================
# ETAPE 2 : Matrice adjacence
# =====================================================================
n_buses = len(bus_id_list)
row_idx, col_idx, weights = [], [], []
for u, v, data in G2.edges(data=True):
    if u in bus_id_to_idx and v in bus_id_to_idx:
        i, j = bus_id_to_idx[u], bus_id_to_idx[v]
        w = data.get('admittance', 1.0)
        row_idx.extend([i, j])
        col_idx.extend([j, i])
        weights.extend([w, w])
W = csr_matrix((weights, (row_idx, col_idx)), shape=(n_buses, n_buses))
D_diag = np.array(W.sum(axis=1)).flatten()
D_diag_safe = np.where(D_diag > 0, D_diag, 1.0)

# =====================================================================
# ETAPE 3 : Spectral embedding (8 dims)
# =====================================================================
print('\n--- Etape 3: Spectral embedding ---')
largest_cc_v2 = components_v2[0]
G2_cc = G2.subgraph(largest_cc_v2)
try:
    L_cc = nx.normalized_laplacian_matrix(G2_cc).astype(float)
    n_spectral_v2 = 8
    eigenvalues_v2, eigenvectors_v2 = eigsh(L_cc, k=n_spectral_v2 + 1, which='SM', tol=1e-4)
    spectral_v2 = eigenvectors_v2[:, 1:n_spectral_v2 + 1]
    cc_nodes_v2 = list(G2_cc.nodes())
    cc_node_to_pos_v2 = {n: i for i, n in enumerate(cc_nodes_v2)}
    spectral_features_v2 = np.zeros((n_buses, n_spectral_v2))
    for i, bid in enumerate(bus_id_list):
        if bid in cc_node_to_pos_v2:
            spectral_features_v2[i] = spectral_v2[cc_node_to_pos_v2[bid]]
    print(f'  {n_spectral_v2} dimensions, {len(cc_nodes_v2)} noeuds')
except Exception as e:
    print(f'  ECHEC: {e}')
    spectral_features_v2 = np.zeros((n_buses, 8))
    n_spectral_v2 = 8

# =====================================================================
# ETAPE 4 : Features statiques v2
# =====================================================================
print('\n--- Etape 4: Features statiques v2 ---')
t0 = time.time()

degree_v2 = np.array([G2.degree(bid) if bid in G2 else 0 for bid in bus_id_list], dtype=float)
weighted_degree = D_diag.copy()

print('  Betweenness...')
betweenness_v2 = nx.betweenness_centrality(G2, k=min(300, G2.number_of_nodes()), weight='impedance', seed=42)
betweenness_v2 = np.array([betweenness_v2.get(bid, 0) for bid in bus_id_list])

print('  Closeness...')
closeness_v2 = nx.closeness_centrality(G2, distance='impedance')
closeness_v2 = np.array([closeness_v2.get(bid, 0) for bid in bus_id_list])

pagerank_v2 = nx.pagerank(G2, weight='admittance', max_iter=100)
pagerank_v2 = np.array([pagerank_v2.get(bid, 0) for bid in bus_id_list])

clustering_v2 = nx.clustering(G2)
clustering_v2 = np.array([clustering_v2.get(bid, 0) for bid in bus_id_list])

print('  Distances electriques (k=10)...')
k_elec_v2 = 10
elec_dist_v2 = np.full((n_buses, k_elec_v2), 10.0)
for i, bid in enumerate(bus_id_list):
    if bid not in G2:
        continue
    try:
        lengths = nx.single_source_dijkstra_path_length(G2, bid, weight='impedance', cutoff=10.0)
        sorted_dists = sorted([(d, n) for n, d in lengths.items() if n != bid])
        for j in range(min(k_elec_v2, len(sorted_dists))):
            elec_dist_v2[i, j] = sorted_dists[j][0]
    except:
        pass

n_elec_neighbors_1 = np.sum(elec_dist_v2 < 1.0, axis=1).astype(float)
n_elec_neighbors_3 = np.sum(elec_dist_v2 < 3.0, axis=1).astype(float)
n_elec_neighbors_5 = np.sum(elec_dist_v2 < 5.0, axis=1).astype(float)

elec_geo_ratio = np.zeros(n_buses)
for i, bid in enumerate(bus_id_list):
    if bid not in G2:
        continue
    neighbors = list(G2.neighbors(bid))
    if not neighbors:
        continue
    nn_bid = neighbors[0]
    if nn_bid in bus_id_to_idx:
        nn_idx = bus_id_to_idx[nn_bid]
        geo_dist = np.linalg.norm(bus_coords[i] - bus_coords[nn_idx]) / 1000
        elec_dist_1 = G2[bid][nn_bid].get('impedance', 1.0)
        elec_geo_ratio[i] = elec_dist_1 / max(geo_dist, 0.1)

graph_static_v2 = np.column_stack([
    degree_v2, weighted_degree, betweenness_v2, closeness_v2,
    pagerank_v2, clustering_v2, total_s_nom, max_v_nom, mean_admittance,
    elec_dist_v2,
    n_elec_neighbors_1, n_elec_neighbors_3, n_elec_neighbors_5,
    elec_geo_ratio,
    spectral_features_v2,
])
graph_static_v2_names = (
    ['degree', 'weighted_degree', 'betweenness', 'closeness',
     'pagerank', 'clustering', 'total_s_nom', 'max_v_nom', 'mean_admittance'] +
    [f'elec_dist_{j+1}' for j in range(k_elec_v2)] +
    ['n_elec_nn_1', 'n_elec_nn_3', 'n_elec_nn_5', 'elec_geo_ratio'] +
    [f'spectral_{j+1}' for j in range(n_spectral_v2)]
)
print(f'  {graph_static_v2.shape[1]} features, {time.time()-t0:.0f}s')

scaler_v2 = StandardScaler()
graph_static_v2_scaled = scaler_v2.fit_transform(graph_static_v2)

# =====================================================================
# ETAPE 5 : Features dynamiques v2 — FIXED (pred_idw au lieu de ci_local)
# =====================================================================
print('\n--- Etape 5: Features dynamiques v2 (FIXED — no leakage) ---')
print('  gradient = pred_idw - adm_ci_1hop (PAS ci_local)')
print('  diffusion_residual = pred_idw - diffused_ci (PAS ci_local)')
print('  delta_national = pred_idw - ci_national (PAS ci_local)')

def compute_dynamic_v2_fixed(bus_pos, t, ci_matrix, G2, bus_id_list, bus_id_to_idx,
                              pred_idw_value):
    """Features dynamiques SANS leakage: utilise pred_idw au lieu de ci_local."""
    bid = bus_id_list[bus_pos]
    ci_snap = ci_matrix[t, :]
    ci_clean = np.where(ci_snap == 0, np.nan, ci_snap).astype(float)
    
    # Proxy pour le CI local = prediction IDW (PAS la vraie valeur)
    ci_proxy = pred_idw_value
    
    features = []
    
    # --- 1. CI voisins graphe 1-hop pondéré admittance ---
    neighbors_1 = list(G2.neighbors(bid)) if bid in G2 else []
    
    ci_1hop_vals, w_1hop_vals = [], []
    for n in neighbors_1:
        if n in bus_id_to_idx:
            n_idx = bus_id_to_idx[n]
            ci_val = ci_clean[n_idx]
            if not np.isnan(ci_val):
                ci_1hop_vals.append(ci_val)
                w_1hop_vals.append(G2[bid][n].get('admittance', 1.0))
    
    if ci_1hop_vals:
        ci_1 = np.array(ci_1hop_vals)
        w_1 = np.array(w_1hop_vals)
        w_1n = w_1 / w_1.sum()
        adm_ci_1 = float(np.sum(w_1n * ci_1))
        features.extend([
            adm_ci_1,
            np.std(ci_1),
            len(ci_1),
            np.max(ci_1) - np.min(ci_1),
        ])
    else:
        adm_ci_1 = ci_proxy  # fallback
        features.extend([np.nan, 0, 0, 0])
    
    # --- 2. CI 2-hop et 3-hop ---
    neighbors_2 = set()
    for n in neighbors_1:
        if n in G2:
            for nn in G2.neighbors(n):
                if nn != bid and nn not in set(neighbors_1):
                    neighbors_2.add(nn)
    
    ci_2hop = [ci_clean[bus_id_to_idx[n]] for n in neighbors_2
               if n in bus_id_to_idx and not np.isnan(ci_clean[bus_id_to_idx[n]])]
    if ci_2hop:
        features.extend([np.mean(ci_2hop), np.std(ci_2hop), len(ci_2hop)])
    else:
        features.extend([np.nan, 0, 0])
    
    neighbors_3 = set()
    for n in neighbors_2:
        if n in G2:
            for nn in G2.neighbors(n):
                if nn != bid and nn not in set(neighbors_1) and nn not in neighbors_2:
                    neighbors_3.add(nn)
    
    ci_3hop = [ci_clean[bus_id_to_idx[n]] for n in neighbors_3
               if n in bus_id_to_idx and not np.isnan(ci_clean[bus_id_to_idx[n]])]
    if ci_3hop:
        features.extend([np.mean(ci_3hop), len(ci_3hop)])
    else:
        features.extend([np.nan, 0])
    
    # --- 3. Gradient CI sur graphe — FIXED: pred_idw vs graph neighbors ---
    gradient_1 = ci_proxy - adm_ci_1
    abs_gradient_1 = abs(gradient_1)
    norm_gradient = gradient_1 / (abs(ci_proxy) + abs(adm_ci_1) + 1e-6)
    features.extend([gradient_1, abs_gradient_1, norm_gradient])
    
    # --- 4. Diffusion electrique — FIXED: pred_idw vs diffused ---
    if bid in G2:
        try:
            lengths = nx.single_source_dijkstra_path_length(G2, bid, weight='impedance', cutoff=5.0)
            ci_in_radius = []
            for n, d in lengths.items():
                if n != bid and n in bus_id_to_idx:
                    ci_val = ci_clean[bus_id_to_idx[n]]
                    if not np.isnan(ci_val):
                        ci_in_radius.append((ci_val, d))
            if ci_in_radius:
                ci_vals_r = np.array([x[0] for x in ci_in_radius])
                dists_r = np.array([x[1] for x in ci_in_radius])
                w_elec = 1.0 / (dists_r + 0.01)
                diffused_ci = np.sum(w_elec * ci_vals_r) / np.sum(w_elec)
                diffusion_residual = ci_proxy - diffused_ci  # FIXED
                features.extend([diffused_ci, diffusion_residual, len(ci_in_radius)])
            else:
                features.extend([np.nan, 0, 0])
        except:
            features.extend([np.nan, 0, 0])
    else:
        features.extend([np.nan, 0, 0])
    
    # --- 5. Contexte temporel — FIXED ---
    ci_national = np.nanmean(ci_clean[ci_clean > 0]) if np.any(ci_clean > 0) else np.nan
    if not np.isnan(ci_national):
        ratio_local_national = ci_proxy / (ci_national + 1e-6)  # FIXED
        delta_national = ci_proxy - ci_national  # FIXED
    else:
        ratio_local_national = 1.0
        delta_national = 0
    features.extend([ci_national, ratio_local_national, delta_national])
    
    # --- 6. NOUVEAU: ecart IDW spatial vs IDW graphe ---
    # L'IDW spatial dit X, les voisins graphe disent Y — l'ecart est informatif
    idw_vs_graph = ci_proxy - adm_ci_1 if ci_1hop_vals else 0
    # Ecart IDW spatial vs diffusion electrique
    idw_vs_diffusion = diffusion_residual if 'diffusion_residual' in dir() else 0
    features.extend([idw_vs_graph])
    
    return np.array(features, dtype=float)

dynamic_v2_names = [
    'adm_ci_1hop', 'std_ci_1hop', 'n_1hop', 'range_ci_1hop',
    'mean_ci_2hop', 'std_ci_2hop', 'n_2hop',
    'mean_ci_3hop', 'n_3hop',
    'gradient_idw_graph', 'abs_gradient_idw_graph', 'norm_gradient_idw_graph',
    'diffused_ci_elec', 'diffusion_res_idw', 'n_elec_radius',
    'ci_national', 'ratio_idw_national', 'delta_idw_national',
    'idw_vs_graph_1hop',
]
print(f'  Features dynamiques v2: {len(dynamic_v2_names)}')

# =====================================================================
# ETAPE 6 : Entrainement RF résiduel v2 FIXED
# =====================================================================
print('\n--- Etape 6: Entrainement RF résiduel v2 (FIXED) ---')
t0 = time.time()

train_snaps = [s for s in range(0, 8760, 200) if s not in set(loo_snapshots)][:40]

X_train_v2, y_resid_v2 = [], []
for ti, t in enumerate(train_snaps):
    ci_snap = np.where(ci_matrix[t, :] == 0, np.nan, ci_matrix[t, :])
    valid_mask_t = ~np.isnan(ci_snap)
    valid_idx = np.where(valid_mask_t)[0]
    if len(valid_idx) < 20:
        continue
    rng = np.random.RandomState(t)
    sample = rng.choice(valid_idx, min(100, len(valid_idx)), replace=False)
    
    for bus_pos in sample:
        true_ci = ci_snap[bus_pos]
        if np.isnan(true_ci) or true_ci == 0:
            continue
        
        # IDW prediction
        avail_mask = valid_mask_t.copy()
        avail_mask[bus_pos] = False
        avail = np.where(avail_mask)[0]
        if len(avail) < 3:
            continue
        tree_tmp = cKDTree(bus_coords[avail])
        dd, idx = tree_tmp.query(bus_coords[bus_pos], k=min(7, len(avail)))
        dd_km = dd / 1000
        nci = ci_snap[avail[idx]]
        v = ~np.isnan(nci)
        if v.sum() == 0:
            continue
        dd_c = np.maximum(dd_km[v], 3.0)
        w = 1.0 / (dd_c ** 2.0)
        w = w / w.sum()
        pred_idw = float(np.sum(w * nci[v]))
        resid = true_ci - pred_idw
        
        # Features spatiales
        nn_ci = nci[v]
        feat_spatial = np.concatenate([
            X_all_scaled[bus_pos],
            [np.mean(nn_ci), np.std(nn_ci) if len(nn_ci) > 1 else 0,
             np.min(nn_ci), np.max(nn_ci),
             dd_km[0], np.mean(dd_km[v]), float(v.sum())]
        ])
        
        # Features graphe — FIXED: passe pred_idw
        feat_gs = graph_static_v2_scaled[bus_pos]
        feat_gd = compute_dynamic_v2_fixed(bus_pos, t, ci_matrix, G2, bus_id_list,
                                            bus_id_to_idx, pred_idw)
        
        feat_full = np.concatenate([feat_spatial, feat_gs, feat_gd])
        X_train_v2.append(feat_full)
        y_resid_v2.append(resid)
    
    if (ti + 1) % 10 == 0:
        print(f'    Snap {ti+1}/{len(train_snaps)} ({time.time()-t0:.0f}s)')

X_train_v2 = np.nan_to_num(np.array(X_train_v2), 0)
y_resid_v2 = np.array(y_resid_v2)

print(f'  Training set: {len(X_train_v2)} samples, {X_train_v2.shape[1]} features')

rf_v2 = RandomForestRegressor(n_estimators=100, max_depth=12, random_state=0, n_jobs=-1,
                               min_samples_leaf=4)
rf_v2.fit(X_train_v2, y_resid_v2)
print(f'  RF R2_train: {rf_v2.score(X_train_v2, y_resid_v2):.3f}')

# Feature importance
all_names_v2 = (
    list(features.columns) +
    ['mean_nn_ci', 'std_nn_ci', 'min_nn_ci', 'max_nn_ci', 'd1', 'mean_d', 'n_valid'] +
    graph_static_v2_names +
    dynamic_v2_names
)

# Tronquer si necessaire (au cas ou le nombre de features ne matche pas)
importances_v2 = rf_v2.feature_importances_
if len(all_names_v2) > len(importances_v2):
    all_names_v2 = all_names_v2[:len(importances_v2)]
elif len(all_names_v2) < len(importances_v2):
    all_names_v2 += [f'feat_{i}' for i in range(len(all_names_v2), len(importances_v2))]

sorted_imp_v2 = sorted(zip(all_names_v2, importances_v2), key=lambda x: x[1], reverse=True)
print(f'\n  Top 20 feature importances (FIXED):')
for name, imp in sorted_imp_v2[:20]:
    bar = '#' * int(imp * 200)
    origin = 'GRAPH-DYN' if name in dynamic_v2_names else ('GRAPH-STAT' if name in graph_static_v2_names else 'SPATIAL')
    print(f'    {name:>25s}: {imp:.4f} {bar}  [{origin}]')

imp_spatial = sum(imp for n, imp in sorted_imp_v2 if n in list(features.columns) + ['mean_nn_ci', 'std_nn_ci', 'min_nn_ci', 'max_nn_ci', 'd1', 'mean_d', 'n_valid'])
imp_graph_static = sum(imp for n, imp in sorted_imp_v2 if n in graph_static_v2_names)
imp_graph_dynamic = sum(imp for n, imp in sorted_imp_v2 if n in dynamic_v2_names)
print(f'\n  Importance par catégorie:')
print(f'    Spatiales:        {imp_spatial:.3f} ({imp_spatial*100:.1f}%)')
print(f'    Graphe statiques: {imp_graph_static:.3f} ({imp_graph_static*100:.1f}%)')
print(f'    Graphe dynamiques: {imp_graph_dynamic:.3f} ({imp_graph_dynamic*100:.1f}%)')

print(f'\n  Entrainement: {time.time()-t0:.0f}s')

# =====================================================================
# ETAPE 7 : Prédictions LOO — FIXED
# =====================================================================
print('\n--- Etape 7: Predictions LOO (FIXED) ---')
t0 = time.time()

preds_v2_raw = []
preds_v2_clip = []
ALPHA = 2.0

for ri, (_, row) in enumerate(df_common.iterrows()):
    bus_pos = int(row['bus_pos'])
    t = int(row['t'])
    pred_idw_base = row['IDW']
    
    mask_tmp = np.ones(len(bus_coords), dtype=bool)
    mask_tmp[bus_pos] = False
    tree_tmp = cKDTree(bus_coords[mask_tmp])
    dd_tmp, idx_tmp = tree_tmp.query(bus_coords[bus_pos], k=7)
    dd_km_tmp = dd_tmp / 1000
    orig_tmp = np.where(mask_tmp)[0]
    nci_tmp = ci_matrix[t, orig_tmp[idx_tmp]]
    nci_tmp = np.where(nci_tmp == 0, np.nan, nci_tmp)
    v_tmp = ~np.isnan(nci_tmp)
    
    if v_tmp.sum() > 0:
        nn_ci = nci_tmp[v_tmp]
        feat_spatial = np.concatenate([
            X_all_scaled[bus_pos],
            [np.mean(nn_ci), np.std(nn_ci) if len(nn_ci) > 1 else 0,
             np.min(nn_ci), np.max(nn_ci),
             dd_km_tmp[0], np.mean(dd_km_tmp[v_tmp]), float(v_tmp.sum())]
        ])
        
        feat_gs = graph_static_v2_scaled[bus_pos]
        # FIXED: passe pred_idw_base au lieu de ci_local
        feat_gd = compute_dynamic_v2_fixed(bus_pos, t, ci_matrix, G2, bus_id_list,
                                            bus_id_to_idx, pred_idw_base)
        
        feat_full = np.concatenate([feat_spatial, feat_gs, feat_gd]).reshape(1, -1)
        feat_full = np.nan_to_num(feat_full, 0)
        
        correction = rf_v2.predict(feat_full)[0]
        preds_v2_raw.append(pred_idw_base + correction)
        
        bound = ALPHA * max(abs(pred_idw_base), 5.0)
        preds_v2_clip.append(max(0, pred_idw_base + np.clip(correction, -bound, bound)))
    else:
        preds_v2_raw.append(pred_idw_base)
        preds_v2_clip.append(pred_idw_base)
    
    if (ri + 1) % 200 == 0:
        print(f'    {ri+1}/{len(df_common)} ({time.time()-t0:.0f}s)')

preds_v2_raw = np.array(preds_v2_raw)
preds_v2_clip = np.array(preds_v2_clip)
print(f'  Predictions: {time.time()-t0:.0f}s')

# =====================================================================
# ETAPE 8 : Résultats comparatifs
# =====================================================================
print('\n' + '=' * 90)
print('  RESULTATS — GRAPH-AWARE v2 FIXED (sans leakage)')
print('=' * 90)

true_v = df_common['true_ci'].values

COMPARE = {
    'IDW': df_common['IDW'].values,
    'IDW_corrected': df_common['IDW_corrected'].values,
    'IDW_graph v1': preds_graph,
    'IDW_graph v2 (raw)': preds_v2_raw,
    f'IDW_graph v2 (α={ALPHA})': preds_v2_clip,
}

print(f'\n  {"Modele":>25s} {"MAE":>7s} {"wMAPE%":>8s} {"R2":>7s} {"Bias":>7s}  {"P95":>7s}')
print('  ' + '-' * 70)

for name, pred in COMPARE.items():
    ae = np.abs(true_v - pred)
    mae = np.mean(ae)
    wmape = np.sum(ae) / np.sum(true_v) * 100
    r2 = 1 - np.sum((true_v - pred)**2) / np.sum((true_v - true_v.mean())**2)
    bias = np.mean(pred - true_v)
    p95 = np.percentile(ae, 95)
    flag = ' <<<' if wmape <= 20 else (' **' if wmape < 22 else '')
    print(f'  {name:>25s} {mae:7.1f} {wmape:8.1f} {r2:7.3f} {bias:+7.1f}  {p95:7.1f}{flag}')

# Detail par tranche
ci_bins = [(0, 5), (5, 20), (20, 50), (50, 100), (100, 300), (300, 9999)]
for name, pred in [('IDW_corrected', df_common['IDW_corrected'].values),
                    ('IDW_graph v1', preds_graph),
                    ('IDW_graph v2', preds_v2_raw)]:
    ae = np.abs(true_v - pred)
    print(f'\n  {name}:')
    print(f'  {"CI range":>15s} {"MAE":>7s} {"wMAPE%":>8s} {"Biais":>7s} {"n":>6s}')
    for lo, hi in ci_bins:
        m = (true_v >= lo) & (true_v < hi)
        if m.sum() == 0:
            continue
        print(f'  {f"{lo}-{hi}":>15s} {np.mean(ae[m]):7.1f} {np.sum(ae[m])/max(np.sum(true_v[m]),1)*100:8.1f} {np.mean(pred[m]-true_v[m]):+7.1f} {m.sum():6d}')

# Verdict
wmape_corr = np.sum(np.abs(true_v - df_common['IDW_corrected'].values)) / np.sum(true_v) * 100
wmape_v1 = np.sum(np.abs(true_v - preds_graph)) / np.sum(true_v) * 100
wmape_v2 = np.sum(np.abs(true_v - preds_v2_raw)) / np.sum(true_v) * 100

print(f'\n  {"=" * 70}')
print(f'  Progression:')
print(f'    IDW baseline      : 27.6%')
print(f'    IDW_corrected     : {wmape_corr:.1f}%')
print(f'    IDW_graph v1      : {wmape_v1:.1f}%')
print(f'    IDW_graph v2 FIXED: {wmape_v2:.1f}%')
print(f'  Gain total IDW→v2   : {27.6 - wmape_v2:+.1f} pts')

if wmape_v2 < 20:
    print(f'  >>> wMAPE < 20% — OBJECTIF ATTEINT <<<')
elif wmape_v2 < wmape_v1:
    print(f'  >>> v2 améliore v1 de {wmape_v1 - wmape_v2:.1f} pts <<<')
else:
    print(f'  >>> v2 n\'améliore pas v1 — les features graphe supplémentaires sont redondantes <<<')
print(f'  {"=" * 70}')


  GRAPH-AWARE v2 (FIXED) — sans leakage ci_local

--- Etape 1: Connexion des composantes isolées ---
  Ponts: 68, Composantes: 26, Plus grande: 4211 (95.9%)

--- Etape 3: Spectral embedding ---
  8 dimensions, 4211 noeuds

--- Etape 4: Features statiques v2 ---
  Betweenness...
  Closeness...
  Distances electriques (k=10)...
  31 features, 25s

--- Etape 5: Features dynamiques v2 (FIXED — no leakage) ---
  gradient = pred_idw - adm_ci_1hop (PAS ci_local)
  diffusion_residual = pred_idw - diffused_ci (PAS ci_local)
  delta_national = pred_idw - ci_national (PAS ci_local)
  Features dynamiques v2: 19

--- Etape 6: Entrainement RF résiduel v2 (FIXED) ---
    Snap 10/37 (0s)
    Snap 20/37 (1s)
    Snap 30/37 (1s)
  Training set: 3700 samples, 70 features
  RF R2_train: 0.885

  Top 20 feature importances (FIXED):
                      cap_gas: 0.2626 ####################################################  [SPATIAL]
           gradient_idw_graph: 0.1626 ################################  [GR

In [15]:
# === Cellule 15 : RECONSTRUCTION LAPLACIENNE SUR GRAPHE ===
# Modèle fermé, pas de RF — solution analytique:
#   ŷ = (I + λL)^{-1} y
# 
# Principe: on observe CI aux bus voisins, on reconstruit CI au bus cible
# en imposant un lissage sur le graphe électrique (Laplacien pondéré).
#
# 3 variantes testées:
# A) Laplacien pur (LOO: retire le bus, reconstruit)
# B) Laplacien + IDW prior (combine spatial et graphe)
# C) Laplacien + correction résiduelle RF (hybride)
import time
from scipy.sparse import eye, diags
from scipy.sparse.linalg import spsolve

print('=' * 90)
print('  RECONSTRUCTION LAPLACIENNE SUR GRAPHE ELECTRIQUE')
print('=' * 90)

# =====================================================================
# ETAPE 1 : Construire le Laplacien pondéré du graphe connecté G2
# =====================================================================
print('\n--- Etape 1: Laplacien pondéré ---')

# Matrice d'adjacence pondérée par admittance (déjà calculée: W)
# Laplacien combinatoire: L = D - W
# Laplacien normalisé: L_norm = I - D^{-1/2} W D^{-1/2}
# On utilise le Laplacien combinatoire pour la reconstruction

n = len(bus_id_list)
I_sparse = eye(n, format='csr')

# L = D - W (combinatoire)
D_sparse = diags(D_diag, 0, format='csr')
L_comb = D_sparse - W

# L normalisé symétrique: D^{-1/2} L D^{-1/2} = I - D^{-1/2} W D^{-1/2}
D_inv_sqrt = diags(1.0 / np.sqrt(D_diag_safe), 0, format='csr')
L_sym = I_sparse - D_inv_sqrt @ W @ D_inv_sqrt

print(f'  n={n}, nnz(W)={W.nnz}, nnz(L)={L_comb.nnz}')
print(f'  D_diag: min={D_diag.min():.2f}, max={D_diag.max():.2f}')

# =====================================================================
# ETAPE 2 : Reconstruction LOO — Variante A (Laplacien pur)
# =====================================================================
print('\n--- Etape 2: Variante A — Reconstruction Laplacienne pure ---')
print('  Pour chaque (bus_cible, t):')
print('    1. Observer CI aux bus voisins (masquer le bus cible)')
print('    2. Résoudre (I + λL)u = y  avec y_cible = 0, y_observé = CI observé')
print('    3. ŷ_cible = u_cible')
print()

# Problème: résoudre le système complet 4393x4393 pour chaque point = trop lent
# Solution: reconstruction LOCALE sur un sous-graphe autour du bus cible
# On extrait un sous-graphe de rayon R (en hops) autour du bus cible

def reconstruct_laplacian_local(bus_pos, t, ci_matrix, G2, bus_id_list, bus_id_to_idx,
                                 lam=1.0, radius_hops=3, use_sym=True):
    """Reconstruction Laplacienne locale autour d'un bus."""
    bid = bus_id_list[bus_pos]
    ci_snap = ci_matrix[t, :]
    ci_clean = np.where(ci_snap == 0, np.nan, ci_snap).astype(float)
    
    if bid not in G2:
        return np.nan
    
    # Extraire le sous-graphe de rayon R hops
    # BFS depuis bus_pos
    visited = {bid: 0}
    queue = [bid]
    while queue:
        current = queue.pop(0)
        if visited[current] >= radius_hops:
            continue
        for neighbor in G2.neighbors(current):
            if neighbor not in visited:
                visited[neighbor] = visited[current] + 1
                queue.append(neighbor)
    
    local_nodes = list(visited.keys())
    if len(local_nodes) < 3:
        return np.nan
    
    # Mapper les noeuds locaux vers indices 0..m
    local_to_idx = {n: i for i, n in enumerate(local_nodes)}
    m = len(local_nodes)
    target_local_idx = local_to_idx[bid]
    
    # Construire la matrice d'adjacence locale
    rows, cols, vals = [], [], []
    for u_node in local_nodes:
        for v_node in G2.neighbors(u_node):
            if v_node in local_to_idx:
                i, j = local_to_idx[u_node], local_to_idx[v_node]
                w = G2[u_node][v_node].get('admittance', 1.0)
                rows.append(i)
                cols.append(j)
                vals.append(w)
    
    W_local = csr_matrix((vals, (rows, cols)), shape=(m, m))
    D_local = np.array(W_local.sum(axis=1)).flatten()
    D_local_safe = np.where(D_local > 0, D_local, 1.0)
    
    if use_sym:
        # Laplacien normalisé symétrique
        D_inv_sqrt_local = diags(1.0 / np.sqrt(D_local_safe), 0, format='csr')
        I_local = eye(m, format='csr')
        L_local = I_local - D_inv_sqrt_local @ W_local @ D_inv_sqrt_local
    else:
        # Laplacien combinatoire
        D_local_mat = diags(D_local, 0, format='csr')
        L_local = D_local_mat - W_local
    
    # Vecteur d'observation: CI aux noeuds observés, 0 au noeud cible
    y_obs = np.zeros(m)
    obs_mask = np.zeros(m, dtype=bool)
    
    for node in local_nodes:
        if node == bid:
            continue  # LOO: on masque le bus cible
        if node in bus_id_to_idx:
            global_idx = bus_id_to_idx[node]
            ci_val = ci_clean[global_idx]
            if not np.isnan(ci_val):
                local_idx = local_to_idx[node]
                y_obs[local_idx] = ci_val
                obs_mask[local_idx] = True
    
    n_obs = obs_mask.sum()
    if n_obs < 2:
        return np.nan
    
    # Résoudre: (Σ_obs + λ L) u = Σ_obs y
    # où Σ_obs est la matrice diagonale d'observation (1 aux noeuds observés, 0 sinon)
    Sigma = diags(obs_mask.astype(float), 0, format='csr')
    A = Sigma + lam * L_local
    b = Sigma @ y_obs
    
    try:
        u = spsolve(A, b)
        result = float(u[target_local_idx])
        if np.isnan(result) or abs(result) > 2000:
            return np.nan
        return max(0, result)
    except:
        return np.nan


# Test rapide avec différents lambda
print('  Calibration lambda sur 100 points...')
test_indices = np.random.RandomState(42).choice(len(df_common), 100, replace=False)
df_test = df_common.iloc[test_indices]
true_test = df_test['true_ci'].values

lambdas = [0.01, 0.05, 0.1, 0.2, 0.5, 1.0, 2.0, 5.0, 10.0]
radii = [2, 3, 4]

print(f'\n  {"lambda":>8s} {"r":>3s} {"MAE":>7s} {"wMAPE%":>8s} {"R2":>7s} {"n_valid":>8s}')
print('  ' + '-' * 50)

best_config = None
best_wmape = 999

for radius in radii:
    for lam in lambdas:
        preds_test = []
        for _, row in df_test.iterrows():
            bp, t = int(row['bus_pos']), int(row['t'])
            p = reconstruct_laplacian_local(bp, t, ci_matrix, G2, bus_id_list,
                                             bus_id_to_idx, lam=lam, radius_hops=radius)
            preds_test.append(p)
        
        preds_test = np.array(preds_test)
        valid = ~np.isnan(preds_test)
        if valid.sum() < 50:
            continue
        
        ae = np.abs(true_test[valid] - preds_test[valid])
        mae = np.mean(ae)
        wmape = np.sum(ae) / np.sum(true_test[valid]) * 100
        ss_res = np.sum((true_test[valid] - preds_test[valid])**2)
        ss_tot = np.sum((true_test[valid] - true_test[valid].mean())**2)
        r2 = 1 - ss_res / ss_tot if ss_tot > 0 else 0
        
        flag = ' <<<' if wmape < best_wmape else ''
        if wmape < best_wmape:
            best_wmape = wmape
            best_config = (lam, radius)
        
        print(f'  {lam:8.2f} {radius:3d} {mae:7.1f} {wmape:8.1f} {r2:7.3f} {valid.sum():8d}{flag}')

print(f'\n  Meilleur: lambda={best_config[0]}, radius={best_config[1]}')
best_lam, best_radius = best_config

# =====================================================================
# ETAPE 3 : Evaluation complète — Variante A (Laplacien pur)
# =====================================================================
print(f'\n--- Etape 3: Laplacien pur (λ={best_lam}, r={best_radius}) sur 1000 points ---')
t0 = time.time()

preds_lap_pure = []
for ri, (_, row) in enumerate(df_common.iterrows()):
    bp, t = int(row['bus_pos']), int(row['t'])
    p = reconstruct_laplacian_local(bp, t, ci_matrix, G2, bus_id_list,
                                     bus_id_to_idx, lam=best_lam, radius_hops=best_radius)
    preds_lap_pure.append(p if not np.isnan(p) else row['IDW'])  # fallback IDW
    
    if (ri + 1) % 200 == 0:
        print(f'    {ri+1}/1000 ({time.time()-t0:.0f}s)')

preds_lap_pure = np.array(preds_lap_pure)
print(f'  Terminé: {time.time()-t0:.0f}s')

# =====================================================================
# ETAPE 4 : Variante B — Laplacien + IDW prior
# =====================================================================
print('\n--- Etape 4: Variante B — Laplacien + IDW prior ---')
print('  Ajouter un terme de prior au bus cible: (u_cible - pred_idw)² / sigma²')

def reconstruct_laplacian_with_prior(bus_pos, t, ci_matrix, G2, bus_id_list, bus_id_to_idx,
                                      pred_idw_value, lam=1.0, sigma_prior=0.5,
                                      radius_hops=3):
    """Reconstruction Laplacienne avec prior IDW au bus cible."""
    bid = bus_id_list[bus_pos]
    ci_snap = ci_matrix[t, :]
    ci_clean = np.where(ci_snap == 0, np.nan, ci_snap).astype(float)
    
    if bid not in G2:
        return pred_idw_value
    
    # BFS
    visited = {bid: 0}
    queue = [bid]
    while queue:
        current = queue.pop(0)
        if visited[current] >= radius_hops:
            continue
        for neighbor in G2.neighbors(current):
            if neighbor not in visited:
                visited[neighbor] = visited[current] + 1
                queue.append(neighbor)
    
    local_nodes = list(visited.keys())
    if len(local_nodes) < 3:
        return pred_idw_value
    
    local_to_idx = {n: i for i, n in enumerate(local_nodes)}
    m = len(local_nodes)
    target_local_idx = local_to_idx[bid]
    
    # Matrice adjacence locale
    rows, cols, vals = [], [], []
    for u_node in local_nodes:
        for v_node in G2.neighbors(u_node):
            if v_node in local_to_idx:
                i, j = local_to_idx[u_node], local_to_idx[v_node]
                w = G2[u_node][v_node].get('admittance', 1.0)
                rows.append(i)
                cols.append(j)
                vals.append(w)
    
    W_local = csr_matrix((vals, (rows, cols)), shape=(m, m))
    D_local = np.array(W_local.sum(axis=1)).flatten()
    D_local_safe = np.where(D_local > 0, D_local, 1.0)
    D_inv_sqrt_local = diags(1.0 / np.sqrt(D_local_safe), 0, format='csr')
    I_local = eye(m, format='csr')
    L_local = I_local - D_inv_sqrt_local @ W_local @ D_inv_sqrt_local
    
    # Observations
    y_obs = np.zeros(m)
    obs_weights = np.zeros(m)
    
    for node in local_nodes:
        if node == bid:
            continue
        if node in bus_id_to_idx:
            global_idx = bus_id_to_idx[node]
            ci_val = ci_clean[global_idx]
            if not np.isnan(ci_val):
                local_idx = local_to_idx[node]
                y_obs[local_idx] = ci_val
                obs_weights[local_idx] = 1.0
    
    # Ajouter le prior IDW au bus cible
    y_obs[target_local_idx] = pred_idw_value
    obs_weights[target_local_idx] = sigma_prior  # poids du prior (< 1 = prior faible)
    
    if np.sum(obs_weights) < 1:
        return pred_idw_value
    
    Sigma = diags(obs_weights, 0, format='csr')
    A = Sigma + lam * L_local
    b = Sigma @ y_obs
    
    try:
        u = spsolve(A, b)
        result = float(u[target_local_idx])
        if np.isnan(result) or abs(result) > 2000:
            return pred_idw_value
        return max(0, result)
    except:
        return pred_idw_value

# Calibration sigma_prior
print('  Calibration sigma_prior...')
sigmas = [0.1, 0.2, 0.3, 0.5, 0.7, 1.0]
print(f'  {"sigma":>8s} {"MAE":>7s} {"wMAPE%":>8s} {"R2":>7s}')

best_sigma = 0.5
best_sigma_wmape = 999

for sigma in sigmas:
    preds_test_b = []
    for _, row in df_test.iterrows():
        bp, t = int(row['bus_pos']), int(row['t'])
        p = reconstruct_laplacian_with_prior(bp, t, ci_matrix, G2, bus_id_list,
                                              bus_id_to_idx, row['IDW'],
                                              lam=best_lam, sigma_prior=sigma,
                                              radius_hops=best_radius)
        preds_test_b.append(p)
    
    preds_test_b = np.array(preds_test_b)
    ae = np.abs(true_test - preds_test_b)
    wmape = np.sum(ae) / np.sum(true_test) * 100
    mae = np.mean(ae)
    r2 = 1 - np.sum((true_test - preds_test_b)**2) / np.sum((true_test - true_test.mean())**2)
    
    flag = ' <<<' if wmape < best_sigma_wmape else ''
    if wmape < best_sigma_wmape:
        best_sigma = sigma
        best_sigma_wmape = wmape
    print(f'  {sigma:8.2f} {mae:7.1f} {wmape:8.1f} {r2:7.3f}{flag}')

print(f'  Meilleur sigma_prior={best_sigma}')

# Evaluation complète B
print(f'\n  Evaluation complète B (σ={best_sigma})...')
t0 = time.time()

preds_lap_prior = []
for ri, (_, row) in enumerate(df_common.iterrows()):
    bp, t = int(row['bus_pos']), int(row['t'])
    p = reconstruct_laplacian_with_prior(bp, t, ci_matrix, G2, bus_id_list,
                                          bus_id_to_idx, row['IDW'],
                                          lam=best_lam, sigma_prior=best_sigma,
                                          radius_hops=best_radius)
    preds_lap_prior.append(p)
    
    if (ri + 1) % 200 == 0:
        print(f'    {ri+1}/1000 ({time.time()-t0:.0f}s)')

preds_lap_prior = np.array(preds_lap_prior)
print(f'  Terminé: {time.time()-t0:.0f}s')

# =====================================================================
# ETAPE 5 : Variante C — Laplacien + correction RF graph-aware
# =====================================================================
print('\n--- Etape 5: Variante C — Laplacien + RF résiduel ---')
print('  pred_C = pred_Laplacien_B + rf_correction')

# Entrainer un RF résiduel sur les erreurs du Laplacien
# Utiliser les mêmes features graph-aware que v2
print('  Construction training set...')
t0 = time.time()

train_snaps_c = [s for s in range(0, 8760, 200) if s not in set(loo_snapshots)][:30]

X_train_c, y_resid_c = [], []
for ti, t in enumerate(train_snaps_c):
    ci_snap = np.where(ci_matrix[t, :] == 0, np.nan, ci_matrix[t, :])
    valid_mask_t = ~np.isnan(ci_snap)
    valid_idx = np.where(valid_mask_t)[0]
    if len(valid_idx) < 20:
        continue
    rng = np.random.RandomState(t + 9999)
    sample = rng.choice(valid_idx, min(80, len(valid_idx)), replace=False)
    
    for bus_pos in sample:
        true_ci = ci_snap[bus_pos]
        if np.isnan(true_ci) or true_ci == 0:
            continue
        
        # IDW prediction
        avail_mask = valid_mask_t.copy()
        avail_mask[bus_pos] = False
        avail = np.where(avail_mask)[0]
        if len(avail) < 3:
            continue
        tree_tmp = cKDTree(bus_coords[avail])
        dd, idx = tree_tmp.query(bus_coords[bus_pos], k=min(7, len(avail)))
        dd_km = dd / 1000
        nci = ci_snap[avail[idx]]
        v = ~np.isnan(nci)
        if v.sum() == 0:
            continue
        dd_c = np.maximum(dd_km[v], 3.0)
        w = 1.0 / (dd_c ** 2.0)
        w = w / w.sum()
        pred_idw = float(np.sum(w * nci[v]))
        
        # Laplacien prediction
        pred_lap = reconstruct_laplacian_with_prior(
            bus_pos, t, ci_matrix, G2, bus_id_list, bus_id_to_idx,
            pred_idw, lam=best_lam, sigma_prior=best_sigma, radius_hops=best_radius)
        
        resid_lap = true_ci - pred_lap
        
        # Features: covariables + graphe statiques + features dynamiques legeres
        nn_ci = nci[v]
        feat = np.concatenate([
            X_all_scaled[bus_pos],  # 13
            [np.mean(nn_ci), np.std(nn_ci) if len(nn_ci) > 1 else 0,
             np.min(nn_ci), np.max(nn_ci),
             dd_km[0], np.mean(dd_km[v]), float(v.sum())],  # 7
            graph_static_v2_scaled[bus_pos],  # 31
            [pred_idw, pred_lap, pred_idw - pred_lap],  # 3: écart IDW/Laplacien
        ])
        
        X_train_c.append(feat)
        y_resid_c.append(resid_lap)

X_train_c = np.nan_to_num(np.array(X_train_c), 0)
y_resid_c = np.array(y_resid_c)

print(f'  Training: {len(X_train_c)} samples, {X_train_c.shape[1]} features')

rf_lap = RandomForestRegressor(n_estimators=80, max_depth=10, random_state=0, n_jobs=-1,
                                min_samples_leaf=5)
rf_lap.fit(X_train_c, y_resid_c)
print(f'  RF R2_train: {rf_lap.score(X_train_c, y_resid_c):.3f}')
print(f'  Training: {time.time()-t0:.0f}s')

# Predictions complètes C
print('\n  Predictions C...')
t0 = time.time()

preds_lap_rf = []
for ri, (_, row) in enumerate(df_common.iterrows()):
    bp, t = int(row['bus_pos']), int(row['t'])
    pred_idw_base = row['IDW']
    pred_lap = preds_lap_prior[ri]
    
    # Features pour RF
    mask_tmp = np.ones(len(bus_coords), dtype=bool)
    mask_tmp[bp] = False
    tree_tmp = cKDTree(bus_coords[mask_tmp])
    dd_tmp, idx_tmp = tree_tmp.query(bus_coords[bp], k=7)
    dd_km_tmp = dd_tmp / 1000
    orig_tmp = np.where(mask_tmp)[0]
    nci_tmp = ci_matrix[t, orig_tmp[idx_tmp]]
    nci_tmp = np.where(nci_tmp == 0, np.nan, nci_tmp)
    v_tmp = ~np.isnan(nci_tmp)
    
    if v_tmp.sum() > 0:
        nn_ci = nci_tmp[v_tmp]
        feat = np.concatenate([
            X_all_scaled[bp],
            [np.mean(nn_ci), np.std(nn_ci) if len(nn_ci) > 1 else 0,
             np.min(nn_ci), np.max(nn_ci),
             dd_km_tmp[0], np.mean(dd_km_tmp[v_tmp]), float(v_tmp.sum())],
            graph_static_v2_scaled[bp],
            [pred_idw_base, pred_lap, pred_idw_base - pred_lap],
        ]).reshape(1, -1)
        feat = np.nan_to_num(feat, 0)
        correction = rf_lap.predict(feat)[0]
        # Clip la correction
        bound = 2.0 * max(abs(pred_lap), 5.0)
        correction = np.clip(correction, -bound, bound)
        preds_lap_rf.append(max(0, pred_lap + correction))
    else:
        preds_lap_rf.append(pred_lap)
    
    if (ri + 1) % 200 == 0:
        print(f'    {ri+1}/1000 ({time.time()-t0:.0f}s)')

preds_lap_rf = np.array(preds_lap_rf)
print(f'  Terminé: {time.time()-t0:.0f}s')

# =====================================================================
# ETAPE 6 : Résultats comparatifs — TOUS LES MODELES
# =====================================================================
print('\n' + '=' * 90)
print('  TABLEAU FINAL — TOUS LES MODELES')
print('=' * 90)

true_v = df_common['true_ci'].values

ALL_MODELS = {
    'IDW': df_common['IDW'].values,
    'IDW_corrected': df_common['IDW_corrected'].values,
    'IDW_graph v1': preds_graph,
    'IDW_graph v2': preds_v2_raw,
    'Laplacien pur': preds_lap_pure,
    'Laplacien+prior': preds_lap_prior,
    'Laplacien+RF': preds_lap_rf,
}

print(f'\n  {"Modele":>20s} {"MAE":>7s} {"wMAPE%":>8s} {"R2":>7s} {"Bias":>7s}  {"P95":>7s} {"P99":>7s}')
print('  ' + '-' * 75)

for name, pred in ALL_MODELS.items():
    ae = np.abs(true_v - pred)
    mae = np.mean(ae)
    wmape = np.sum(ae) / np.sum(true_v) * 100
    r2 = 1 - np.sum((true_v - pred)**2) / np.sum((true_v - true_v.mean())**2)
    bias = np.mean(pred - true_v)
    p95 = np.percentile(ae, 95)
    p99 = np.percentile(ae, 99)
    flag = ' <<<' if wmape <= 17 else (' **' if wmape < 20 else '')
    print(f'  {name:>20s} {mae:7.1f} {wmape:8.1f} {r2:7.3f} {bias:+7.1f}  {p95:7.1f} {p99:7.1f}{flag}')

# Détail par tranche pour les meilleurs
ci_bins = [(0, 5), (5, 20), (20, 50), (50, 100), (100, 300), (300, 9999)]
TOP_MODELS = ['IDW_graph v2', 'Laplacien+prior', 'Laplacien+RF']
print(f'\n  --- Détail par tranche (top modèles) ---')
for name in TOP_MODELS:
    pred = ALL_MODELS[name]
    ae = np.abs(true_v - pred)
    print(f'\n  {name}:')
    print(f'  {"CI range":>15s} {"MAE":>7s} {"wMAPE%":>8s} {"Biais":>7s} {"n":>6s}')
    for lo, hi in ci_bins:
        m = (true_v >= lo) & (true_v < hi)
        if m.sum() == 0:
            continue
        print(f'  {f"{lo}-{hi}":>15s} {np.mean(ae[m]):7.1f} '
              f'{np.sum(ae[m])/max(np.sum(true_v[m]),1)*100:8.1f} '
              f'{np.mean(pred[m]-true_v[m]):+7.1f} {m.sum():6d}')

# =====================================================================
# VERDICT FINAL
# =====================================================================
best_model_name = min(ALL_MODELS.keys(),
                       key=lambda k: np.sum(np.abs(true_v - ALL_MODELS[k])) / np.sum(true_v))
best_pred = ALL_MODELS[best_model_name]
best_wmape = np.sum(np.abs(true_v - best_pred)) / np.sum(true_v) * 100
best_mae = np.mean(np.abs(true_v - best_pred))
best_r2 = 1 - np.sum((true_v - best_pred)**2) / np.sum((true_v - true_v.mean())**2)

print(f'\n  {"=" * 70}')
print(f'  MEILLEUR MODELE: {best_model_name}')
print(f'  MAE={best_mae:.1f}  wMAPE={best_wmape:.1f}%  R2={best_r2:.3f}')
print(f'\n  Progression complète:')
print(f'    IDW baseline       : 27.6% wMAPE')
print(f'    + correction RF    : 22.6% (-5.0 pts)')
print(f'    + features graphe  : 21.1% (-1.5 pts)')
print(f'    + graphe connecté  : 17.8% (-3.3 pts)')

lap_wmape = np.sum(np.abs(true_v - preds_lap_prior)) / np.sum(true_v) * 100
print(f'    Laplacien+prior    : {lap_wmape:.1f}%')
laprf_wmape = np.sum(np.abs(true_v - preds_lap_rf)) / np.sum(true_v) * 100
print(f'    Laplacien+RF       : {laprf_wmape:.1f}%')

print(f'\n  Gain total: {27.6 - best_wmape:+.1f} pts de wMAPE')
print(f'  {"=" * 70}')


  RECONSTRUCTION LAPLACIENNE SUR GRAPHE ELECTRIQUE

--- Etape 1: Laplacien pondéré ---
  n=4393, nnz(W)=10282, nnz(L)=14673
  D_diag: min=0.00, max=2001.21

--- Etape 2: Variante A — Reconstruction Laplacienne pure ---
  Pour chaque (bus_cible, t):
    1. Observer CI aux bus voisins (masquer le bus cible)
    2. Résoudre (I + λL)u = y  avec y_cible = 0, y_observé = CI observé
    3. ŷ_cible = u_cible

  Calibration lambda sur 100 points...

    lambda   r     MAE   wMAPE%      R2  n_valid
  --------------------------------------------------
      0.01   2    28.7     42.7   0.793       99 <<<
      0.05   2    28.4     42.2   0.796       99 <<<
      0.10   2    28.0     41.6   0.800       99 <<<
      0.20   2    27.4     40.8   0.806       99 <<<
      0.50   2    26.2     39.0   0.818       99 <<<
      1.00   2    25.3     37.6   0.827       99 <<<
      2.00   2    24.9     37.0   0.834       99 <<<
      5.00   2    24.7     36.8   0.840       99 <<<
     10.00   2    25.3     37

In [16]:
# === Cellule 16 : AUDIT FINAL — Leakage + Ablation + Safe Prod + Business ===
import time

print('=' * 90)
print('  AUDIT FINAL — IDW_graph v2')
print('=' * 90)

# =====================================================================
# 1. AUDIT LEAKAGE — Vérification formelle
# =====================================================================
print('\n' + '=' * 90)
print('  1. AUDIT LEAKAGE')
print('=' * 90)

# Vérifions que compute_dynamic_v2_fixed n'accède JAMAIS à ci_matrix[t, bus_pos]
# Pour ça: on corrompt la valeur cible et on vérifie que les prédictions ne changent pas

print('  Test de corruption: remplacer ci_matrix[t, bus_pos] par une valeur aberrante')
print('  Si les prédictions changent → LEAKAGE')

np.random.seed(42)
test_sample = np.random.choice(len(df_common), 50, replace=False)
ci_matrix_backup = ci_matrix.copy()

preds_original = []
preds_corrupted = []

for idx in test_sample:
    row = df_common.iloc[idx]
    bp, t = int(row['bus_pos']), int(row['t'])
    pred_idw_base = row['IDW']
    
    # Prediction originale
    mask_tmp = np.ones(len(bus_coords), dtype=bool)
    mask_tmp[bp] = False
    tree_tmp = cKDTree(bus_coords[mask_tmp])
    dd_tmp, idx_tmp = tree_tmp.query(bus_coords[bp], k=7)
    dd_km_tmp = dd_tmp / 1000
    orig_tmp = np.where(mask_tmp)[0]
    nci_tmp = ci_matrix[t, orig_tmp[idx_tmp]]
    nci_tmp = np.where(nci_tmp == 0, np.nan, nci_tmp)
    v_tmp = ~np.isnan(nci_tmp)
    
    if v_tmp.sum() > 0:
        nn_ci = nci_tmp[v_tmp]
        feat_spatial = np.concatenate([
            X_all_scaled[bp],
            [np.mean(nn_ci), np.std(nn_ci) if len(nn_ci) > 1 else 0,
             np.min(nn_ci), np.max(nn_ci),
             dd_km_tmp[0], np.mean(dd_km_tmp[v_tmp]), float(v_tmp.sum())]
        ])
        feat_gs = graph_static_v2_scaled[bp]
        feat_gd = compute_dynamic_v2_fixed(bp, t, ci_matrix, G2, bus_id_list,
                                            bus_id_to_idx, pred_idw_base)
        feat_full = np.concatenate([feat_spatial, feat_gs, feat_gd]).reshape(1, -1)
        feat_full = np.nan_to_num(feat_full, 0)
        correction = rf_v2.predict(feat_full)[0]
        preds_original.append(pred_idw_base + correction)
        
        # Corrompre ci_matrix[t, bus_pos] avec une valeur aberrante
        original_val = ci_matrix[t, bp]
        ci_matrix[t, bp] = 99999.0  # valeur absurde
        
        # Recalculer IDW (ne devrait pas changer car bus_pos est masqué)
        # Features spatiales: IDW exclut bus_pos → pas de changement
        # Features dynamiques: compute_dynamic_v2_fixed utilise pred_idw, pas ci_local
        # MAIS: les voisins 1-hop sur le graphe lisent ci_matrix[t, neighbor]
        # Est-ce que bus_pos apparaît comme voisin d'un autre bus dans le BFS ?
        # NON: le BFS part de bus_pos mais ne l'inclut PAS comme voisin de lui-même
        
        feat_gd_c = compute_dynamic_v2_fixed(bp, t, ci_matrix, G2, bus_id_list,
                                              bus_id_to_idx, pred_idw_base)
        feat_full_c = np.concatenate([feat_spatial, feat_gs, feat_gd_c]).reshape(1, -1)
        feat_full_c = np.nan_to_num(feat_full_c, 0)
        correction_c = rf_v2.predict(feat_full_c)[0]
        preds_corrupted.append(pred_idw_base + correction_c)
        
        # Restaurer
        ci_matrix[t, bp] = original_val
    else:
        preds_original.append(pred_idw_base)
        preds_corrupted.append(pred_idw_base)

preds_original = np.array(preds_original)
preds_corrupted = np.array(preds_corrupted)

# Comparer
diff = np.abs(preds_original - preds_corrupted)
max_diff = np.max(diff)
mean_diff = np.mean(diff)
n_changed = np.sum(diff > 0.01)

print(f'\n  Résultat corruption test (50 points):')
print(f'    Max diff:     {max_diff:.6f}')
print(f'    Mean diff:    {mean_diff:.6f}')
print(f'    n changé (>0.01): {n_changed}/50')

if max_diff < 0.01:
    print(f'    ✓ PASS: aucune fuite — ci_matrix[t, bus_pos] n\'affecte pas les prédictions')
    LEAK_PASS = True
else:
    # Vérifier si c'est via les features dynamiques (voisins graphe)
    # bus_pos peut être voisin 1-hop d'un de SES voisins graphe
    # Dans ce cas, quand on calcule adm_ci_1hop pour bus_pos, on lit les voisins
    # Mais le BFS part de bus_pos et les voisins lisent LEUR CI, pas celui de bus_pos
    # SAUF si bus_pos est aussi voisin graphe d'un noeud dans le sous-graphe local
    print(f'    ⚠ ATTENTION: {n_changed} prédictions changent')
    print(f'    Cause probable: bus_pos apparaît comme voisin graphe dans le BFS')
    print(f'    → les features 1-hop lisent ci_matrix[t, bus_pos] via les voisins')
    print(f'    Impact moyen: {mean_diff:.2f} gCO2/kWh')
    
    # Quantifier l'impact relatif
    if mean_diff < 1.0:
        print(f'    Impact < 1 gCO2 → leakage mineur, acceptable')
        LEAK_PASS = True
    else:
        print(f'    Impact > 1 gCO2 → leakage significatif !')
        LEAK_PASS = False

# Vérifier aussi: le bus cible est-il exclu des voisins graphe ?
print(f'\n  Vérification: bus cible dans voisins graphe ?')
n_self_in_neighbors = 0
for idx in test_sample[:10]:
    row = df_common.iloc[idx]
    bp = int(row['bus_pos'])
    bid = bus_id_list[bp]
    if bid in G2:
        neighbors = list(G2.neighbors(bid))
        # Le bus est son propre voisin dans le BFS ?
        # Non: on fait BFS depuis bid, les voisins sont les AUTRES noeuds
        # Mais: dans compute_dynamic_v2_fixed, on parcourt neighbors_1 = G2.neighbors(bid)
        # Le bus bid n'est PAS dans G2.neighbors(bid) (pas de self-loop)
        if bid in neighbors:
            n_self_in_neighbors += 1
            print(f'    BUS {bp} ({bid}): SELF-LOOP DETECTE !')

if n_self_in_neighbors == 0:
    print(f'    ✓ Pas de self-loop: le bus cible n\'est jamais son propre voisin graphe')

# =====================================================================
# 2. ABLATION STRICTE — Isoler chaque brique
# =====================================================================
print('\n' + '=' * 90)
print('  2. ABLATION STRICTE')
print('=' * 90)
print('  Isoler la contribution de chaque composant')

# Configurations d'ablation
# On réentraîne un RF pour chaque config avec les features correspondantes
ablation_configs = {
    'A: spatial seul': {
        'spatial': True, 'graph_static': False, 'graph_dynamic': False,
    },
    'B: + graphe stat': {
        'spatial': True, 'graph_static': True, 'graph_dynamic': False,
    },
    'C: + graphe dyn': {
        'spatial': True, 'graph_static': False, 'graph_dynamic': True,
    },
    'D: stat + dyn': {
        'spatial': True, 'graph_static': True, 'graph_dynamic': True,
    },
    'E: full v2': {
        'spatial': True, 'graph_static': True, 'graph_dynamic': True,
    },
}

# Réutiliser les données d'entraînement de v2
# On a déjà X_train_v2 et y_resid_v2 de la cellule 14
# Structure: [13 covariables | 7 nn spatiales | 31 graphe stat | 19 graphe dyn]
n_cov = 13
n_nn = 7
n_gs = graph_static_v2.shape[1]  # 31
n_gd = len(dynamic_v2_names)  # 19

spatial_cols = list(range(0, n_cov + n_nn))  # 0..19
gs_cols = list(range(n_cov + n_nn, n_cov + n_nn + n_gs))  # 20..50
gd_cols = list(range(n_cov + n_nn + n_gs, n_cov + n_nn + n_gs + n_gd))  # 51..69

print(f'  Colonnes: spatial={len(spatial_cols)}, graphe_stat={len(gs_cols)}, graphe_dyn={len(gd_cols)}')
print(f'  Total features v2: {X_train_v2.shape[1]}')

t0 = time.time()
ablation_results = {}

print(f'\n  {"Config":>20s} {"MAE":>7s} {"wMAPE%":>8s} {"R2":>7s} {"Bias":>7s} {"P95":>7s}')
print('  ' + '-' * 65)

true_v = df_common['true_ci'].values

for config_name, flags in ablation_configs.items():
    # Sélectionner les colonnes
    cols = []
    if flags['spatial']:
        cols.extend(spatial_cols)
    if flags['graph_static']:
        cols.extend(gs_cols)
    if flags['graph_dynamic']:
        cols.extend(gd_cols)
    
    if not cols:
        continue
    
    # Entraîner RF sur ces colonnes
    X_train_sub = X_train_v2[:, cols]
    rf_sub = RandomForestRegressor(n_estimators=100, max_depth=12, random_state=0,
                                    n_jobs=-1, min_samples_leaf=4)
    rf_sub.fit(X_train_sub, y_resid_v2)
    
    # Prédire sur le set LOO complet
    # Réutiliser les features déjà calculées pendant l'étape 7 de v2
    # On doit recalculer... non, on peut construire le vecteur complet et sous-sélectionner
    preds_sub = []
    for ri, (_, row) in enumerate(df_common.iterrows()):
        bp, t = int(row['bus_pos']), int(row['t'])
        pred_idw_base = row['IDW']
        
        mask_tmp = np.ones(len(bus_coords), dtype=bool)
        mask_tmp[bp] = False
        tree_tmp = cKDTree(bus_coords[mask_tmp])
        dd_tmp, idx_tmp = tree_tmp.query(bus_coords[bp], k=7)
        dd_km_tmp = dd_tmp / 1000
        orig_tmp = np.where(mask_tmp)[0]
        nci_tmp = ci_matrix[t, orig_tmp[idx_tmp]]
        nci_tmp = np.where(nci_tmp == 0, np.nan, nci_tmp)
        v_tmp = ~np.isnan(nci_tmp)
        
        if v_tmp.sum() > 0:
            nn_ci = nci_tmp[v_tmp]
            feat_spatial = np.concatenate([
                X_all_scaled[bp],
                [np.mean(nn_ci), np.std(nn_ci) if len(nn_ci) > 1 else 0,
                 np.min(nn_ci), np.max(nn_ci),
                 dd_km_tmp[0], np.mean(dd_km_tmp[v_tmp]), float(v_tmp.sum())]
            ])
            feat_gs = graph_static_v2_scaled[bp]
            feat_gd = compute_dynamic_v2_fixed(bp, t, ci_matrix, G2, bus_id_list,
                                                bus_id_to_idx, pred_idw_base)
            feat_full = np.concatenate([feat_spatial, feat_gs, feat_gd])
            feat_sub = feat_full[cols]
            feat_sub = np.nan_to_num(feat_sub.reshape(1, -1), 0)
            correction = rf_sub.predict(feat_sub)[0]
            preds_sub.append(pred_idw_base + correction)
        else:
            preds_sub.append(pred_idw_base)
    
    preds_sub = np.array(preds_sub)
    ae = np.abs(true_v - preds_sub)
    mae = np.mean(ae)
    wmape = np.sum(ae) / np.sum(true_v) * 100
    r2 = 1 - np.sum((true_v - preds_sub)**2) / np.sum((true_v - true_v.mean())**2)
    bias = np.mean(preds_sub - true_v)
    p95 = np.percentile(ae, 95)
    
    ablation_results[config_name] = {'wmape': wmape, 'mae': mae, 'r2': r2, 'preds': preds_sub}
    print(f'  {config_name:>20s} {mae:7.1f} {wmape:8.1f} {r2:7.3f} {bias:+7.1f} {p95:7.1f}')

# Contributions marginales
print(f'\n  Contributions marginales:')
base_wmape = ablation_results['A: spatial seul']['wmape']
for name in ablation_configs:
    delta = ablation_results[name]['wmape'] - base_wmape
    print(f'    {name:>20s}: {delta:+.1f} pts vs spatial seul')

print(f'\n  Ablation terminée en {time.time()-t0:.0f}s')

# =====================================================================
# 3. SAFE PROD — Clip asymétrique + score de confiance
# =====================================================================
print('\n' + '=' * 90)
print('  3. SAFE PROD — Clip asymétrique')
print('=' * 90)

pred_idw_all = df_common['IDW'].values
corrections_v2 = preds_v2_raw - pred_idw_all

# Le problème: biais -14.2 sur CI>300 = le modèle sous-estime les hautes CI
# Solution: clip asymétrique — plus permissif vers le haut, plus strict vers le bas
# quand l'IDW prédit un CI élevé

print('  Corrections v2:')
print(f'    Mean: {np.mean(corrections_v2):+.1f}')
print(f'    Std:  {np.std(corrections_v2):.1f}')

# Analyser la direction des corrections par tranche
for lo, hi in [(0, 20), (20, 50), (50, 100), (100, 300), (300, 9999)]:
    m = (pred_idw_all >= lo) & (pred_idw_all < hi)
    if m.sum() > 0:
        mean_corr = np.mean(corrections_v2[m])
        print(f'    IDW {lo}-{hi}: mean correction = {mean_corr:+.1f} (n={m.sum()})')

# Tester des configs safe prod
print(f'\n  Configs safe prod:')
print(f'  {"Config":>35s} {"MAE":>7s} {"wMAPE%":>8s} {"Bias":>7s} {"Bias100+":>9s} {"P95":>7s}')
print('  ' + '-' * 80)

safe_configs = [
    ('raw (aucun clip)', None, None),
    ('clip sym α=2.0', 2.0, 2.0),
    ('clip sym α=1.5', 1.5, 1.5),
    ('clip asym α-=1.0 α+=2.0', 1.0, 2.0),  # strict vers le bas, permissif vers le haut
    ('clip asym α-=0.5 α+=2.0', 0.5, 2.0),
    ('clip asym + boost100+', 'boost', None),  # réduire la correction negative quand IDW>100
]

m_high = true_v >= 100

for name, alpha_neg, alpha_pos in safe_configs:
    if name == 'raw (aucun clip)':
        pred_safe = preds_v2_raw.copy()
    elif alpha_neg == 'boost':
        # Réduire les corrections négatives quand IDW > 100
        pred_safe = pred_idw_all.copy()
        for i in range(len(corrections_v2)):
            corr = corrections_v2[i]
            idw = pred_idw_all[i]
            if idw > 100 and corr < 0:
                # Réduire la correction négative de 50%
                corr = corr * 0.5
            bound = 2.0 * max(abs(idw), 5.0)
            corr = np.clip(corr, -bound, bound)
            pred_safe[i] = max(0, idw + corr)
    else:
        pred_safe = pred_idw_all.copy()
        for i in range(len(corrections_v2)):
            corr = corrections_v2[i]
            idw = pred_idw_all[i]
            bound_base = max(abs(idw), 5.0)
            bound_neg = alpha_neg * bound_base
            bound_pos = alpha_pos * bound_base
            corr = np.clip(corr, -bound_neg, bound_pos)
            pred_safe[i] = max(0, idw + corr)
    
    ae = np.abs(true_v - pred_safe)
    mae = np.mean(ae)
    wmape = np.sum(ae) / np.sum(true_v) * 100
    bias = np.mean(pred_safe - true_v)
    bias_high = np.mean(pred_safe[m_high] - true_v[m_high])
    p95 = np.percentile(ae, 95)
    
    print(f'  {name:>35s} {mae:7.1f} {wmape:8.1f} {bias:+7.1f} {bias_high:+9.1f} {p95:7.1f}')

# =====================================================================
# 4. METRIQUES BUSINESS — Crédibilité produit
# =====================================================================
print('\n' + '=' * 90)
print('  4. METRIQUES BUSINESS — Crédibilité produit')
print('=' * 90)

# Comparer IDW_corrected (ancien champion) vs IDW_graph v2 (nouveau)
models_biz = {
    'IDW_corrected': df_common['IDW_corrected'].values,
    'IDW_graph v2': preds_v2_raw,
}

for name, pred in models_biz.items():
    errors = pred - true_v
    ae = np.abs(errors)
    
    print(f'\n  === {name} ===')
    
    # 4a. Fréquence des erreurs graves
    print(f'  Erreurs graves:')
    for threshold in [20, 50, 100]:
        n = np.sum(ae > threshold)
        pct = n / len(ae) * 100
        print(f'    |erreur| > {threshold} gCO2: {n}/{len(ae)} ({pct:.1f}%)')
    
    # 4b. Erreurs signées négatives sur CI > 100 (sous-estimation dangereuse)
    m100 = true_v >= 100
    if m100.sum() > 0:
        neg_errors_100 = errors[m100] < 0
        n_under = neg_errors_100.sum()
        pct_under = n_under / m100.sum() * 100
        mean_under = np.mean(errors[m100][neg_errors_100]) if n_under > 0 else 0
        print(f'  Sous-estimation CI>100:')
        print(f'    n sous-estime: {n_under}/{m100.sum()} ({pct_under:.0f}%)')
        print(f'    biais moyen sous-estimation: {mean_under:.1f} gCO2/kWh')
    
    # 4c. Calibration par tranche (biais signé)
    print(f'  Calibration (biais signé par tranche):')
    for lo, hi in [(0, 20), (20, 50), (50, 100), (100, 200), (200, 400), (400, 9999)]:
        m = (true_v >= lo) & (true_v < hi)
        if m.sum() > 5:
            bias_t = np.mean(errors[m])
            std_t = np.std(errors[m])
            print(f'    CI {lo:>3d}-{hi:>4d}: bias={bias_t:+6.1f} ± {std_t:5.1f} (n={m.sum()})')
    
    # 4d. Erreurs "aberrantes visuellement" (prédiction dans le mauvais régime)
    print(f'  Erreurs de régime (prédiction dans le mauvais "monde"):')
    # Cas 1: true > 100 mais pred < 50 (fossile prédit comme nucléaire)
    case1 = (true_v > 100) & (pred < 50)
    # Cas 2: true < 20 mais pred > 100 (nucléaire prédit comme fossile)
    case2 = (true_v < 20) & (pred > 100)
    # Cas 3: true > 200 mais pred < 100
    case3 = (true_v > 200) & (pred < 100)
    
    print(f'    True>100, Pred<50:  {case1.sum()}/{m100.sum()} '
          f'({case1.sum()/max(m100.sum(),1)*100:.1f}%)')
    print(f'    True<20, Pred>100:  {case2.sum()}/{(true_v<20).sum()} '
          f'({case2.sum()/max((true_v<20).sum(),1)*100:.1f}%)')
    print(f'    True>200, Pred<100: {case3.sum()}/{(true_v>200).sum()} '
          f'({case3.sum()/max((true_v>200).sum(),1)*100:.1f}%)')

# =====================================================================
# SYNTHESE FINALE
# =====================================================================
print('\n' + '=' * 90)
print('  SYNTHESE AUDIT')
print('=' * 90)

print(f'\n  1. Leakage:    {"✓ PASS" if LEAK_PASS else "⚠ FAIL"}')
print(f'  2. Ablation:   voir tableau ci-dessus')
print(f'  3. Safe prod:  clip asymétrique recommandé')
print(f'  4. Business:   voir métriques ci-dessus')

print(f'\n  Discours propre pour le produit:')
print(f'  ─────────────────────────────────')
print(f'  "IDW_graph v2 est le meilleur compromis global observé,')
print(f'   avec un gain majeur sur les régimes intermédiaires (5-100 gCO2),')
print(f'   au prix d\'une sous-estimation accrue des très fortes CI (>200).')
print(f'   En production: clip asymétrique pour limiter la sous-estimation')
print(f'   sur les régimes élevés."')
print(f'\n  {"=" * 70}')


  AUDIT FINAL — IDW_graph v2

  1. AUDIT LEAKAGE
  Test de corruption: remplacer ci_matrix[t, bus_pos] par une valeur aberrante
  Si les prédictions changent → LEAKAGE

  Résultat corruption test (50 points):
    Max diff:     45.480921
    Mean diff:    3.267925
    n changé (>0.01): 29/50
    ⚠ ATTENTION: 29 prédictions changent
    Cause probable: bus_pos apparaît comme voisin graphe dans le BFS
    → les features 1-hop lisent ci_matrix[t, bus_pos] via les voisins
    Impact moyen: 3.27 gCO2/kWh
    Impact > 1 gCO2 → leakage significatif !

  Vérification: bus cible dans voisins graphe ?
    ✓ Pas de self-loop: le bus cible n'est jamais son propre voisin graphe

  2. ABLATION STRICTE
  Isoler la contribution de chaque composant
  Colonnes: spatial=20, graphe_stat=31, graphe_dyn=19
  Total features v2: 70

                Config     MAE   wMAPE%      R2    Bias     P95
  -----------------------------------------------------------------
       A: spatial seul    18.6     21.8   0.918 

In [17]:

# === Cellule 17 : IDW_graph v3 — ZERO LEAKAGE (bus_pos masque dans ci_clean) ===
# Fix: ci_clean[bus_pos] = NaN AVANT tout calcul de voisins graphe
# Aussi: drop graph static features (ablation: +0.4pts pire)
import time

print('=' * 90)
print('  IDW_graph v3 — ZERO LEAKAGE')
print('  Fix: ci_clean[bus_pos] = NaN avant toute lecture voisins')
print('=' * 90)

# =====================================================================
# ETAPE 1 : Fonction dynamique v3 — bus_pos masque dans ci_clean
# =====================================================================

def compute_dynamic_v3(bus_pos, t, ci_matrix, G2, bus_id_list, bus_id_to_idx,
                       pred_idw_value):
    bid = bus_id_list[bus_pos]
    ci_snap = ci_matrix[t, :]
    ci_clean = np.where(ci_snap == 0, np.nan, ci_snap).astype(float)

    # === FIX LEAKAGE: masquer bus_pos AVANT tout calcul ===
    ci_clean[bus_pos] = np.nan

    ci_proxy = pred_idw_value
    features = []

    # --- 1. CI voisins graphe 1-hop pondere admittance ---
    neighbors_1 = list(G2.neighbors(bid)) if bid in G2 else []

    ci_1hop_vals, w_1hop_vals = [], []
    for n in neighbors_1:
        if n in bus_id_to_idx:
            n_idx = bus_id_to_idx[n]
            ci_val = ci_clean[n_idx]
            if not np.isnan(ci_val):
                ci_1hop_vals.append(ci_val)
                w_1hop_vals.append(G2[bid][n].get('admittance', 1.0))

    if ci_1hop_vals:
        ci_1 = np.array(ci_1hop_vals)
        w_1 = np.array(w_1hop_vals)
        w_1n = w_1 / w_1.sum()
        adm_ci_1 = float(np.sum(w_1n * ci_1))
        features.extend([adm_ci_1, np.std(ci_1), len(ci_1), np.max(ci_1) - np.min(ci_1)])
    else:
        adm_ci_1 = ci_proxy
        features.extend([np.nan, 0, 0, 0])

    # --- 2. CI 2-hop et 3-hop ---
    neighbors_2 = set()
    for n in neighbors_1:
        if n in G2:
            for nn in G2.neighbors(n):
                if nn != bid and nn not in set(neighbors_1):
                    neighbors_2.add(nn)

    ci_2hop = [ci_clean[bus_id_to_idx[n]] for n in neighbors_2
               if n in bus_id_to_idx and not np.isnan(ci_clean[bus_id_to_idx[n]])]
    if ci_2hop:
        features.extend([np.mean(ci_2hop), np.std(ci_2hop), len(ci_2hop)])
    else:
        features.extend([np.nan, 0, 0])

    neighbors_3 = set()
    for n in neighbors_2:
        if n in G2:
            for nn in G2.neighbors(n):
                if nn != bid and nn not in set(neighbors_1) and nn not in neighbors_2:
                    neighbors_3.add(nn)

    ci_3hop = [ci_clean[bus_id_to_idx[n]] for n in neighbors_3
               if n in bus_id_to_idx and not np.isnan(ci_clean[bus_id_to_idx[n]])]
    if ci_3hop:
        features.extend([np.mean(ci_3hop), len(ci_3hop)])
    else:
        features.extend([np.nan, 0])

    # --- 3. Gradient CI sur graphe ---
    gradient_1 = ci_proxy - adm_ci_1
    abs_gradient_1 = abs(gradient_1)
    norm_gradient = gradient_1 / (abs(ci_proxy) + abs(adm_ci_1) + 1e-6)
    features.extend([gradient_1, abs_gradient_1, norm_gradient])

    # --- 4. Diffusion electrique ---
    diffusion_residual = 0.0
    if bid in G2:
        try:
            lengths = nx.single_source_dijkstra_path_length(G2, bid, weight='impedance', cutoff=5.0)
            ci_in_radius = []
            for n, d in lengths.items():
                if n != bid and n in bus_id_to_idx:
                    ci_val = ci_clean[bus_id_to_idx[n]]
                    if not np.isnan(ci_val):
                        ci_in_radius.append((ci_val, d))
            if ci_in_radius:
                ci_vals_r = np.array([x[0] for x in ci_in_radius])
                dists_r = np.array([x[1] for x in ci_in_radius])
                w_elec = 1.0 / (dists_r + 0.01)
                diffused_ci = np.sum(w_elec * ci_vals_r) / np.sum(w_elec)
                diffusion_residual = ci_proxy - diffused_ci
                features.extend([diffused_ci, diffusion_residual, len(ci_in_radius)])
            else:
                features.extend([np.nan, 0, 0])
        except:
            features.extend([np.nan, 0, 0])
    else:
        features.extend([np.nan, 0, 0])

    # --- 5. Contexte national (aussi sans bus_pos) ---
    valid_ci = ci_clean[ci_clean > 0] if np.any(ci_clean > 0) else np.array([])
    ci_national = np.nanmean(valid_ci) if len(valid_ci) > 0 else np.nan
    if not np.isnan(ci_national):
        ratio_local_national = ci_proxy / (ci_national + 1e-6)
        delta_national = ci_proxy - ci_national
    else:
        ratio_local_national = 1.0
        delta_national = 0
    features.extend([ci_national, ratio_local_national, delta_national])

    # --- 6. Ecart IDW spatial vs IDW graphe ---
    idw_vs_graph = ci_proxy - adm_ci_1 if ci_1hop_vals else 0
    features.extend([idw_vs_graph])

    return np.array(features, dtype=float)

print('  compute_dynamic_v3 definie — ci_clean[bus_pos] = NaN')

# =====================================================================
# ETAPE 2 : Entrainement RF v3
# =====================================================================
print('\n--- Etape 2: Entrainement RF v3 ---')
t0 = time.time()

train_snaps = [s for s in range(0, 8760, 200) if s not in set(loo_snapshots)][:40]

X_train_v3, y_resid_v3 = [], []
for ti, t in enumerate(train_snaps):
    ci_snap = np.where(ci_matrix[t, :] == 0, np.nan, ci_matrix[t, :])
    valid_mask_t = ~np.isnan(ci_snap)
    valid_idx = np.where(valid_mask_t)[0]
    if len(valid_idx) < 20:
        continue
    rng = np.random.RandomState(t)
    sample = rng.choice(valid_idx, min(100, len(valid_idx)), replace=False)

    for bus_pos in sample:
        true_ci = ci_snap[bus_pos]
        if np.isnan(true_ci) or true_ci == 0:
            continue

        avail_mask = valid_mask_t.copy()
        avail_mask[bus_pos] = False
        avail = np.where(avail_mask)[0]
        if len(avail) < 3:
            continue
        tree_tmp = cKDTree(bus_coords[avail])
        dd, idx = tree_tmp.query(bus_coords[bus_pos], k=min(7, len(avail)))
        dd_km = dd / 1000
        nci = ci_snap[avail[idx]]
        v = ~np.isnan(nci)
        if v.sum() == 0:
            continue
        dd_c = np.maximum(dd_km[v], 3.0)
        w = 1.0 / (dd_c ** 2.0)
        w = w / w.sum()
        pred_idw = float(np.sum(w * nci[v]))
        resid = true_ci - pred_idw

        nn_ci = nci[v]
        feat_spatial = np.concatenate([
            X_all_scaled[bus_pos],
            [np.mean(nn_ci), np.std(nn_ci) if len(nn_ci) > 1 else 0,
             np.min(nn_ci), np.max(nn_ci),
             dd_km[0], np.mean(dd_km[v]), float(v.sum())]
        ])

        feat_gd = compute_dynamic_v3(bus_pos, t, ci_matrix, G2, bus_id_list,
                                      bus_id_to_idx, pred_idw)

        feat_full = np.concatenate([feat_spatial, feat_gd])
        X_train_v3.append(feat_full)
        y_resid_v3.append(resid)

    if (ti + 1) % 10 == 0:
        print(f'    Snap {ti+1}/{len(train_snaps)} ({time.time()-t0:.0f}s)')

X_train_v3 = np.nan_to_num(np.array(X_train_v3), 0)
y_resid_v3 = np.array(y_resid_v3)
print(f'  Training set: {len(X_train_v3)} samples, {X_train_v3.shape[1]} features')

rf_v3 = RandomForestRegressor(n_estimators=100, max_depth=12, random_state=0, n_jobs=-1,
                               min_samples_leaf=4)
rf_v3.fit(X_train_v3, y_resid_v3)
print(f'  RF R2_train: {rf_v3.score(X_train_v3, y_resid_v3):.3f}')

# Feature importance
all_names_v3 = (
    list(features.columns) +
    ['mean_nn_ci', 'std_nn_ci', 'min_nn_ci', 'max_nn_ci', 'd1', 'mean_d', 'n_valid'] +
    dynamic_v2_names
)
importances_v3 = rf_v3.feature_importances_
if len(all_names_v3) > len(importances_v3):
    all_names_v3 = all_names_v3[:len(importances_v3)]
elif len(all_names_v3) < len(importances_v3):
    all_names_v3 += [f'feat_{i}' for i in range(len(all_names_v3), len(importances_v3))]

sorted_imp_v3 = sorted(zip(all_names_v3, importances_v3), key=lambda x: x[1], reverse=True)
print(f'\n  Top 15 feature importances (v3 CLEAN):')
for name, imp in sorted_imp_v3[:15]:
    bar = '#' * int(imp * 200)
    origin = 'GRAPH-DYN' if name in dynamic_v2_names else 'SPATIAL'
    print(f'    {name:>25s}: {imp:.4f} {bar}  [{origin}]')

imp_spatial = sum(imp for n, imp in sorted_imp_v3 if n not in dynamic_v2_names)
imp_graph_dynamic = sum(imp for n, imp in sorted_imp_v3 if n in dynamic_v2_names)
print(f'\n  Importance: Spatial {imp_spatial:.1%}, Graph dyn {imp_graph_dynamic:.1%}')
print(f'  Entrainement: {time.time()-t0:.0f}s')

# =====================================================================
# ETAPE 3 : Predictions LOO v3
# =====================================================================
print('\n--- Etape 3: Predictions LOO ---')
t0 = time.time()

preds_v3 = []
ALPHA = 1.0

for ri, (_, row) in enumerate(df_common.iterrows()):
    bus_pos = int(row['bus_pos'])
    t = int(row['t'])
    pred_idw_base = row['IDW']

    mask_tmp = np.ones(len(bus_coords), dtype=bool)
    mask_tmp[bus_pos] = False
    tree_tmp = cKDTree(bus_coords[mask_tmp])
    dd_tmp, idx_tmp = tree_tmp.query(bus_coords[bus_pos], k=7)
    dd_km_tmp = dd_tmp / 1000
    orig_tmp = np.where(mask_tmp)[0]
    nci_tmp = ci_matrix[t, orig_tmp[idx_tmp]]
    nci_tmp = np.where(nci_tmp == 0, np.nan, nci_tmp)
    v_tmp = ~np.isnan(nci_tmp)

    if v_tmp.sum() > 0:
        nn_ci = nci_tmp[v_tmp]
        feat_spatial = np.concatenate([
            X_all_scaled[bus_pos],
            [np.mean(nn_ci), np.std(nn_ci) if len(nn_ci) > 1 else 0,
             np.min(nn_ci), np.max(nn_ci),
             dd_km_tmp[0], np.mean(dd_km_tmp[v_tmp]), float(v_tmp.sum())]
        ])

        feat_gd = compute_dynamic_v3(bus_pos, t, ci_matrix, G2, bus_id_list,
                                      bus_id_to_idx, pred_idw_base)

        feat_full = np.concatenate([feat_spatial, feat_gd]).reshape(1, -1)
        feat_full = np.nan_to_num(feat_full, 0)

        correction = rf_v3.predict(feat_full)[0]
        bound = ALPHA * max(abs(pred_idw_base), 5.0)
        preds_v3.append(max(0, pred_idw_base + np.clip(correction, -bound, bound)))
    else:
        preds_v3.append(pred_idw_base)

    if (ri + 1) % 200 == 0:
        print(f'    {ri+1}/{len(df_common)} ({time.time()-t0:.0f}s)')

preds_v3 = np.array(preds_v3)
print(f'  Predictions: {time.time()-t0:.0f}s')

# =====================================================================
# ETAPE 4 : Audit leakage v3
# =====================================================================
print('\n--- Etape 4: Audit leakage ---')

np.random.seed(42)
test_sample = np.random.choice(len(df_common), 50, replace=False)

preds_orig3, preds_corr3 = [], []

for idx in test_sample:
    row = df_common.iloc[idx]
    bp, t = int(row['bus_pos']), int(row['t'])
    pred_idw_base = row['IDW']

    mask_tmp = np.ones(len(bus_coords), dtype=bool)
    mask_tmp[bp] = False
    tree_tmp = cKDTree(bus_coords[mask_tmp])
    dd_tmp, idx_tmp = tree_tmp.query(bus_coords[bp], k=7)
    dd_km_tmp = dd_tmp / 1000
    orig_tmp = np.where(mask_tmp)[0]
    nci_tmp = ci_matrix[t, orig_tmp[idx_tmp]]
    nci_tmp = np.where(nci_tmp == 0, np.nan, nci_tmp)
    v_tmp = ~np.isnan(nci_tmp)

    if v_tmp.sum() > 0:
        nn_ci = nci_tmp[v_tmp]
        feat_spatial = np.concatenate([
            X_all_scaled[bp],
            [np.mean(nn_ci), np.std(nn_ci) if len(nn_ci) > 1 else 0,
             np.min(nn_ci), np.max(nn_ci),
             dd_km_tmp[0], np.mean(dd_km_tmp[v_tmp]), float(v_tmp.sum())]
        ])

        # Prediction originale
        feat_gd = compute_dynamic_v3(bp, t, ci_matrix, G2, bus_id_list,
                                      bus_id_to_idx, pred_idw_base)
        feat_full = np.concatenate([feat_spatial, feat_gd]).reshape(1, -1)
        feat_full = np.nan_to_num(feat_full, 0)
        correction = rf_v3.predict(feat_full)[0]
        preds_orig3.append(pred_idw_base + correction)

        # Corrompre ci_matrix[t, bus_pos]
        original_val = ci_matrix[t, bp]
        ci_matrix[t, bp] = 99999.0

        feat_gd_c = compute_dynamic_v3(bp, t, ci_matrix, G2, bus_id_list,
                                        bus_id_to_idx, pred_idw_base)
        feat_full_c = np.concatenate([feat_spatial, feat_gd_c]).reshape(1, -1)
        feat_full_c = np.nan_to_num(feat_full_c, 0)
        correction_c = rf_v3.predict(feat_full_c)[0]
        preds_corr3.append(pred_idw_base + correction_c)

        ci_matrix[t, bp] = original_val
    else:
        preds_orig3.append(pred_idw_base)
        preds_corr3.append(pred_idw_base)

preds_orig3 = np.array(preds_orig3)
preds_corr3 = np.array(preds_corr3)

diff3 = np.abs(preds_orig3 - preds_corr3)
max_diff3 = np.max(diff3)
mean_diff3 = np.mean(diff3)
n_changed3 = np.sum(diff3 > 0.01)

print(f'\n  Resultat corruption test (50 points):')
print(f'    Max diff:     {max_diff3:.6f}')
print(f'    Mean diff:    {mean_diff3:.6f}')
print(f'    n change (>0.01): {n_changed3}/50')

if max_diff3 < 0.01:
    print(f'    PASS: ZERO LEAKAGE confirme')
    LEAK_V3 = True
else:
    print(f'    FAIL: leakage residuel detecte')
    LEAK_V3 = False

# =====================================================================
# ETAPE 5 : Resultats comparatifs + Safe Prod
# =====================================================================
print('\n' + '=' * 90)
print('  RESULTATS v3 CLEAN')
print('=' * 90)

true_v = df_common['true_ci'].values

# Safe prod: boost100+
preds_v3_boost = []
for ri, (_, row) in enumerate(df_common.iterrows()):
    bus_pos = int(row['bus_pos'])
    t = int(row['t'])
    pred_idw_base = row['IDW']

    mask_tmp = np.ones(len(bus_coords), dtype=bool)
    mask_tmp[bus_pos] = False
    tree_tmp = cKDTree(bus_coords[mask_tmp])
    dd_tmp, idx_tmp = tree_tmp.query(bus_coords[bus_pos], k=7)
    dd_km_tmp = dd_tmp / 1000
    orig_tmp = np.where(mask_tmp)[0]
    nci_tmp = ci_matrix[t, orig_tmp[idx_tmp]]
    nci_tmp = np.where(nci_tmp == 0, np.nan, nci_tmp)
    v_tmp = ~np.isnan(nci_tmp)

    if v_tmp.sum() > 0:
        nn_ci = nci_tmp[v_tmp]
        feat_spatial = np.concatenate([
            X_all_scaled[bus_pos],
            [np.mean(nn_ci), np.std(nn_ci) if len(nn_ci) > 1 else 0,
             np.min(nn_ci), np.max(nn_ci),
             dd_km_tmp[0], np.mean(dd_km_tmp[v_tmp]), float(v_tmp.sum())]
        ])
        feat_gd = compute_dynamic_v3(bus_pos, t, ci_matrix, G2, bus_id_list,
                                      bus_id_to_idx, pred_idw_base)
        feat_full = np.concatenate([feat_spatial, feat_gd]).reshape(1, -1)
        feat_full = np.nan_to_num(feat_full, 0)
        correction = rf_v3.predict(feat_full)[0]

        # Boost100+: reduce negative corrections 50% when IDW > 100
        if pred_idw_base > 100 and correction < 0:
            correction = correction * 0.5

        bound = ALPHA * max(abs(pred_idw_base), 5.0)
        preds_v3_boost.append(max(0, pred_idw_base + np.clip(correction, -bound, bound)))
    else:
        preds_v3_boost.append(pred_idw_base)

preds_v3_boost = np.array(preds_v3_boost)

COMPARE = {
    'IDW (baseline)': df_common['IDW'].values,
    'IDW_corrected (v0)': df_common['IDW_corrected'].values,
    'IDW_graph v1': preds_graph,
    'IDW_graph v2 (leak)': preds_v2_clip,
    'IDW_graph v3 (clean)': preds_v3,
    'v3 + boost100+': preds_v3_boost,
}

print(f'\n  {"Modele":>25s} {"MAE":>7s} {"wMAPE%":>8s} {"R2":>7s} {"Bias":>7s}  {"P95":>7s}')
print('  ' + '-' * 70)

for name, pred in COMPARE.items():
    ae = np.abs(true_v - pred)
    mae = np.mean(ae)
    wmape = np.sum(ae) / np.sum(true_v) * 100
    r2 = 1 - np.sum((true_v - pred)**2) / np.sum((true_v - true_v.mean())**2)
    bias = np.mean(pred - true_v)
    p95 = np.percentile(ae, 95)
    flag = ' <<<' if 'v3' in name else ''
    print(f'  {name:>25s} {mae:7.1f} {wmape:8.1f} {r2:7.3f} {bias:+7.1f}  {p95:7.1f}{flag}')

# Detail par tranche
ci_bins = [(0, 5), (5, 20), (20, 50), (50, 100), (100, 300), (300, 9999)]
for name, pred in [('IDW', df_common['IDW'].values), ('v3 clean', preds_v3), ('v3 boost', preds_v3_boost)]:
    ae = np.abs(true_v - pred)
    bias_v = pred - true_v
    print(f'\n  {name}:')
    for lo, hi in ci_bins:
        m = (true_v >= lo) & (true_v < hi)
        if m.sum() == 0:
            continue
        print(f'    CI [{lo:>4d},{hi:>4d}): n={m.sum():>3d}, MAE={np.mean(ae[m]):6.1f}, '
              f'bias={np.mean(bias_v[m]):+6.1f}, wMAPE={np.sum(ae[m])/np.sum(true_v[m])*100:5.1f}%')

# =====================================================================
# ETAPE 6 : Business metrics
# =====================================================================
print('\n' + '=' * 90)
print('  BUSINESS METRICS')
print('=' * 90)

for name, pred in [('IDW', df_common['IDW'].values), ('v3 clean', preds_v3), ('v3 boost', preds_v3_boost)]:
    ae = np.abs(true_v - pred)
    pct_20 = np.mean(ae > 20) * 100
    pct_50 = np.mean(ae > 50) * 100
    pct_100 = np.mean(ae > 100) * 100

    regime_true = np.where(true_v < 50, 'nucleaire', np.where(true_v < 200, 'mix', 'fossile'))
    regime_pred = np.where(pred < 50, 'nucleaire', np.where(pred < 200, 'mix', 'fossile'))
    misclass = np.mean(regime_true != regime_pred) * 100
    foss_nuc = np.sum((regime_true == 'fossile') & (regime_pred == 'nucleaire'))

    high_mask = true_v > 100
    if high_mask.sum() > 0:
        under_est_high = np.mean(pred[high_mask] < true_v[high_mask] - 20) * 100
    else:
        under_est_high = 0

    print(f'\n  {name}:')
    print(f'    Erreur > 20g: {pct_20:.1f}%, > 50g: {pct_50:.1f}%, > 100g: {pct_100:.1f}%')
    print(f'    Regime misclass: {misclass:.1f}%, fossile->nucleaire: {foss_nuc}')
    print(f'    Sous-estimation >20g sur CI>100: {under_est_high:.1f}%')

print('\n' + '=' * 90)
print('  VERDICT FINAL')
print('=' * 90)
print(f'  Leakage: {"PASS" if LEAK_V3 else "FAIL"}')
print(f'  Modele: IDW_graph v3 (spatial + graph dyn, sans graph static)')
print(f'  Alpha clip: {ALPHA}')



  IDW_graph v3 — ZERO LEAKAGE
  Fix: ci_clean[bus_pos] = NaN avant toute lecture voisins
  compute_dynamic_v3 definie — ci_clean[bus_pos] = NaN

--- Etape 2: Entrainement RF v3 ---
    Snap 10/37 (0s)
    Snap 20/37 (1s)
    Snap 30/37 (1s)
  Training set: 3700 samples, 39 features
  RF R2_train: 0.879

  Top 15 feature importances (v3 CLEAN):
                      cap_gas: 0.2663 #####################################################  [SPATIAL]
            idw_vs_graph_1hop: 0.1678 #################################  [GRAPH-DYN]
           gradient_idw_graph: 0.1559 ###############################  [GRAPH-DYN]
                  std_ci_1hop: 0.0435 ########  [GRAPH-DYN]
                range_ci_1hop: 0.0402 ########  [GRAPH-DYN]
                  cap_biomass: 0.0338 ######  [SPATIAL]
       abs_gradient_idw_graph: 0.0282 #####  [GRAPH-DYN]
           ratio_idw_national: 0.0226 ####  [GRAPH-DYN]
           delta_idw_national: 0.0195 ###  [GRAPH-DYN]
                 fossil_ratio: 0.0190 #

In [18]:
# === Cellule 18 : COMPARAISON Dijkstra vs Sparse Graph Filter ===
# 4 variantes, meme framework LOO, meme RF, meme leakage audit
# V1: v3 Dijkstra (baseline Cell 17)
# V2: v3 sans diffusion (drop 3 features Dijkstra)
# V3: v3 sparse filter (g_mix + delta_hop12 + delta_hop23 remplacent Dijkstra)
# V4: v3 sparse+ (g_mix + delta_1hop + delta_hop12 + delta_hop23)
import time
from scipy.sparse import csr_matrix

print('=' * 90)
print('  COMPARAISON : Dijkstra vs Sparse Graph Filter')
print('=' * 90)

# =====================================================================
# ETAPE 0 : Construction matrice W_norm sparse (admittance, row-normalized)
# =====================================================================
print('\n--- Etape 0: Construction W_norm sparse ---')

n_buses = len(bus_id_list)
rows, cols, vals = [], [], []
for u, v, d in G2.edges(data=True):
    if u in bus_id_to_idx and v in bus_id_to_idx:
        i, j = bus_id_to_idx[u], bus_id_to_idx[v]
        adm = d.get('admittance', 1.0)
        rows.extend([i, j])
        cols.extend([j, i])
        vals.extend([adm, adm])

W = csr_matrix((vals, (rows, cols)), shape=(n_buses, n_buses))
# Row-normalize
row_sums = np.array(W.sum(axis=1)).flatten()
row_sums[row_sums == 0] = 1.0
W_norm = csr_matrix(W / row_sums[:, np.newaxis])

print(f'  W_norm: {W_norm.shape}, nnz={W_norm.nnz}, density={W_norm.nnz/n_buses**2:.4f}')

# =====================================================================
# Fonctions dynamiques par variante
# =====================================================================

def _base_features_c18(bus_pos, ci_clean, G2, bus_id_list, bus_id_to_idx, pred_idw):
    """Features communes: 1-hop, 2-hop, 3-hop, gradient, national, idw_vs_graph (16 features)"""
    bid = bus_id_list[bus_pos]
    ci_proxy = pred_idw
    features = []

    # 1-hop
    neighbors_1 = list(G2.neighbors(bid)) if bid in G2 else []
    ci_1hop_vals, w_1hop_vals = [], []
    for n in neighbors_1:
        if n in bus_id_to_idx:
            n_idx = bus_id_to_idx[n]
            ci_val = ci_clean[n_idx]
            if not np.isnan(ci_val):
                ci_1hop_vals.append(ci_val)
                w_1hop_vals.append(G2[bid][n].get('admittance', 1.0))
    if ci_1hop_vals:
        ci_1 = np.array(ci_1hop_vals)
        w_1 = np.array(w_1hop_vals)
        w_1n = w_1 / w_1.sum()
        adm_ci_1 = float(np.sum(w_1n * ci_1))
        features.extend([adm_ci_1, np.std(ci_1), len(ci_1), np.max(ci_1) - np.min(ci_1)])
    else:
        adm_ci_1 = ci_proxy
        features.extend([np.nan, 0, 0, 0])

    # 2-hop
    neighbors_2 = set()
    for n in neighbors_1:
        if n in G2:
            for nn in G2.neighbors(n):
                if nn != bid and nn not in set(neighbors_1):
                    neighbors_2.add(nn)
    ci_2hop = [ci_clean[bus_id_to_idx[n]] for n in neighbors_2
               if n in bus_id_to_idx and not np.isnan(ci_clean[bus_id_to_idx[n]])]
    if ci_2hop:
        features.extend([np.mean(ci_2hop), np.std(ci_2hop), len(ci_2hop)])
    else:
        features.extend([np.nan, 0, 0])

    # 3-hop
    neighbors_3 = set()
    for n in neighbors_2:
        if n in G2:
            for nn in G2.neighbors(n):
                if nn != bid and nn not in set(neighbors_1) and nn not in neighbors_2:
                    neighbors_3.add(nn)
    ci_3hop = [ci_clean[bus_id_to_idx[n]] for n in neighbors_3
               if n in bus_id_to_idx and not np.isnan(ci_clean[bus_id_to_idx[n]])]
    if ci_3hop:
        features.extend([np.mean(ci_3hop), len(ci_3hop)])
    else:
        features.extend([np.nan, 0])

    # Gradient
    gradient_1 = ci_proxy - adm_ci_1
    features.extend([gradient_1, abs(gradient_1), gradient_1 / (abs(ci_proxy) + abs(adm_ci_1) + 1e-6)])

    # National (sans bus_pos)
    valid_ci = ci_clean[ci_clean > 0] if np.any(ci_clean > 0) else np.array([])
    ci_national = np.nanmean(valid_ci) if len(valid_ci) > 0 else np.nan
    if not np.isnan(ci_national):
        features.extend([ci_national, ci_proxy / (ci_national + 1e-6), ci_proxy - ci_national])
    else:
        features.extend([np.nan, 1.0, 0])

    # IDW vs graph
    features.append(ci_proxy - adm_ci_1 if ci_1hop_vals else 0)

    return np.array(features, dtype=float)


def dynamic_dijkstra_c18(bus_pos, ci_clean, G2, bus_id_list, bus_id_to_idx, pred_idw):
    """V1: base (16) + Dijkstra (3) = 19 features"""
    base = _base_features_c18(bus_pos, ci_clean, G2, bus_id_list, bus_id_to_idx, pred_idw)
    bid = bus_id_list[bus_pos]
    ci_proxy = pred_idw

    if bid in G2:
        try:
            lengths = nx.single_source_dijkstra_path_length(G2, bid, weight='impedance', cutoff=5.0)
            ci_in_radius = []
            for n, d in lengths.items():
                if n != bid and n in bus_id_to_idx:
                    ci_val = ci_clean[bus_id_to_idx[n]]
                    if not np.isnan(ci_val):
                        ci_in_radius.append((ci_val, d))
            if ci_in_radius:
                ci_vals_r = np.array([x[0] for x in ci_in_radius])
                dists_r = np.array([x[1] for x in ci_in_radius])
                w_elec = 1.0 / (dists_r + 0.01)
                diffused_ci = np.sum(w_elec * ci_vals_r) / np.sum(w_elec)
                diff_feats = [diffused_ci, ci_proxy - diffused_ci, len(ci_in_radius)]
            else:
                diff_feats = [np.nan, 0, 0]
        except:
            diff_feats = [np.nan, 0, 0]
    else:
        diff_feats = [np.nan, 0, 0]

    # Insert diffusion at position 12 (after gradient[9:12], before national[12:15])
    out = np.concatenate([base[:12], diff_feats, base[12:]])
    return out


def dynamic_no_diff_c18(bus_pos, ci_clean, G2, bus_id_list, bus_id_to_idx, pred_idw):
    """V2: base (16) + zeros (3) = 19 features (Dijkstra dropped)"""
    base = _base_features_c18(bus_pos, ci_clean, G2, bus_id_list, bus_id_to_idx, pred_idw)
    out = np.concatenate([base[:12], [np.nan, 0, 0], base[12:]])
    return out


def dynamic_sparse_c18(bus_pos, ci_clean, W_norm, pred_idw, G2, bus_id_list, bus_id_to_idx):
    """V3: base (16) + sparse filter (3) = 19 features
    Replace Dijkstra with: g_mix, delta_hop12, delta_hop23"""
    base = _base_features_c18(bus_pos, ci_clean, G2, bus_id_list, bus_id_to_idx, pred_idw)

    # Sparse polynomial: g_k = W_norm^k @ ci_clean
    ci_vec = ci_clean.copy()
    ci_vec[np.isnan(ci_vec)] = 0  # sparse matmul needs no NaN

    g1 = W_norm.dot(ci_vec)  # 1-hop smooth
    g2 = W_norm.dot(g1)       # 2-hop smooth
    g3 = W_norm.dot(g2)       # 3-hop smooth

    # g_mix polynomial: weighted combination
    g_mix = 0.60 * pred_idw + 0.25 * g1[bus_pos] + 0.10 * g2[bus_pos] + 0.05 * g3[bus_pos]

    # Delta features (band-pass info)
    delta_hop12 = g1[bus_pos] - g2[bus_pos]
    delta_hop23 = g2[bus_pos] - g3[bus_pos]

    sparse_feats = [g_mix, delta_hop12, delta_hop23]
    out = np.concatenate([base[:12], sparse_feats, base[12:]])
    return out


def dynamic_sparse_plus_c18(bus_pos, ci_clean, W_norm, pred_idw, G2, bus_id_list, bus_id_to_idx):
    """V4: base (16) + sparse filter (4) = 20 features
    g_mix, delta_1hop, delta_hop12, delta_hop23"""
    base = _base_features_c18(bus_pos, ci_clean, G2, bus_id_list, bus_id_to_idx, pred_idw)

    ci_vec = ci_clean.copy()
    ci_vec[np.isnan(ci_vec)] = 0

    g1 = W_norm.dot(ci_vec)
    g2 = W_norm.dot(g1)
    g3 = W_norm.dot(g2)

    g_mix = 0.60 * pred_idw + 0.25 * g1[bus_pos] + 0.10 * g2[bus_pos] + 0.05 * g3[bus_pos]
    delta_1hop = pred_idw - g1[bus_pos]   # local vs 1-hop (anti-lissage signal)
    delta_hop12 = g1[bus_pos] - g2[bus_pos]
    delta_hop23 = g2[bus_pos] - g3[bus_pos]

    sparse_feats = [g_mix, delta_1hop, delta_hop12, delta_hop23]
    out = np.concatenate([base[:12], sparse_feats, base[12:]])
    return out


VARIANTS = {
    'V1_dijkstra': {'fn': 'dijkstra', 'n_diff_feats': 3},
    'V2_no_diff':  {'fn': 'no_diff',  'n_diff_feats': 3},
    'V3_sparse':   {'fn': 'sparse',   'n_diff_feats': 3},
    'V4_sparse+':  {'fn': 'sparse+',  'n_diff_feats': 4},
}

# =====================================================================
# ETAPE 1 : Entrainement RF par variante
# =====================================================================
print('\n--- Etape 1: Entrainement des 4 variantes ---')

train_snaps = [s for s in range(0, 8760, 200) if s not in set(loo_snapshots)][:40]
rfs_c18 = {}
train_times = {}

for vname, vconf in VARIANTS.items():
    t0 = time.time()
    X_train, y_train = [], []

    for ti, t in enumerate(train_snaps):
        ci_snap = np.where(ci_matrix[t, :] == 0, np.nan, ci_matrix[t, :])
        valid_mask_t = ~np.isnan(ci_snap)
        valid_idx = np.where(valid_mask_t)[0]
        if len(valid_idx) < 20:
            continue
        rng = np.random.RandomState(t)
        sample = rng.choice(valid_idx, min(100, len(valid_idx)), replace=False)

        for bus_pos in sample:
            true_ci = ci_snap[bus_pos]
            if np.isnan(true_ci) or true_ci == 0:
                continue

            avail_mask = valid_mask_t.copy()
            avail_mask[bus_pos] = False
            avail = np.where(avail_mask)[0]
            if len(avail) < 3:
                continue
            tree_tmp = cKDTree(bus_coords[avail])
            dd, idx = tree_tmp.query(bus_coords[bus_pos], k=min(7, len(avail)))
            dd_km = dd / 1000
            nci = ci_snap[avail[idx]]
            v = ~np.isnan(nci)
            if v.sum() == 0:
                continue
            dd_c = np.maximum(dd_km[v], 3.0)
            w = 1.0 / (dd_c ** 2.0)
            w = w / w.sum()
            pred_idw = float(np.sum(w * nci[v]))
            resid = true_ci - pred_idw

            nn_ci = nci[v]
            feat_spatial = np.concatenate([
                X_all_scaled[bus_pos],
                [np.mean(nn_ci), np.std(nn_ci) if len(nn_ci) > 1 else 0,
                 np.min(nn_ci), np.max(nn_ci),
                 dd_km[0], np.mean(dd_km[v]), float(v.sum())]
            ])

            # ci_clean avec masque leakage
            ci_clean = np.where(ci_snap == 0, np.nan, ci_snap).astype(float)
            ci_clean[bus_pos] = np.nan

            if vconf['fn'] == 'dijkstra':
                feat_gd = dynamic_dijkstra_c18(bus_pos, ci_clean, G2, bus_id_list, bus_id_to_idx, pred_idw)
            elif vconf['fn'] == 'no_diff':
                feat_gd = dynamic_no_diff_c18(bus_pos, ci_clean, G2, bus_id_list, bus_id_to_idx, pred_idw)
            elif vconf['fn'] == 'sparse':
                feat_gd = dynamic_sparse_c18(bus_pos, ci_clean, W_norm, pred_idw, G2, bus_id_list, bus_id_to_idx)
            elif vconf['fn'] == 'sparse+':
                feat_gd = dynamic_sparse_plus_c18(bus_pos, ci_clean, W_norm, pred_idw, G2, bus_id_list, bus_id_to_idx)

            feat_full = np.concatenate([feat_spatial, feat_gd])
            X_train.append(feat_full)
            y_train.append(resid)

    X_train = np.nan_to_num(np.array(X_train), 0)
    y_train = np.array(y_train)

    rf = RandomForestRegressor(n_estimators=100, max_depth=12, random_state=0, n_jobs=-1,
                                min_samples_leaf=4)
    rf.fit(X_train, y_train)
    rfs_c18[vname] = rf
    train_times[vname] = time.time() - t0
    print(f'  {vname}: {len(X_train)} samples, {X_train.shape[1]} feats, '
          f'R2_train={rf.score(X_train, y_train):.3f}, time={train_times[vname]:.0f}s')

# =====================================================================
# ETAPE 2 : Predictions LOO par variante (+ boost100+ + clip)
# =====================================================================
print('\n--- Etape 2: Predictions LOO ---')

ALPHA = 1.0
all_preds_c18 = {}
pred_times = {}

for vname, vconf in VARIANTS.items():
    t0 = time.time()
    preds = []
    rf = rfs_c18[vname]

    for ri, (_, row) in enumerate(df_common.iterrows()):
        bus_pos = int(row['bus_pos'])
        t = int(row['t'])
        pred_idw_base = row['IDW']

        mask_tmp = np.ones(len(bus_coords), dtype=bool)
        mask_tmp[bus_pos] = False
        tree_tmp = cKDTree(bus_coords[mask_tmp])
        dd_tmp, idx_tmp = tree_tmp.query(bus_coords[bus_pos], k=7)
        dd_km_tmp = dd_tmp / 1000
        orig_tmp = np.where(mask_tmp)[0]
        nci_tmp = ci_matrix[t, orig_tmp[idx_tmp]]
        nci_tmp = np.where(nci_tmp == 0, np.nan, nci_tmp)
        v_tmp = ~np.isnan(nci_tmp)

        if v_tmp.sum() > 0:
            nn_ci = nci_tmp[v_tmp]
            feat_spatial = np.concatenate([
                X_all_scaled[bus_pos],
                [np.mean(nn_ci), np.std(nn_ci) if len(nn_ci) > 1 else 0,
                 np.min(nn_ci), np.max(nn_ci),
                 dd_km_tmp[0], np.mean(dd_km_tmp[v_tmp]), float(v_tmp.sum())]
            ])

            ci_clean = np.where(ci_matrix[t, :] == 0, np.nan, ci_matrix[t, :]).astype(float)
            ci_clean[bus_pos] = np.nan

            if vconf['fn'] == 'dijkstra':
                feat_gd = dynamic_dijkstra_c18(bus_pos, ci_clean, G2, bus_id_list, bus_id_to_idx, pred_idw_base)
            elif vconf['fn'] == 'no_diff':
                feat_gd = dynamic_no_diff_c18(bus_pos, ci_clean, G2, bus_id_list, bus_id_to_idx, pred_idw_base)
            elif vconf['fn'] == 'sparse':
                feat_gd = dynamic_sparse_c18(bus_pos, ci_clean, W_norm, pred_idw_base, G2, bus_id_list, bus_id_to_idx)
            elif vconf['fn'] == 'sparse+':
                feat_gd = dynamic_sparse_plus_c18(bus_pos, ci_clean, W_norm, pred_idw_base, G2, bus_id_list, bus_id_to_idx)

            feat_full = np.concatenate([feat_spatial, feat_gd]).reshape(1, -1)
            feat_full = np.nan_to_num(feat_full, 0)
            correction = rf.predict(feat_full)[0]

            # Boost100+
            if pred_idw_base > 100 and correction < 0:
                correction = correction * 0.5

            bound = ALPHA * max(abs(pred_idw_base), 5.0)
            preds.append(max(0, pred_idw_base + np.clip(correction, -bound, bound)))
        else:
            preds.append(pred_idw_base)

        if (ri + 1) % 200 == 0:
            elapsed = time.time() - t0
            print(f'    {vname}: {ri+1}/{len(df_common)} ({elapsed:.0f}s)')

    all_preds_c18[vname] = np.array(preds)
    pred_times[vname] = time.time() - t0
    print(f'  {vname}: predictions done in {pred_times[vname]:.0f}s')

# =====================================================================
# ETAPE 3 : Audit leakage (variantes sparse uniquement)
# =====================================================================
print('\n--- Etape 3: Audit leakage (V3, V4) ---')

np.random.seed(42)
test_sample_c18 = np.random.choice(len(df_common), 50, replace=False)

for vname in ['V3_sparse', 'V4_sparse+']:
    vconf = VARIANTS[vname]
    rf = rfs_c18[vname]
    preds_orig, preds_corr = [], []

    for idx in test_sample_c18:
        row = df_common.iloc[idx]
        bp, t = int(row['bus_pos']), int(row['t'])
        pred_idw_base = row['IDW']

        mask_tmp = np.ones(len(bus_coords), dtype=bool)
        mask_tmp[bp] = False
        tree_tmp = cKDTree(bus_coords[mask_tmp])
        dd_tmp, idx_tmp = tree_tmp.query(bus_coords[bp], k=7)
        dd_km_tmp = dd_tmp / 1000
        orig_tmp = np.where(mask_tmp)[0]
        nci_tmp = ci_matrix[t, orig_tmp[idx_tmp]]
        nci_tmp = np.where(nci_tmp == 0, np.nan, nci_tmp)
        v_tmp = ~np.isnan(nci_tmp)

        if v_tmp.sum() > 0:
            nn_ci = nci_tmp[v_tmp]
            feat_spatial = np.concatenate([
                X_all_scaled[bp],
                [np.mean(nn_ci), np.std(nn_ci) if len(nn_ci) > 1 else 0,
                 np.min(nn_ci), np.max(nn_ci),
                 dd_km_tmp[0], np.mean(dd_km_tmp[v_tmp]), float(v_tmp.sum())]
            ])

            # Prediction originale
            ci_clean = np.where(ci_matrix[t, :] == 0, np.nan, ci_matrix[t, :]).astype(float)
            ci_clean[bp] = np.nan
            if vconf['fn'] == 'sparse':
                feat_gd = dynamic_sparse_c18(bp, ci_clean, W_norm, pred_idw_base, G2, bus_id_list, bus_id_to_idx)
            else:
                feat_gd = dynamic_sparse_plus_c18(bp, ci_clean, W_norm, pred_idw_base, G2, bus_id_list, bus_id_to_idx)
            feat_full = np.concatenate([feat_spatial, feat_gd]).reshape(1, -1)
            feat_full = np.nan_to_num(feat_full, 0)
            correction = rf.predict(feat_full)[0]
            preds_orig.append(pred_idw_base + correction)

            # Corrompre ci_matrix
            original_val = ci_matrix[t, bp]
            ci_matrix[t, bp] = 99999.0
            ci_clean2 = np.where(ci_matrix[t, :] == 0, np.nan, ci_matrix[t, :]).astype(float)
            ci_clean2[bp] = np.nan
            if vconf['fn'] == 'sparse':
                feat_gd_c = dynamic_sparse_c18(bp, ci_clean2, W_norm, pred_idw_base, G2, bus_id_list, bus_id_to_idx)
            else:
                feat_gd_c = dynamic_sparse_plus_c18(bp, ci_clean2, W_norm, pred_idw_base, G2, bus_id_list, bus_id_to_idx)
            feat_full_c = np.concatenate([feat_spatial, feat_gd_c]).reshape(1, -1)
            feat_full_c = np.nan_to_num(feat_full_c, 0)
            correction_c = rf.predict(feat_full_c)[0]
            preds_corr.append(pred_idw_base + correction_c)
            ci_matrix[t, bp] = original_val
        else:
            preds_orig.append(pred_idw_base)
            preds_corr.append(pred_idw_base)

    diff = np.abs(np.array(preds_orig) - np.array(preds_corr))
    print(f'  {vname}: max_diff={np.max(diff):.6f}, mean_diff={np.mean(diff):.6f}, '
          f'n_changed={np.sum(diff > 0.01)}/50  '
          f'{"PASS" if np.max(diff) < 0.01 else "FAIL"}')

# =====================================================================
# ETAPE 4 : Tableau comparatif final
# =====================================================================
print('\n' + '=' * 90)
print('  RESULTATS COMPARATIFS')
print('=' * 90)

true_v = df_common['true_ci'].values

# Ajouter IDW baseline pour reference
all_compare_c18 = {'IDW (baseline)': df_common['IDW'].values}
all_compare_c18.update(all_preds_c18)

print(f'\n  {"Variante":>20s} {"MAE":>7s} {"wMAPE%":>8s} {"R2":>7s} {"Bias":>7s} '
      f'{"P95":>7s} {"Train_s":>8s} {"Pred_s":>8s}')
print('  ' + '-' * 85)

for name, pred in all_compare_c18.items():
    ae = np.abs(true_v - pred)
    mae = np.mean(ae)
    wmape = np.sum(ae) / np.sum(true_v) * 100
    r2 = 1 - np.sum((true_v - pred)**2) / np.sum((true_v - true_v.mean())**2)
    bias = np.mean(pred - true_v)
    p95 = np.percentile(ae, 95)
    tt = train_times.get(name, 0)
    pt = pred_times.get(name, 0)
    flag = ' <<<' if 'sparse' in name.lower() else ''
    print(f'  {name:>20s} {mae:7.1f} {wmape:8.1f} {r2:7.3f} {bias:+7.1f} '
          f'{p95:7.1f} {tt:8.0f} {pt:8.0f}{flag}')

# =====================================================================
# ETAPE 5 : Feature importance comparee (Dijkstra vs Sparse)
# =====================================================================
print('\n--- Feature importance: Dijkstra vs Sparse ---')

diff_feat_names = {
    'V1_dijkstra': ['diffused_ci_elec', 'diffusion_residual', 'n_in_radius'],
    'V2_no_diff':  ['zero_0', 'zero_1', 'zero_2'],
    'V3_sparse':   ['g_mix', 'delta_hop12', 'delta_hop23'],
    'V4_sparse+':  ['g_mix', 'delta_1hop', 'delta_hop12', 'delta_hop23'],
}

for vname, rf in rfs_c18.items():
    imp = rf.feature_importances_
    # Diffusion features position: spatial(13+7=20) + base_dyn(12) = index 32
    n_spatial = X_all_scaled.shape[1] + 7
    n_base_dyn = 12  # 1hop(4) + 2hop(3) + 3hop(2) + gradient(3)
    diff_start = n_spatial + n_base_dyn
    n_diff = VARIANTS[vname]['n_diff_feats']
    diff_end = diff_start + n_diff

    imp_diff = imp[diff_start:diff_end]
    imp_total_diff = imp_diff.sum()
    names = diff_feat_names[vname]

    detail = ', '.join(f'{n}={v:.3f}' for n, v in zip(names, imp_diff))
    print(f'  {vname}: diffusion_total={imp_total_diff:.3f}  ({detail})')

# =====================================================================
# ETAPE 6 : Speedup et verdict
# =====================================================================
print('\n' + '=' * 90)
print('  VERDICT')
print('=' * 90)

# Calcul speedup
if 'V1_dijkstra' in pred_times and 'V3_sparse' in pred_times:
    speedup = pred_times['V1_dijkstra'] / max(pred_times['V3_sparse'], 0.1)
    print(f'  Speedup V3_sparse vs V1_dijkstra: {speedup:.1f}x')

if 'V1_dijkstra' in pred_times and 'V4_sparse+' in pred_times:
    speedup4 = pred_times['V1_dijkstra'] / max(pred_times['V4_sparse+'], 0.1)
    print(f'  Speedup V4_sparse+ vs V1_dijkstra: {speedup4:.1f}x')

# Meilleure variante par wMAPE
best_name = None
best_wmape = 999
for name, pred in all_preds_c18.items():
    ae = np.abs(true_v - pred)
    wmape = np.sum(ae) / np.sum(true_v) * 100
    if wmape < best_wmape:
        best_wmape = wmape
        best_name = name

print(f'\n  Meilleure variante: {best_name} (wMAPE={best_wmape:.1f}%)')
print(f'  Recommendation: remplacer Dijkstra par sparse filter si wMAPE comparable')
print(f'  et speedup significatif (>2x)')


  COMPARAISON : Dijkstra vs Sparse Graph Filter

--- Etape 0: Construction W_norm sparse ---
  W_norm: (4393, 4393), nnz=10282, density=0.0005

--- Etape 1: Entrainement des 4 variantes ---
  V1_dijkstra: 3700 samples, 39 feats, R2_train=0.879, time=2s
  V2_no_diff: 3700 samples, 39 feats, R2_train=0.881, time=2s
  V3_sparse: 3700 samples, 39 feats, R2_train=0.877, time=2s
  V4_sparse+: 3700 samples, 40 feats, R2_train=0.876, time=2s

--- Etape 2: Predictions LOO ---
    V1_dijkstra: 200/1000 (5s)
    V1_dijkstra: 400/1000 (10s)
    V1_dijkstra: 600/1000 (16s)
    V1_dijkstra: 800/1000 (21s)
    V1_dijkstra: 1000/1000 (26s)
  V1_dijkstra: predictions done in 26s
    V2_no_diff: 200/1000 (5s)
    V2_no_diff: 400/1000 (11s)
    V2_no_diff: 600/1000 (16s)
    V2_no_diff: 800/1000 (21s)
    V2_no_diff: 1000/1000 (26s)
  V2_no_diff: predictions done in 26s
    V3_sparse: 200/1000 (5s)
    V3_sparse: 400/1000 (10s)
    V3_sparse: 600/1000 (15s)
    V3_sparse: 800/1000 (20s)
    V3_sparse: 10